# Polyp FL complete study - RUN ALL

Is notebook ko Jupyter ke home folder mein upload karein (jahan `PraNet_data/` aur purane `results_*` folders hain), kholen aur **Kernel -> Restart & Run All** dabayein.

- Cell 1 poora code `polyp_fl_complete_study.py` file mein likh deta hai (violin figure fix is mein pehle se shamil hai).
- Har step alag process mein chalta hai. Ek step fail ho to baaqi chalte rehte hain.
- Har step ka log `log_<step>.txt` mein save hota hai. Tables: `outputs/tables`, figures: `outputs/figures`, reports: `outputs/reports`.
- Purane trained models skip ho jate hain, is liye dobara run karna safe hai.

In [ ]:
%%writefile polyp_fl_complete_study.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
polyp_fl_complete_study.py  --  ONE file: the complete, submission-ready experiment code
=========================================================================================
Study : local vs pooled vs federated (FedAvg / FedProx / FedBN) polyp segmentation, plain ResNet-50 U-Net,
        PraNet protocol (900 Kvasir-SEG + 550 CVC-ClinicDB for training; Kvasir 100, CVC-ClinicDB 62,
        CVC-300 60, CVC-ColonDB 380, ETIS 196 for testing), leakage-aware evaluation, XAI, robustness.

This file merges  (1) Kvasir_SEG_updated_17_OCTOBER  and  (2) CVC_updated_final_journal_17_OCTOBER  (every cell,
nothing dropped) and ADDS every experiment/analysis that the final critical review (Review 3) asked for:

   M2  protocol-control arms (pooled: constant LR + last epoch | FedAvg: best round + LR decay)      -> step  m2
   M3  test/interval consistency: paired sign-flip test on the MEAN difference (Holm), bootstrap CI,
       Bonferroni-adjusted CI, Hodges-Lehmann median difference (used in EVERY comparison table)    -> all table steps
   M6  FedBN as ONE deployable model: probability-map ensemble + AdaBN; seen-site FedBN = personalised -> step m6
   M8  CVC-ClinicDB re-split by video-sequence group (hash-graph components), retrain 4 regimes      -> step m8
   S1  "unseen (image-weighted)" column (636 images) with its own paired test                        -> step tables
   S2  FedAvg with equal aggregation weights (0.5/0.5)                                                -> step s2
   S3  explanation-vs-accuracy: IoU-Dice correlation + encoder-layer Grad-CAM++ for all six regimes   -> step xai_layers
   S4  YOUR stronger-augmentation recipe (aug2 notebook, all 6 regimes x 5 seeds, Table XI) + corruption test -> step aug2
   S5  empty-prediction counts, precision with empties = 0, per-image validation Dice (logged every epoch;
       --val-per-image makes it the checkpoint-selection metric; step valtest reports whether selection would change) -> steps pr_hd95, valtest
   S6  field-of-view hashing + SSIM >= 0.9 confirmation + montage of flagged pairs                    -> step leak
   M7  reproducibility package + SUBMISSION_CHECKLIST.md (the [FILL] items you must complete)         -> step repro
   M1/M4/M5/S7/S8 are WRITING items: the numbers they need are produced in  outputs/numbers_for_paper.md

HOW TO RUN
----------
   pip install tensorflow==2.12.0 opencv-python-headless scikit-learn scipy pandas matplotlib scikit-image lime tqdm h5py
   python polyp_fl_complete_study.py --list                    # show all steps
   python polyp_fl_complete_study.py --quick                   # 10-15 min smoke test (1 seed, 2 epochs, *_quicktest folders)
   python polyp_fl_complete_study.py --steps core              # everything that the two notebooks did
   python polyp_fl_complete_study.py --steps review            # only the NEW review experiments
   python polyp_fl_complete_study.py                           # = --steps all
   python polyp_fl_complete_study.py --steps m2,m6 --seeds 42,123
   python polyp_fl_complete_study.py --val-per-image ...       # S5: per-image validation Dice selects checkpoints of NEW centralised runs (default = old protocol)
   inside Jupyter:  %run polyp_fl_complete_study.py --steps tables
Training is RESUMABLE and SHARED with your notebooks: result folders / file names are identical
(results_cvc, results_kvasir, results_fed, <tag>_seed<s>.weights.h5 / .npz), so everything you already trained is SKIPPED.
Every table is written to  outputs/tables/*.csv  and every figure to  outputs/figures/*.png|pdf  automatically.
"""
import os, sys, re, json, glob, time, shutil, pickle, random, itertools, argparse, datetime, platform, subprocess
import warnings, contextlib, gc, math

# ----------------------------------------------------------------------------------------------------------------
# command line (parse_known_args -> also works inside Jupyter / %run)
# ----------------------------------------------------------------------------------------------------------------
_ap = argparse.ArgumentParser(description="Complete polyp-segmentation FL study (see module docstring)")
_ap.add_argument("--steps", default="all", help="comma list of steps or groups: all | core | review | <step names> (see --list)")
_ap.add_argument("--skip", default="", help="comma list of steps to skip")
_ap.add_argument("--quick", action="store_true", help="QUICK_TEST: 1 seed, 2 epochs, *_quicktest folders")
_ap.add_argument("--root", default="PraNet_data", help="folder that contains TrainDataset and TestDataset")
_ap.add_argument("--seeds", default="", help="override seeds, e.g. 42,123,2024")
_ap.add_argument("--out", default="outputs", help="folder for tables / figures / reports")
_ap.add_argument("--list", action="store_true", help="list steps and exit")
_ap.add_argument("--val-per-image", action="store_true", dest="val_per_image",
                 help="S5: select the best checkpoint / drive ReduceLROnPlateau with the EXACT per-image validation Dice instead of the Keras batch-average "
                      "(only for NEW centralised runs; use it for ALL regimes of a study or none - already trained runs are never re-trained)")
ARGS, _UNKNOWN = _ap.parse_known_args()
VAL_PER_IMAGE = bool(ARGS.val_per_image)      # default False = the protocol of every already-trained model (Keras val_dice_coef selects the checkpoint)

# GPU choice MUST happen before TensorFlow is imported (picks the GPU with most free memory unless you set CUDA_VISIBLE_DEVICES)
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "1")
if not os.environ.get("CUDA_VISIBLE_DEVICES"):
    try:
        _o = subprocess.run("nvidia-smi --query-gpu=index,memory.free --format=csv,noheader,nounits", shell=True,
                            capture_output=True, text=True, timeout=30).stdout
        _free = {int(l.split(",")[0]): int(l.split(",")[1]) for l in _o.strip().splitlines() if "," in l}
        if _free:
            os.environ["CUDA_VISIBLE_DEVICES"] = str(max(_free, key=_free.get))
            print("GPU free memory (MiB):", _free, "-> using GPU", os.environ["CUDA_VISIBLE_DEVICES"])
    except Exception as _e:                                   # no nvidia-smi: let TensorFlow decide
        print("nvidia-smi not available:", repr(_e)[:80])

import numpy as np
import pandas as pd
import cv2
from scipy import ndimage, stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

import matplotlib
INTERACTIVE = "ipykernel" in sys.modules or hasattr(sys, "ps1")
if not INTERACTIVE:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 70)
plt.rcParams.update({"font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
                     "legend.fontsize": 8, "figure.dpi": 100, "savefig.dpi": 300})          # >= 8 pt everywhere (review M7)

IMG_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")

# ----------------------------------------------------------------------------------------------------------------
# output helpers: replace notebook `display()` / `plt.show()` -> every table and figure is SAVED automatically
# ----------------------------------------------------------------------------------------------------------------
OUT_ROOT = ARGS.out + ("_quicktest" if ARGS.quick else "")
TAB_DIR, FIG_DIR, REP_DIR = (os.path.join(OUT_ROOT, d) for d in ("tables", "figures", "reports"))
for _d in (TAB_DIR, FIG_DIR, REP_DIR):
    os.makedirs(_d, exist_ok=True)
_STEP = ["init"]
_CNT = {"tab": 0, "fig": 0}


def set_step(name):
    _STEP[0] = name
    _CNT["tab"] = _CNT["fig"] = 0


def _slug(s):
    return re.sub(r"[^A-Za-z0-9_.+-]+", "_", str(s)).strip("_")[:90]


def display(obj, name=None, latex=False, show=True):
    """Notebook-`display` replacement: prints the table and saves it as CSV (and LaTeX when name+latex are given)."""
    if isinstance(obj, pd.DataFrame):
        if show:
            print(obj.to_string(max_colwidth=70))
        _CNT["tab"] += 1
        fn = _slug(name) if name else f"{_STEP[0]}_table{_CNT['tab']:02d}"
        try:
            obj.to_csv(os.path.join(TAB_DIR, fn + ".csv"), index=not isinstance(obj.index, pd.RangeIndex))
            if latex:
                with open(os.path.join(TAB_DIR, fn + ".tex"), "w") as f:
                    f.write(obj.to_latex(index=not isinstance(obj.index, pd.RangeIndex), escape=True))
        except Exception as e:
            print("   (table not saved:", repr(e)[:100], ")")
    elif isinstance(obj, pd.Series):
        display(obj.to_frame(), name=name, latex=latex, show=show)
    else:
        print(obj)


_orig_show = plt.show


def _save_and_close(*a, **k):
    for num in plt.get_fignums():
        fig = plt.figure(num)
        _CNT["fig"] += 1
        base = os.path.join(FIG_DIR, f"{_STEP[0]}_fig{_CNT['fig']:02d}")
        try:
            fig.savefig(base + ".png", dpi=200, bbox_inches="tight")
            fig.savefig(base + ".pdf", bbox_inches="tight")
        except Exception as e:
            print("   (figure not saved:", repr(e)[:100], ")")
    if INTERACTIVE:
        try:
            _orig_show()
        except Exception:
            pass
    plt.close("all")


plt.show = _save_and_close


def save_fig_named(fig, name):
    """Save a figure under an explicit name (png + pdf) and close it."""
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{_slug(name)}.{ext}"), dpi=300, bbox_inches="tight")
    plt.close(fig)

IMG_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")


# ----------------------------------------------------------------------------
# 1. files / loading / split
# ----------------------------------------------------------------------------
def pair_files(img_dir, mask_dir):
    """Pair images and masks by base-name (extension-insensitive). Returns two sorted lists."""
    imgs = sorted(f for f in os.listdir(img_dir) if f.lower().endswith(IMG_EXTS))
    masks = {os.path.splitext(f)[0].lower(): f
             for f in os.listdir(mask_dir) if f.lower().endswith(IMG_EXTS)}
    ip, mp = [], []
    for f in imgs:
        k = os.path.splitext(f)[0].lower()
        if k in masks:
            ip.append(os.path.join(img_dir, f))
            mp.append(os.path.join(mask_dir, masks[k]))
    print(f"[pair_files] {img_dir}: {len(imgs)} images, {len(masks)} masks -> {len(ip)} pairs")
    assert len(ip) > 0, "No image/mask pairs found - check folder names and file-name matching"
    return ip, mp


def apply_clahe(img_rgb_u8, clip=2.0, tile=8):
    lab = cv2.cvtColor(img_rgb_u8, cv2.COLOR_RGB2LAB)
    lab[:, :, 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def load_arrays(img_paths, mask_paths, size):
    """Load RAW (no CLAHE, no normalisation) uint8 arrays. X:(N,S,S,3) RGB, Y:(N,S,S,1) in {0,1}.
    cv2 is used so .tif/.bmp (ETIS / ColonDB) work too."""
    n = len(img_paths)
    X = np.zeros((n, size, size, 3), np.uint8)
    Y = np.zeros((n, size, size, 1), np.uint8)
    for i, (ip, mp) in enumerate(zip(img_paths, mask_paths)):
        im = cv2.imread(ip, cv2.IMREAD_COLOR)
        m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        if im is None or m is None:
            raise RuntimeError(f"Cannot read {ip} or {mp}")
        im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        X[i] = cv2.resize(im, (size, size), interpolation=cv2.INTER_LINEAR)
        Y[i, ..., 0] = (cv2.resize(m, (size, size), interpolation=cv2.INTER_NEAREST) > 127)
    return X, Y


def make_split(n, names=None, seed=42, val=0.15, test=0.15):
    """Index split of the ORIGINAL images (do this BEFORE any augmentation)."""
    idx = np.arange(n)
    tr, tmp = train_test_split(idx, test_size=val + test, random_state=seed)
    va, te = train_test_split(tmp, test_size=test / (val + test), random_state=seed)
    return {"seed": seed, "n": int(n),
            "names": [os.path.basename(x) for x in names] if names is not None else None,
            "train": sorted(map(int, tr)), "val": sorted(map(int, va)), "test": sorted(map(int, te))}


def save_split(split, path):
    with open(path, "w") as f:
        json.dump(split, f)


def load_split(path):
    with open(path) as f:
        return json.load(f)


def prepare_all(P, size, seed=42):
    """P = {'cvc':(img_dir,mask_dir), 'kvasir':(...), 'colondb':(...), 'etis':(...)}.
    Loads every dataset ONCE (raw uint8). cvc & kvasir get a deterministic 70/15/15 split of the ORIGINAL images
    (same seed => identical split in every notebook). colondb & etis are test-only.
    Returns D (everything) and TEST = {'kvasir': test split, 'cvc': test split, 'colondb': all, 'etis': all}
    -> the SAME four test sets are used for every model, so paired comparisons are valid."""
    D = {}
    for key, (idir, mdir) in P.items():
        ip, mp = pair_files(idir, mdir)
        X, Y = load_arrays(ip, mp, size)
        e = {"img_paths": ip, "X": X, "Y": Y}
        if key in ("cvc", "kvasir"):
            sp = make_split(len(ip), ip, seed)
            e["split"] = sp
            for part in ("train", "val", "test"):
                e[part] = (X[sp[part]], Y[sp[part]])
        else:
            e["test"] = (X, Y)
        D[key] = e
    return D, {k: D[k]["test"] for k in D}


def _subdir(path, candidates):
    for c in os.listdir(path):
        if c.lower() in candidates and os.path.isdir(os.path.join(path, c)):
            return os.path.join(path, c)
    raise FileNotFoundError(f"none of {candidates} found inside {path} (found: {os.listdir(path)})")


def _child(path, name):
    """case-insensitive child directory called `name` (or None)"""
    if path and os.path.isdir(path):
        for c in os.listdir(path):
            if c.lower() == name and os.path.isdir(os.path.join(path, c)):
                return os.path.join(path, c)
    return None


def _find_dir(name, bases, max_depth=4):
    """search for a directory called `name` (case-insensitive) under every base (depth-limited, hidden dirs skipped)"""
    skip = {"site-packages", "dist-packages", "node_modules", "__pycache__", "proc", "sys", "dev"}
    for base in bases:
        if not base or not os.path.isdir(base):
            continue
        base = os.path.abspath(base)
        b_depth = base.count(os.sep)
        for dp, dn, _ in os.walk(base):
            dn[:] = [d for d in dn if not d.startswith(".") and d not in skip]
            for d in dn:
                if d.lower() == name:
                    return os.path.join(dp, d)
            if dp.count(os.sep) - b_depth >= max_depth:
                dn[:] = []
    return None


def locate_pranet(root=None):
    """Find the PraNet TrainDataset and TestDataset folders. They may be:
         root/TrainDataset + root/TestDataset, or anywhere below the working dir, /tf, /content, /kaggle/input, home ...,
         or doubly nested (TrainDataset/TrainDataset/...), or unzipped into two different places."""
    bases = [root, os.getcwd(), "/tf", "/content", "/kaggle/input", "/workspace", "/data", "/mnt",
             os.path.expanduser("~"), ".."]
    found = {}
    for key, name in (("train", "traindataset"), ("test", "testdataset")):
        d = _child(root, name) or _find_dir(name, bases)
        if d is not None:                               # TrainDataset/TrainDataset/... (double unzip)
            inner = _child(d, name)
            if inner is not None and _child(d, "images") is None and _child(d, "image") is None:
                d = inner
        found[key] = d
    if found["train"] is None or found["test"] is None:
        raise FileNotFoundError(
            f"PraNet folders nahi mile (TrainDataset: {found['train']}, TestDataset: {found['test']}).\n"
            f"Current working dir: {os.getcwd()}  ->  isme ye folders hain: {sorted(os.listdir('.'))[:25]}\n"
            "Notebook ke ek cell mein ye chalao aur jo path aaye usay PRANET_ROOT mein daalo:\n"
            "    !find / -maxdepth 6 -type d -iname 'TrainDataset' 2>/dev/null\n"
            "    !find / -maxdepth 6 -type d -iname 'TestDataset'  2>/dev/null")
    print("[locate_pranet] TrainDataset:", found["train"], "| TestDataset:", found["test"])
    return found["train"], found["test"]


def prepare_pranet(root, size, val_frac=0.1, seed=42):
    """STANDARD PraNet protocol (like-for-like with PraNet / HarDNet-MSEG / UACANet / Polyp-PVT numbers):
         TrainDataset/{images|image}, {masks|gt}              1450 = 900 Kvasir + 550 CVC-ClinicDB
         TestDataset/<name>/{images|image}, {masks|gt}        Kvasir(100) CVC-ClinicDB(62) CVC-300(60) CVC-ColonDB(380) ETIS(196)
       `root` = folder that contains TrainDataset and TestDataset. If it is wrong/missing, the folders are SEARCHED
       automatically (working dir, /tf, /content, /kaggle/input, home ...). Folder names are matched case-insensitively.
       Train images are split by SOURCE (CVC frames have numeric file names, Kvasir has hash names).
       `val_frac` of each source is held out as validation (used ONLY for checkpoint selection).
       Returns D (same layout as prepare_all) and TEST = the 5 official test sets."""
    imgs, msks = ["images", "image"], ["masks", "gt", "mask"]
    tr_root, te_root = locate_pranet(root)
    ip, mp = pair_files(_subdir(tr_root, imgs), _subdir(tr_root, msks))
    src = np.array(["cvc" if os.path.splitext(os.path.basename(p))[0].isdigit() else "kvasir" for p in ip])
    print("[prepare_pranet] train sources:", {k: int((src == k).sum()) for k in ("kvasir", "cvc")}, "(expected kvasir 900, cvc 550)")
    X, Y = load_arrays(ip, mp, size)
    D = {}
    for k in ("kvasir", "cvc"):
        idx = np.where(src == k)[0]
        tr, va = train_test_split(idx, test_size=val_frac, random_state=seed)
        D[k] = {"train": (X[np.sort(tr)], Y[np.sort(tr)]), "val": (X[np.sort(va)], Y[np.sort(va)]),
                "paths_train": [ip[i] for i in np.sort(tr)], "paths_val": [ip[i] for i in np.sort(va)],
                "split": {"train": [os.path.basename(ip[i]) for i in np.sort(tr)], "val": [os.path.basename(ip[i]) for i in np.sort(va)]}}
    inner = _child(te_root, "testdataset")                 # TestDataset/TestDataset/... (double unzip)
    if inner is not None:
        te_root = inner
    name_map = {"kvasir": "kvasir", "clinic": "cvc", "300": "cvc300", "colon": "colondb", "etis": "etis"}
    for d in sorted(os.listdir(te_root)):
        full = os.path.join(te_root, d)
        key = next((v for kk, v in name_map.items() if kk in d.lower()), None)
        if not os.path.isdir(full) or key is None:
            continue
        tip, tmp = pair_files(_subdir(full, imgs), _subdir(full, msks))
        Xt, Yt = load_arrays(tip, tmp, size)
        D.setdefault(key, {})["test"] = (Xt, Yt)
        D[key]["paths_test"] = tip
        if "split" in D[key]:
            D[key]["split"]["test"] = [os.path.basename(p) for p in tip]
    TEST = {k: D[k]["test"] for k in ("kvasir", "cvc", "cvc300", "colondb", "etis") if k in D and "test" in D[k]}
    missing = [k for k in ("kvasir", "cvc", "cvc300", "colondb", "etis") if k not in TEST]
    if missing:
        print("WARNING: ye test sets nahi mile:", missing, "-> TestDataset ke andar folder naam check karo:", sorted(os.listdir(te_root)))
    print("[prepare_pranet] test sets:", {k: v[0].shape[0] for k, v in TEST.items()}, "(expected 100/62/60/380/196)")
    return D, TEST


def sub(arr_or_list, idx):
    if isinstance(arr_or_list, np.ndarray):
        return arr_or_list[idx]
    return [arr_or_list[i] for i in idx]


# ----------------------------------------------------------------------------
# 2. pHash near-duplicate detector
# ----------------------------------------------------------------------------
def phash_bits(img_bgr, hash_size=8, highfreq=4):
    g = img_bgr if img_bgr.ndim == 2 else cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, (hash_size * highfreq,) * 2, interpolation=cv2.INTER_AREA).astype(np.float32)
    d = cv2.dct(g)[:hash_size, :hash_size].flatten()
    return d > np.median(d[1:])


def near_duplicates(paths_train, paths_other, max_dist=8):
    """For every image in `paths_other` (e.g. test) find the nearest pHash neighbour in `paths_train`.
    Returns dict(frac_flagged, n_flagged, nearest_dist). Expected for a clean split of independent
    images: frac close to 0. CVC-ClinicDB frames come from video sequences, so some neighbours are
    unavoidable -> REPORT the number in the paper instead of hiding it."""
    ha = np.stack([phash_bits(cv2.imread(p)) for p in paths_train])
    hb = np.stack([phash_bits(cv2.imread(p)) for p in paths_other])
    dist = (ha[:, None, :] != hb[None, :, :]).sum(2)          # (Ntrain, Nother)
    nearest = dist.min(0)
    flagged = int((nearest <= max_dist).sum())
    return {"n_other": len(paths_other), "n_flagged": flagged,
            "frac_flagged": flagged / len(paths_other), "nearest_dist": nearest, "max_dist": max_dist}


# ----------------------------------------------------------------------------
# 3. metrics
# ----------------------------------------------------------------------------
def per_image_metrics(y_true, y_pred, eps=1e-8):
    """y_true, y_pred: (N,H,W[,1]) binary/prob arrays (thresholded at 0.5 here). Returns dict of (N,) arrays."""
    n = len(y_true)
    yt = y_true.reshape(n, -1) > 0.5
    yp = y_pred.reshape(n, -1) > 0.5
    tp = (yt & yp).sum(1).astype(np.float64)
    fp = ((~yt) & yp).sum(1).astype(np.float64)
    fn = (yt & (~yp)).sum(1).astype(np.float64)
    tn = ((~yt) & (~yp)).sum(1).astype(np.float64)
    return {"dice": (2 * tp + eps) / (2 * tp + fp + fn + eps),
            "iou": (tp + eps) / (tp + fp + fn + eps),
            "precision": (tp + eps) / (tp + fp + eps),
            "recall": (tp + eps) / (tp + fn + eps),
            "accuracy": (tp + tn) / (tp + tn + fp + fn),
            "tp": tp, "fp": fp, "fn": fn, "tn": tn}


def summarize(m):
    """Headline numbers. mDice / mIoU (mean of per-image) are the ones to compare with PraNet-style papers.
    micro_* (pooled pixels) are kept ONLY for transparency / backward-compat with the old tables."""
    tp, fp, fn = m["tp"].sum(), m["fp"].sum(), m["fn"].sum()
    return {"mDice": float(m["dice"].mean()), "mIoU": float(m["iou"].mean()),
            "mPrecision": float(m["precision"].mean()), "mRecall": float(m["recall"].mean()),
            "mAccuracy": float(m["accuracy"].mean()),
            "micro_dice": float(2 * tp / (2 * tp + fp + fn + 1e-8)),
            "micro_iou": float(tp / (tp + fp + fn + 1e-8)), "n": int(len(m["dice"]))}


def bootstrap_ci(x, n_boot=10000, alpha=0.05, seed=0):
    x = np.asarray(x, dtype=np.float64)
    rng = np.random.default_rng(seed)
    means = np.empty(n_boot)
    for b in range(0, n_boot, 1000):                      # chunked -> low memory
        k = min(1000, n_boot - b)
        means[b:b + k] = x[rng.integers(0, len(x), (k, len(x)))].mean(1)
    return float(np.percentile(means, 100 * alpha / 2)), float(np.percentile(means, 100 * (1 - alpha / 2)))


def largest_component(mask01):
    m = (mask01 > 0.5).astype(np.uint8)
    lab, num = ndimage.label(m)
    if num == 0:
        return m
    sizes = ndimage.sum(m, lab, range(1, num + 1))
    return (lab == 1 + int(np.argmax(sizes))).astype(np.uint8)


def binarize(probs, thr=0.5, post=False):
    p = (probs > thr).astype(np.uint8)
    if post:
        p = np.stack([largest_component(x.squeeze())[..., None] for x in p])
    return p


# ----------------------------------------------------------------------------
# 4. multi-seed statistics
# ----------------------------------------------------------------------------
def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (len(p) - rank) * p[i])
        adj[i] = min(1.0, run)
    return adj


def paired_wilcoxon(a, b):
    a, b = np.asarray(a), np.asarray(b)
    if np.allclose(a, b):
        return 1.0
    return float(stats.wilcoxon(a, b, zero_method="wilcox", alternative="two-sided").pvalue)


def compare_configs(per_image, ref, seed_boot=0):
    """per_image: {config_name: [array(N,) for each seed]}  (SAME image order everywhere!)
    Statistical unit = the IMAGE: per-image Dice is first averaged over seeds, then ONE paired Wilcoxon
    per comparison (NOT 3 x N pseudo-replicates).  Holm correction across all comparisons vs `ref`.
    Returns list of row-dicts (feed to pandas.DataFrame)."""
    rows, names = [], [k for k in per_image if k != ref]
    ref_avg = np.mean(np.stack(per_image[ref]), 0)
    pvals = [paired_wilcoxon(ref_avg, np.mean(np.stack(per_image[k]), 0)) for k in names]
    adj = holm(pvals) if pvals else []
    seed_means = lambda k: np.array([a.mean() for a in per_image[k]])
    rows.append({"config": ref, "mDice_mean": seed_means(ref).mean(), "mDice_std_over_seeds": seed_means(ref).std(ddof=1) if len(per_image[ref]) > 1 else np.nan,
                 "diff_vs_ref": 0.0, "diff_CI95": (0.0, 0.0), "p_raw": np.nan, "p_holm": np.nan})
    for k, p, pa in zip(names, pvals, adj):
        avg = np.mean(np.stack(per_image[k]), 0)
        rows.append({"config": k, "mDice_mean": seed_means(k).mean(),
                     "mDice_std_over_seeds": seed_means(k).std(ddof=1) if len(per_image[k]) > 1 else np.nan,
                     "diff_vs_ref": float((avg - ref_avg).mean()),
                     "diff_CI95": bootstrap_ci(avg - ref_avg, seed=seed_boot), "p_raw": p, "p_holm": float(pa)})
    return rows


def load_metric(tag, seeds, key, metric="dice", out_dir="results"):
    """list (one (N,) array per seed) of per-image `metric` ('dice'|'iou') saved by run_seeds / save_fed_result"""
    return [np.load(os.path.join(out_dir, f"{tag}_seed{s}.npz"))[f"{metric}__{key}"] for s in seeds]


def results_table(tag, seeds, keys, out_dir="results"):
    """One row per test set: mDice (mean +- std over seeds), 95% bootstrap CI of the seed-averaged per-image Dice, mIoU."""
    import pandas as pd
    rows = []
    for k in keys:
        d, i = load_metric(tag, seeds, k, "dice", out_dir), load_metric(tag, seeds, k, "iou", out_dir)
        dm = np.mean(d, 0)
        lo, hi = bootstrap_ci(dm)
        sd = np.std([x.mean() for x in d], ddof=1) if len(d) > 1 else np.nan
        rows.append({"test_set": k, "n_images": len(dm), "mDice": np.mean([x.mean() for x in d]), "std_over_seeds": sd,
                     "CI95_low": lo, "CI95_high": hi, "mIoU": np.mean([x.mean() for x in i])})
    return pd.DataFrame(rows).round(4)


# ----------------------------------------------------------------------------
# 5. polyp-size stratification
# ----------------------------------------------------------------------------
def size_bins(masks):
    n = len(masks)
    area = masks.reshape(n, -1).sum(1).astype(np.float64)
    frac = area / masks[0].size
    p33, p66 = np.percentile(area, [33, 66])
    lab = np.where(area <= p33, "Small", np.where(area <= p66, "Medium", "Large"))
    return lab, frac, (float(p33), float(p66))


def dice_by_bin(dice, labels, order=("Small", "Medium", "Large")):
    return {b: (float(dice[labels == b].mean()), int((labels == b).sum())) for b in order}


# ----------------------------------------------------------------------------
# 6. synthetic corruptions: imgs float32 RGB in [0,1], shape (N,H,W,3); sev 0 = clean
# ----------------------------------------------------------------------------
def _gauss_noise(x, s, rng):
    sig = [0.02, 0.04, 0.06, 0.08, 0.10][s - 1]
    return np.clip(x + rng.normal(0, sig, x.shape).astype(np.float32), 0, 1)


def _gauss_blur(x, s, rng):
    k = [3, 5, 7, 9, 11][s - 1]
    return np.clip(np.stack([cv2.GaussianBlur(i, (k, k), 0) for i in x]), 0, 1).astype(np.float32)


def _bright_contrast(x, s, rng):
    b, c = [(0.05, .95), (0.10, .90), (0.15, .85), (0.20, .80), (0.25, .75)][s - 1]
    return np.clip((x - 0.5) * c + 0.5 + b, 0, 1).astype(np.float32)


def _jpeg(x, s, rng):
    q = [80, 60, 40, 20, 10][s - 1]
    out = np.zeros_like(x)
    for i, im in enumerate(x):
        bgr = cv2.cvtColor((im * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
        dec = cv2.imdecode(cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, q])[1], cv2.IMREAD_COLOR)
        out[i] = cv2.cvtColor(dec, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return out


def _motion_blur(x, s, rng):
    k = [5, 9, 13, 17, 21][s - 1]
    ker = np.zeros((k, k), np.float32)
    ker[k // 2, :] = 1.0 / k
    return np.clip(np.stack([cv2.filter2D(i, -1, ker) for i in x]), 0, 1).astype(np.float32)


CORRUPTIONS = {"Gaussian Noise": _gauss_noise, "Gaussian Blur": _gauss_blur,
               "Brightness/Contrast": _bright_contrast, "JPEG Compression": _jpeg, "Motion Blur": _motion_blur}


def corrupt(name, x01, sev, seed=0):
    if sev == 0:
        return x01.astype(np.float32)
    return CORRUPTIONS[name](x01.astype(np.float32), sev, np.random.default_rng(seed + sev))


# ----------------------------------------------------------------------------
# 7. XAI metrics
# ----------------------------------------------------------------------------
def _norm(h):
    h = np.asarray(h, np.float64)
    return (h - h.min()) / (h.max() - h.min() + 1e-12)


def xai_metrics(heatmap, gt):
    """heatmap: (H,W) >=0 ; gt: (H,W[,1]) binary.
    pointing_game : majority of the (tied) maximum pixels lie inside GT  (fixes the 'argmax of a binary
                    LIME mask = first pixel' artefact that made LIME = 0.000)
    energy_pg     : fraction of heat-map energy inside GT
    iou_topg      : IoU between GT and the TOP-g pixels, g = |GT|  (ceiling = 1.0; the old '50th percentile'
                    threshold had ceiling 2*|GT|/|image| ~ 0.2-0.4 -> not interpretable)
    auroc / auprc : threshold-free pixel-wise localisation quality"""
    gt = np.asarray(gt).squeeze() > 0.5
    h = _norm(heatmap.squeeze())
    flat, y = h.ravel(), gt.ravel().astype(int)
    k = int(gt.sum())
    hit = bool(gt[h >= h.max() - 1e-9].mean() >= 0.5)
    energy = float((h * gt).sum() / (h.sum() + 1e-12))
    if k == 0 or k == gt.size:
        iou = auroc = auprc = np.nan
    else:
        top = np.zeros(flat.shape, bool)
        top[np.argpartition(-flat + 1e-9 * np.random.default_rng(0).random(flat.shape), k - 1)[:k]] = True
        top = top.reshape(gt.shape)
        iou = float((top & gt).sum() / ((top | gt).sum() + 1e-12))
        auroc = float(roc_auc_score(y, flat))
        auprc = float(average_precision_score(y, flat))
    return {"pointing_game": hit, "energy_pg": energy, "iou_topg": iou, "auroc": auroc, "auprc": auprc}


def center_prior(h, w, sigma_frac=0.25):
    yy, xx = np.mgrid[0:h, 0:w]
    return np.exp(-(((yy - h / 2) / (sigma_frac * h)) ** 2 + ((xx - w / 2) / (sigma_frac * w)) ** 2) / 2)


def random_map(h, w, seed=0):
    return np.random.default_rng(seed).random((h, w))


def lime_heatmap(explainer, img_uint8, predict_fn, num_samples=300, n_segments=50, compactness=10, seed=0):
    """CONTINUOUS LIME heat-map for class 1 (= polyp).
    Old code used top_labels=1 -> because a polyp covers ~10% of the image, the top label was usually
    class 0 (background), i.e. LIME explained 'not-polyp' (-> pointing game 0.000).
    predict_fn(batch_uint8) must return (B,2) probabilities [background, polyp]."""
    from skimage.segmentation import slic
    seg_fn = lambda im: slic(im, n_segments=n_segments, compactness=compactness, start_label=0)
    import lime.lime_image as _li
    kw = dict(labels=(1,), top_labels=None, hide_color=0, num_samples=num_samples,
              segmentation_fn=seg_fn, random_seed=seed)
    _orig = getattr(_li, "tqdm", None)                 # progress bar band (har lime version par chalta hai)
    if _orig is not None:
        _li.tqdm = lambda it, *a, **k: it
    try:
        exp = explainer.explain_instance(img_uint8, predict_fn, **kw)
    finally:
        if _orig is not None:
            _li.tqdm = _orig
    seg = exp.segments
    hm = np.zeros(seg.shape, np.float32)
    for sp, wgt in dict(exp.local_exp[1]).items():
        hm[seg == sp] = wgt
    return np.maximum(hm, 0)


# ----------------------------------------------------------------------------
# 8. HD95 (boundary quality) + relative robustness
# ----------------------------------------------------------------------------
def hd95_batch(y_true, y_pred, empty_penalty="diag"):
    """95th-percentile Hausdorff distance PER IMAGE, in pixels. y_true, y_pred: (N,H,W[,1]).
    If the GT or the prediction is empty the distance is undefined: penalty = image diagonal ('diag') or NaN ('nan').
    Returns (hd95 array (N,), number_of_empty_cases)."""
    from scipy.ndimage import distance_transform_edt, binary_erosion
    n = len(y_true)
    out = np.zeros(n)
    n_empty = 0
    for i in range(n):
        a = np.asarray(y_true[i]).squeeze() > 0.5
        b = np.asarray(y_pred[i]).squeeze() > 0.5
        diag = float(np.hypot(*a.shape))
        if not a.any() or not b.any():
            n_empty += 1
            out[i] = diag if empty_penalty == "diag" else np.nan
            continue
        sa, sb = a & ~binary_erosion(a), b & ~binary_erosion(b)
        d_ab = distance_transform_edt(~sb)[sa]
        d_ba = distance_transform_edt(~sa)[sb]
        out[i] = max(np.percentile(d_ab, 95), np.percentile(d_ba, 95))
    return out, n_empty


def relative_robustness(rb):
    """rb = {corruption: [Dice at severity 0 (clean), 1, ..., 5]} (output of robustness_eval).
    Returns {corruption: mean(Dice sev 1..5) / Dice(clean), ..., 'overall': mean over corruptions}  (1.0 = no degradation)."""
    out = {}
    for k, v in rb.items():
        v = np.asarray(v, float)
        out[k] = float(v[1:].mean() / (v[0] + 1e-8))
    out["overall"] = float(np.mean(list(out.values())))
    return out


# ----------------------------------------------------------------------------
# 9. display helpers: mean +- SD tables and violin plots (everything is SHOWN in the notebook, nothing is saved)
# ----------------------------------------------------------------------------
def pm(values, nd=3):
    """'0.912 +- 0.004' (SD with ddof=1) from a list of per-seed numbers"""
    v = np.asarray(values, float)
    return f"{v.mean():.{nd}f} \u00b1 {(v.std(ddof=1) if len(v) > 1 else float('nan')):.{nd}f}"


def ms_table(tag, seeds, keys, out_dir="results"):
    """One row per test set. mDice / mIoU as mean +- SD over seeds, SD of the per-image Dice, 95% bootstrap CI (seed-averaged per-image Dice)."""
    import pandas as pd
    rows = []
    for k in keys:
        d, i = load_metric(tag, seeds, k, "dice", out_dir), load_metric(tag, seeds, k, "iou", out_dir)
        dm = np.mean(d, 0)
        lo, hi = bootstrap_ci(dm)
        rows.append({"test_set": k, "n_images": len(dm), "mDice (mean \u00b1 SD over seeds)": pm([x.mean() for x in d]),
                     "mIoU (mean \u00b1 SD over seeds)": pm([x.mean() for x in i]), "per-image Dice SD": round(float(dm.std(ddof=1)), 3),
                     "95% CI (bootstrap)": f"[{lo:.3f}, {hi:.3f}]"})
    return pd.DataFrame(rows)


def violin_plot(groups, title="", ylabel="per-image Dice", ax=None, rotate=0, compact=False):
    """groups: {name: 1-D array of per-image values (seed-averaged)}. Violin + inner box, white dot = mean, text = mean \u00b1 SD."""
    import matplotlib.pyplot as plt
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(max(5, 1.6 * len(groups)), 4.5))
    names = list(groups)
    data = [np.asarray(groups[n], float)[~np.isnan(np.asarray(groups[n], float))] for n in names]
    parts = ax.violinplot(data, showextrema=False, widths=0.8)
    for b in parts["bodies"]:
        b.set_alpha(0.55)
    ax.boxplot(data, widths=0.15, showfliers=False, patch_artist=True, boxprops=dict(facecolor="white"), medianprops=dict(color="black"))
    ax.scatter(range(1, len(data) + 1), [d.mean() for d in data], color="red", zorder=3, s=22, label="mean")
    for j, d in enumerate(data):
        if compact:      # narrow panels (e.g. 2x3 grid): mean only, written vertically so the 6 labels never overlap (mean +- SD is in the tables)
            ax.text(j + 1, 1.02, f"{d.mean():.3f}", ha="center", va="bottom", rotation=90, fontsize=8)
        else:
            ax.text(j + 1, 1.02, f"{d.mean():.3f}\n\u00b1{d.std(ddof=1) if len(d) > 1 else float('nan'):.3f}", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(range(1, len(names) + 1)); ax.set_xticklabels(names, rotation=rotate, ha="right" if rotate else "center")
    ax.set_ylim(-0.02, 1.15); ax.set_ylabel(ylabel); ax.set_title(title, pad=34 if compact else 22); ax.grid(axis="y", alpha=0.3)
    if show:
        plt.tight_layout(); plt.show()
    return ax


def leakage_report(train_paths, test_paths_by_set, max_dist=8):
    """pHash near-duplicate check of EVERY test set against ALL training images. Returns a DataFrame."""
    import pandas as pd
    rows = []
    for k, paths in test_paths_by_set.items():
        nd = near_duplicates(train_paths, paths, max_dist=max_dist)
        rows.append({"test_set": k, "n_test": nd["n_other"], f"near-duplicate of a train image (pHash<={max_dist})": nd["n_flagged"],
                     "fraction": f"{nd['frac_flagged']:.1%}", "exact copies (distance 0)": int((nd["nearest_dist"] == 0).sum())})
    return pd.DataFrame(rows)
AUTOTUNE = tf.data.AUTOTUNE


# ----------------------------------------------------------------------------
# seeds / preprocessing
# ----------------------------------------------------------------------------
def set_seed(seed):
    try:
        tf.keras.utils.set_random_seed(seed)          # TF >= 2.7
    except AttributeError:                            # older TF
        random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    np.random.seed(seed)
    random.seed(seed)


def to_model_input(X_uint8, clahe=False):
    """Raw uint8 RGB (N,H,W,3) -> float32 ResNet50-preprocessed (BGR, mean-subtracted).
    THIS is the single place where preprocessing happens; use it for train/val/test/cross-dataset/robustness/XAI."""
    X = np.stack([apply_clahe(x) for x in X_uint8]) if clahe else X_uint8
    return keras.applications.resnet50.preprocess_input(X.astype(np.float32).copy())


def deprocess(xp):
    """Inverse of ResNet50 preprocess_input -> RGB float in [0,1] (for plots / LIME)."""
    x = (xp + np.array([103.939, 116.779, 123.68], np.float32))[..., ::-1] / 255.0
    return np.clip(x, 0, 1).astype(np.float32)


# ----------------------------------------------------------------------------
# tf.data with train-only augmentation (applied AFTER the split, on the fly)
# ----------------------------------------------------------------------------
def _augment(img, mask):
    """img float32 [0,255] (H,W,3); mask float32 {0,1} (H,W,1). Geometric ops hit image+mask jointly."""
    if tf.random.uniform(()) > 0.5:
        img, mask = tf.image.flip_left_right(img), tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img, mask = tf.image.flip_up_down(img), tf.image.flip_up_down(mask)
    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    img, mask = tf.image.rot90(img, k), tf.image.rot90(mask, k)
    if tf.random.uniform(()) > 0.5:                       # random zoom-in (scale 0.85-1.0)
        s = tf.random.uniform((), 0.85, 1.0)
        oy, ox = tf.random.uniform((), 0.0, 1.0 - s), tf.random.uniform((), 0.0, 1.0 - s)
        box = tf.stack([oy, ox, oy + s, ox + s])[None, :]
        both = tf.concat([img, mask * 255.0], -1)[None]
        hw = (int(img.shape[0]), int(img.shape[1])) if img.shape[0] is not None else tf.shape(img)[:2]
        out = tf.image.crop_and_resize(both, box, [0], hw, method="bilinear")[0]
        img, mask = out[..., :3], tf.cast(out[..., 3:] > 127.5, tf.float32)
    img = tf.image.random_brightness(img, 25.0)           # photometric: image only
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, mask


TRAIN_AUG = "standard"      # global default for training augmentation: "standard" (main, milder) or "strong" (aug2). step aug2 switches it.


def _augment_strong(img, mask):
    """img float32 [0,255] (H,W,3); mask float32 {0,1} (H,W,1). Geometric ops hit image+mask jointly.
    YOUR aug2 recipe (CVC_updated_final_journal_aug2 notebook): stronger TRAIN-ONLY augmentation (zoom 0.70-1.0, brightness/contrast, saturation, hue, light noise)."""
    if tf.random.uniform(()) > 0.5:
        img, mask = tf.image.flip_left_right(img), tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        img, mask = tf.image.flip_up_down(img), tf.image.flip_up_down(mask)
    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    img, mask = tf.image.rot90(img, k), tf.image.rot90(mask, k)
    if tf.random.uniform(()) > 0.3:                       # random zoom-in (scale 0.70-1.0)
        s = tf.random.uniform((), 0.70, 1.0)
        oy, ox = tf.random.uniform((), 0.0, 1.0 - s), tf.random.uniform((), 0.0, 1.0 - s)
        box = tf.stack([oy, ox, oy + s, ox + s])[None, :]
        both = tf.concat([img, mask * 255.0], -1)[None]
        hw = (int(img.shape[0]), int(img.shape[1])) if img.shape[0] is not None else tf.shape(img)[:2]
        out = tf.image.crop_and_resize(both, box, [0], hw, method="bilinear")[0]
        img, mask = out[..., :3], tf.cast(out[..., 3:] > 127.5, tf.float32)
    # photometric: image only (mask untouched)
    img = tf.image.random_brightness(img, 30.0)
    img = tf.image.random_contrast(img, 0.80, 1.20)
    img = tf.clip_by_value(img, 0.0, 255.0)
    x = img / 255.0                                       # saturation/hue need [0,1] range
    x = tf.image.random_saturation(x, 0.8, 1.2)
    x = tf.image.random_hue(x, 0.03)
    img = tf.clip_by_value(x, 0.0, 1.0) * 255.0
    if tf.random.uniform(()) > 0.7:                       # light gaussian noise
        img = img + tf.random.normal(tf.shape(img), 0.0, 5.0)
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, mask


def make_tf_ds(X_uint8, Y_uint8, batch, training=False, repeat=False, seed=42, clahe=False, aug=None):
    """X_uint8 RAW arrays. CLAHE (optional) is applied once up-front, then augmentation, then preprocess_input."""
    aug = TRAIN_AUG if aug is None else aug
    if clahe:
        X_uint8 = np.stack([apply_clahe(x) for x in X_uint8])
    ds = tf.data.Dataset.from_tensor_slices((X_uint8, Y_uint8))
    if training:
        ds = ds.shuffle(len(X_uint8), seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), tf.cast(y, tf.float32)), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(_augment_strong if aug == "strong" else _augment, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (keras.applications.resnet50.preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    if repeat:
        ds = ds.repeat()
    return ds.batch(batch).prefetch(AUTOTUNE)


# ----------------------------------------------------------------------------
# model: plain ResNet-50 U-Net with BN decoder (the single architecture of the paper)
# ----------------------------------------------------------------------------
def _conv_block(x, f, p):
    x = layers.Conv2D(f, 3, padding="same", use_bias=False, name=f"{p}_conv1")(x)
    x = layers.BatchNormalization(name=f"{p}_bn1")(x)
    x = layers.ReLU(name=f"{p}_relu1")(x)
    x = layers.Conv2D(f, 3, padding="same", use_bias=False, name=f"{p}_conv2")(x)
    x = layers.BatchNormalization(name=f"{p}_bn2")(x)
    return layers.ReLU(name=f"{p}_relu2")(x)


GRADCAM_LAYER = "dec4_relu2"          # last decoder block -> use the SAME layer for every model/dataset


def build_unet_resnet50(img_size, weights="imagenet", freeze_encoder=False, skip_connections=True):
    inp = layers.Input((img_size, img_size, 3))
    base = keras.applications.ResNet50(include_top=False, weights=weights, input_tensor=inp)
    if freeze_encoder:
        for l in base.layers:
            l.trainable = False
    s1, s2 = base.get_layer("conv1_relu").output, base.get_layer("conv2_block3_out").output
    s3, s4 = base.get_layer("conv3_block4_out").output, base.get_layer("conv4_block6_out").output
    x = base.output

    def up(x, skip, f, p):
        x = layers.UpSampling2D(name=f"{p}_up")(x)
        if skip_connections:
            x = layers.Concatenate(name=f"{p}_cat")([x, skip])
        return _conv_block(x, f, p)

    x = up(x, s4, 512, "dec1")
    x = up(x, s3, 256, "dec2")
    x = up(x, s2, 128, "dec3")
    x = up(x, s1, 64, "dec4")
    x = layers.UpSampling2D(name="final_up")(x)
    out = layers.Conv2D(1, 1, activation="sigmoid", name="seg_out")(x)
    return keras.Model(inp, out, name="resnet50_unet")


# ----------------------------------------------------------------------------
# loss / metric / training
# ----------------------------------------------------------------------------
def _soft_dice_per_image(y_true, y_pred, eps=1e-6):
    yt = tf.reshape(tf.cast(y_true, tf.float32), (tf.shape(y_true)[0], -1))
    yp = tf.reshape(tf.cast(y_pred, tf.float32), (tf.shape(y_pred)[0], -1))
    return (2 * tf.reduce_sum(yt * yp, 1) + eps) / (tf.reduce_sum(yt, 1) + tf.reduce_sum(yp, 1) + eps)


def bce_dice_loss(y_true, y_pred):
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    return 0.5 * bce + 0.5 * (1.0 - tf.reduce_mean(_soft_dice_per_image(y_true, y_pred)))


def dice_coef(y_true, y_pred):
    """Per-image Dice at 0.5, averaged over the batch (this is what val_dice_coef monitors)."""
    return tf.reduce_mean(_soft_dice_per_image(y_true, tf.cast(y_pred > 0.5, tf.float32)))


def compile_model(model, lr=1e-4, loss=None):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss=loss if loss is not None else bce_dice_loss, metrics=[dice_coef])
    return model


class PerImageValDice(keras.callbacks.Callback):
    """S5: EXACT per-image validation Dice (hard threshold 0.5, every validation image weighted equally, no batch averaging).
    Writes logs["val_dice_img"] every epoch (-> saved in the run's .json history). It is placed BEFORE ModelCheckpoint / ReduceLROnPlateau, so those can monitor it.
    Safety net: if the extra pass ever fails, val_dice_coef is copied instead (training never stops because of this callback)."""

    def __init__(self, val_ds):
        super().__init__()
        self.val_ds = val_ds
        self._warned = False

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            return
        try:
            vals = []
            for batch in self.val_ds:
                xb, yb = batch[0], batch[1]
                if isinstance(yb, (list, tuple)):
                    yb = yb[0]
                p = self.model(xb, training=False)
                if isinstance(p, (list, tuple)):
                    p = p[0]
                vals.append(_soft_dice_per_image(yb, tf.cast(p > 0.5, tf.float32)).numpy())
            logs["val_dice_img"] = float(np.concatenate(vals).mean())
        except Exception as e:
            if not self._warned:
                print("PerImageValDice: extra validation pass failed ->", type(e).__name__, str(e)[:120], "| falling back to val_dice_coef", flush=True)
                self._warned = True
            if "val_dice_coef" in logs:
                logs["val_dice_img"] = float(logs["val_dice_coef"])


def train_model(build_fn, train_ds, val_ds, epochs, ckpt_path, lr=1e-4, lr_patience=5, verbose=2, es_patience=None,
                use_plateau=True, select_best=True):
    """Trains, keeps the epoch with the best VALIDATION Dice (weights-only checkpoint), reloads it, returns (model, history).
    use_plateau=False -> constant learning rate;  select_best=False -> keep the LAST-epoch weights (review M2 control arm).
    Validation metric that selects the checkpoint / drives the LR schedule: Keras batch-average val_dice_coef (default, protocol of all existing runs)
    or the exact per-image val_dice_img when VAL_PER_IMAGE (--val-per-image, review S5). val_dice_img is ALWAYS logged in the history."""
    model = compile_model(build_fn(), lr)
    mon = "val_dice_img" if VAL_PER_IMAGE else "val_dice_coef"
    cbs = [PerImageValDice(val_ds)]                              # must stay FIRST: it creates the log entry that the callbacks below read
    if select_best:
        cbs.append(keras.callbacks.ModelCheckpoint(ckpt_path, monitor=mon, mode="max",
                                                   save_best_only=True, save_weights_only=True))
    if use_plateau:
        cbs.append(keras.callbacks.ReduceLROnPlateau(monitor=mon, mode="max", factor=0.5,
                                                     patience=lr_patience, min_lr=1e-6))
    if es_patience is not None:                                  # default None: sab configs poore epochs chalte hain (reviewer: total epochs match)
        cbs.append(keras.callbacks.EarlyStopping(monitor=mon, mode="max", patience=es_patience, restore_best_weights=False))
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=cbs, verbose=verbose, shuffle=False)
    if select_best:
        model.load_weights(ckpt_path)
    else:
        model.save_weights(ckpt_path)
    hist.history["val_select_per_image"] = [1.0 if VAL_PER_IMAGE else 0.0] * len(hist.history.get("val_dice_coef", hist.history.get("loss", [])))
    return model, hist.history


def best_val_dice(h):
    """best validation Dice of a centralised run from its saved history = the metric that SELECTED its checkpoint
    (per-image val_dice_img if the run was trained with --val-per-image, otherwise the Keras batch-average val_dice_coef)."""
    sel = h.get("val_select_per_image")
    if sel and sel[0] > 0.5 and h.get("val_dice_img"):
        return float(np.max(h["val_dice_img"]))
    return float(np.max(h["val_dice_coef"]))


# ----------------------------------------------------------------------------
# prediction / evaluation
# ----------------------------------------------------------------------------
def predict_probs(model, Xp, bs=8, tta=False, out_idx=0):
    """Low-GPU-memory prediction: only ONE small batch lives on the GPU at a time
    (model.predict(huge_numpy_array) copies the whole array -> out-of-memory on big sets like ColonDB)."""
    def _p(x):
        outs = []
        for i in range(0, len(x), bs):
            p = model.predict_on_batch(np.ascontiguousarray(x[i:i + bs]))
            if isinstance(p, (list, tuple)):
                p = p[out_idx]
            outs.append(np.asarray(p))
        return np.concatenate(outs, 0)
    if not tta:
        return _p(Xp)
    p = _p(Xp)
    p = p + np.flip(_p(np.flip(Xp, 2)), 2)
    p = p + np.flip(_p(np.flip(Xp, 1)), 1)
    p = p + np.flip(np.flip(_p(np.flip(np.flip(Xp, 1), 2)), 1), 2)
    return p / 4.0


def free_gpu_memory():
    """call between big experiments: drops Keras graphs + python garbage"""
    import gc
    keras.backend.clear_session()
    gc.collect()


def evaluate_arrays(model, X_uint8, Y, thr=0.5, clahe=False, tta=False, post=False, bs=8, out_idx=0):
    """Zero-shot / in-domain evaluation on ANY dataset arrays. Returns (summary_dict, per_image_metrics_dict, probs)."""
    probs = predict_probs(model, to_model_input(X_uint8, clahe), bs=bs, tta=tta, out_idx=out_idx)
    m = per_image_metrics(Y, binarize(probs, thr, post))
    return summarize(m), m, probs


def robustness_eval(model, X_uint8, Y, thr=0.5, clahe=False, severities=(0, 1, 2, 3, 4, 5), bs=8, out_idx=0, seed=0):
    """Mean per-image Dice for every corruption x severity (0 = clean reference). Use the FULL test set."""
    x01 = X_uint8.astype(np.float32) / 255.0
    res = {}
    for name in CORRUPTIONS:
        res[name] = []
        for s in severities:
            xc = (corrupt(name, x01, s, seed) * 255.0).round().astype(np.uint8)
            _, m, _ = evaluate_arrays(model, xc, Y, thr, clahe, False, False, bs, out_idx)
            res[name].append(float(m["dice"].mean()))
    return res


# ----------------------------------------------------------------------------
# Grad-CAM++  (Chattopadhyay et al. 2018, closed form)
# ----------------------------------------------------------------------------
def make_gradcam_model(model, layer_name=GRADCAM_LAYER, out_idx=0):
    return keras.Model(model.inputs, [model.get_layer(layer_name).output, model.outputs[out_idx]])


def gradcam_pp(model, x_pre, layer_name=GRADCAM_LAYER, out_idx=0, gm=None):
    """x_pre: ONE preprocessed image (H,W,3). Target score = sum of the predicted polyp probability map.
    Returns (H,W) heat-map in [0,1]. (The old notebook code was plain Grad-CAM: mean of gradients.)"""
    if gm is None:
        gm = make_gradcam_model(model, layer_name, out_idx)
    x = tf.convert_to_tensor(x_pre[None], tf.float32)
    with tf.GradientTape() as tape:
        A, y = gm(x)
        score = tf.reduce_sum(y)
    g = tape.gradient(score, A)
    g2, g3 = g ** 2, g ** 3
    sumA = tf.reduce_sum(A, axis=(1, 2), keepdims=True)
    alpha = tf.where(g != 0.0, g2 / (2.0 * g2 + sumA * g3 + 1e-8), tf.zeros_like(g))
    w = tf.reduce_sum(alpha * tf.nn.relu(g), axis=(1, 2))
    cam = tf.nn.relu(tf.reduce_sum(w[:, None, None, :] * A, axis=-1))[0].numpy()
    cam = cv2.resize(cam, (x_pre.shape[1], x_pre.shape[0]))
    return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)


def make_lime_predict_fn(model, clahe=False, bs=8, out_idx=0):
    """Returns f(batch_uint8)->(B,2) [background, polyp] where polyp = mean predicted polyp probability."""
    def f(batch_uint8):
        p = predict_probs(model, to_model_input(batch_uint8.astype(np.uint8), clahe), bs=bs, out_idx=out_idx)
        frac = p.reshape(len(p), -1).mean(1)
        return np.stack([1 - frac, frac], 1)
    return f


# ----------------------------------------------------------------------------
# Federated learning: FedAvg (standard, BN statistics averaged too) and FedBN
# ----------------------------------------------------------------------------
def bn_weight_indices(model):
    ids = {id(w) for l in model.layers if isinstance(l, layers.BatchNormalization) for w in l.weights}
    return [i for i, w in enumerate(model.weights) if id(w) in ids]


def client_weights(glob, client_bn, bn_idx, c, mode):
    w = list(glob)
    if mode == "fedbn":
        for i in bn_idx:
            w[i] = client_bn[c][i]
    return w


def _make_prox_loss(model, glob, mu):
    """FedProx: local loss + (mu/2) * ||w - w_global||^2 over all trainable weights."""
    idx = {id(w): i for i, w in enumerate(model.weights)}
    pairs = [(v, tf.constant(glob[idx[id(v)]])) for v in model.trainable_variables]

    def prox_loss(y_true, y_pred):
        prox = tf.add_n([tf.reduce_sum(tf.square(v - g)) for v, g in pairs])
        return bce_dice_loss(y_true, y_pred) + 0.5 * mu * prox
    return prox_loss


def run_federated(build_fn, client_ds, client_sizes, client_steps, val_sets, rounds, local_epochs,
                  lr=1e-4, mode="fedavg", seed=42, log=print, mu=0.01):
    """
    client_ds   : list of tf.data (batched, training=True, repeat=True) - one per client
    client_sizes: list of #train images per client (aggregation weights)
    client_steps: steps_per_epoch per client (= size // batch)
    val_sets    : list of validation tf.data (one per client)  - used only for monitoring
    mode        : 'fedavg'  = plain FedAvg, ALL weights incl. BN moving mean/var are averaged (no centralised warm-start!)
                  'fedprox' = FedAvg + proximal term (mu/2)||w - w_global||^2 in every local loss  (mu = `mu`)
                  'fedbn'   = BN layers stay local to every client, everything else is averaged
    Returns dict(glob, client_bn, bn_idx, history). Rebuild a model, then
        model.set_weights(client_weights(glob, client_bn, bn_idx, c, mode))   to evaluate client c.
    """
    set_seed(seed)
    worker = compile_model(build_fn(), lr)               # one reusable worker -> low memory
    bn_idx = bn_weight_indices(worker)
    bn_set = set(bn_idx)
    init = worker.get_weights()
    n = len(client_ds)
    frac = [s / float(sum(client_sizes)) for s in client_sizes]
    glob = [w.copy() for w in init]
    client_bn = [{i: init[i].copy() for i in bn_idx} for _ in range(n)]
    history = []
    for r in range(rounds):
        local = []
        for c in range(n):
            worker.set_weights(client_weights(glob, client_bn, bn_idx, c, mode))
            compile_model(worker, lr, loss=_make_prox_loss(worker, glob, mu) if mode == "fedprox" else None)   # fresh optimiser each round
            worker.fit(client_ds[c], epochs=local_epochs, steps_per_epoch=client_steps[c], verbose=0)
            local.append(worker.get_weights())
        new_glob = []
        for i in range(len(glob)):
            if mode == "fedbn" and i in bn_set:
                new_glob.append(glob[i])
            else:
                new_glob.append(sum(frac[c] * local[c][i] for c in range(n)))
        if mode == "fedbn":
            client_bn = [{i: local[c][i] for i in bn_idx} for c in range(n)]
        glob = new_glob
        dices = []
        for c in range(n):
            worker.set_weights(client_weights(glob, client_bn, bn_idx, c, mode))
            dices.append(float(worker.evaluate(val_sets[c], verbose=0, return_dict=True)["dice_coef"]))
        history.append({"round": r + 1, "val_dice_per_client": dices, "val_dice_mean": float(np.mean(dices))})
        log(f"[{mode}] round {r + 1}/{rounds}  val dice per client: {[round(d, 4) for d in dices]}")
        import gc; gc.collect()
    del worker
    free_gpu_memory()                                     # GPU memory wapas, taake eval_federated ko jagah mile
    return {"glob": glob, "client_bn": client_bn, "bn_idx": bn_idx, "history": history}


# ----------------------------------------------------------------------------
# multi-seed runner: train -> evaluate on SEVERAL test sets -> save per-image Dice/IoU to disk
# ----------------------------------------------------------------------------
def run_seeds(tag, seeds, build_fn, train_arrays, val_arrays, test_sets, epochs, batch,
              out_dir="results", clahe=False, lr=1e-4, verbose=2, resume=True,
              aug=None, use_plateau=True, select_best=True):
    """train_arrays/val_arrays = (X_uint8, Y_uint8); test_sets = {'cvc':(X,Y), 'kvasir':(X,Y), 'etis':(X,Y), ...}
    Saves  results/<tag>_seed<s>.weights.h5   (best-val-Dice weights)
           results/<tag>_seed<s>.npz          (per-image dice__<set>, iou__<set> for every test set)
           results/<tag>_seed<s>.json         (training history)
    SAME image order in every run  ->  paired statistics are valid."""
    os.makedirs(out_dir, exist_ok=True)
    for s in seeds:
        if resume and os.path.exists(os.path.join(out_dir, f"{tag}_seed{s}.weights.h5")) and os.path.exists(os.path.join(out_dir, f"{tag}_seed{s}.npz")):
            print(f"[{tag} seed {s}] pehle se trained (weights + results disk par) -> SKIP, training dobara nahi hogi")
            continue
        set_seed(s)
        tr = make_tf_ds(*train_arrays, batch, training=True, seed=s, clahe=clahe, aug=aug)
        va = make_tf_ds(*val_arrays, batch, training=False, clahe=clahe)
        ck = os.path.join(out_dir, f"{tag}_seed{s}.weights.h5")
        model, hist = train_model(build_fn, tr, va, epochs, ck, lr, verbose=verbose, use_plateau=use_plateau, select_best=select_best)
        pack = {}
        for k, (Xt, Yt) in test_sets.items():
            summ, per, _ = evaluate_arrays(model, Xt, Yt, clahe=clahe, bs=min(batch, 8))
            pack[f"dice__{k}"], pack[f"iou__{k}"] = per["dice"], per["iou"]
            print(f"[{tag} seed {s}] {k:8s} mDice={summ['mDice']:.4f}  mIoU={summ['mIoU']:.4f}  (n={summ['n']})")
        np.savez(os.path.join(out_dir, f"{tag}_seed{s}.npz"), **pack)
        with open(os.path.join(out_dir, f"{tag}_seed{s}.json"), "w") as f:
            json.dump({k: [float(x) for x in v] for k, v in hist.items()}, f)
        keras.backend.clear_session()


def load_dice(tag, seeds, test_key, out_dir="results"):
    """-> list (one array per seed) of per-image Dice, ready for common_utils.compare_configs"""
    return [np.load(os.path.join(out_dir, f"{tag}_seed{s}.npz"))[f"dice__{test_key}"] for s in seeds]


def load_trained(build_fn, ckpt):
    m = build_fn()
    m.load_weights(ckpt)
    return m


def eval_federated(build_fn, out, mode, test_sets, client_for_set=None, clahe=False, bs=8):
    """Evaluate the result of run_federated on several test sets. Returns {set: (dice_per_image, iou_per_image)}.
    fedavg / fedprox : the single global model.
    fedbn  : a test set that belongs to a client (client_for_set={'kvasir':0,'cvc':1}) is evaluated with THAT client's BN;
             every other set (zero-shot) -> metrics averaged over all client models."""
    n = len(out["client_bn"])
    free_gpu_memory()
    m = build_fn()
    res = {}
    for k, (Xt, Yt) in test_sets.items():
        if mode in ("fedavg", "fedprox"):
            clients = [0]
        elif client_for_set and k in client_for_set:
            clients = [client_for_set[k]]
        else:
            clients = list(range(n))
        ds, ious = [], []
        for c in clients:
            m.set_weights(client_weights(out["glob"], out["client_bn"], out["bn_idx"], c, mode))
            _, per, _ = evaluate_arrays(m, Xt, Yt, clahe=clahe, bs=bs)
            ds.append(per["dice"]); ious.append(per["iou"])
        res[k] = (np.mean(ds, 0), np.mean(ious, 0))
    keras.backend.clear_session()
    return res


def save_fed_result(tag, seed, res, out_dir="results"):
    os.makedirs(out_dir, exist_ok=True)
    pack = {}
    for k, (d, i) in res.items():
        pack[f"dice__{k}"], pack[f"iou__{k}"] = d, i
    np.savez(os.path.join(out_dir, f"{tag}_seed{seed}.npz"), **pack)


# ----------------------------------------------------------------------------
# efficiency: params / FLOPs / latency  (re-measure for the NEW architecture & resolution!)
# ----------------------------------------------------------------------------
def efficiency_report(model, img_size, n_warm=10, n_runs=50):
    """Returns dict(params_M, gflops, latency_ms, fps). FLOPs = 2 x multiply-adds counted by the TF profiler
    (if the profiler fails in your TF/Keras version, gflops=None -> use the `keras-flops` package instead).
    NOTE: published tables mostly quote MACs (= FLOPs/2). Check how PraNet/Polyp-PVT numbers were counted and compare in the same unit."""
    import time
    out = {"params_M": model.count_params() / 1e6, "gflops": None}
    try:
        from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2_as_graph
        f = tf.function(lambda x: model(x))
        conc = f.get_concrete_function(tf.TensorSpec([1, img_size, img_size, 3], tf.float32))
        _, graph_def = convert_variables_to_constants_v2_as_graph(conc)
        with tf.Graph().as_default() as g:
            tf.graph_util.import_graph_def(graph_def, name="")
            opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
            opts["output"] = "none"                      # no giant console report
            prof = tf.compat.v1.profiler.profile(g, options=opts)
        out["gflops"] = prof.total_float_ops / 1e9
    except Exception as e:
        print("[efficiency_report] FLOPs not measured:", repr(e)[:120])
    x = np.random.rand(1, img_size, img_size, 3).astype(np.float32)
    for _ in range(n_warm):
        model.predict(x, verbose=0)
    t = time.perf_counter()
    for _ in range(n_runs):
        model.predict(x, verbose=0)
    ms = (time.perf_counter() - t) / n_runs * 1000
    out["latency_ms"], out["fps"] = ms, 1000.0 / ms
    return out


# ----------------------------------------------------------------------------
# qualitative XAI figure for the paper: Image | GT | Prediction | Grad-CAM++ | LIME
# ----------------------------------------------------------------------------
def xai_figure(model, X_uint8, Y, idxs, path=None, clahe=False, lime_samples=300):
    import matplotlib.pyplot as plt
    from lime import lime_image
    idxs = [int(i) for i in idxs]
    Xp = to_model_input(X_uint8[idxs], clahe)
    probs = predict_probs(model, Xp)
    expl = lime_image.LimeImageExplainer(random_state=0)
    pf = make_lime_predict_fn(model, clahe)
    n = len(idxs)
    fig, ax = plt.subplots(n, 5, figsize=(15, 3 * n), squeeze=False)
    for r, i in enumerate(idxs):
        img = X_uint8[i]
        cam = gradcam_pp(model, Xp[r])
        lm = lime_heatmap(expl, img, pf, num_samples=lime_samples)
        lm = lm / (lm.max() + 1e-8)
        ax[r, 0].imshow(img); ax[r, 1].imshow(Y[i, ..., 0], cmap="gray"); ax[r, 2].imshow(probs[r, ..., 0] > 0.5, cmap="gray")
        ax[r, 3].imshow(img); ax[r, 3].imshow(cam, cmap="jet", alpha=0.45)
        ax[r, 4].imshow(img); ax[r, 4].imshow(lm, cmap="jet", alpha=0.45)
        for c, t in enumerate(["Image", "Ground truth", "Prediction", "Grad-CAM++", "LIME"]):
            ax[r, c].axis("off")
            if r == 0:
                ax[r, c].set_title(t)
    plt.tight_layout(); plt.show()


# ----------------------------------------------------------------------------
# save federated models to disk (so XAI / robustness / HD95 cells can load them later, standalone)
# ----------------------------------------------------------------------------
def save_fed_models(build_fn, out, mode, prefix):
    """fedavg / fedprox -> prefix.weights.h5 (one global model);  fedbn -> prefix_client<c>.weights.h5 (global + client's own BN)."""
    m = build_fn()
    clients = [0] if mode in ("fedavg", "fedprox") else list(range(len(out["client_bn"])))
    for c in clients:
        m.set_weights(client_weights(out["glob"], out["client_bn"], out["bn_idx"], c, mode))
        m.save_weights(prefix + (".weights.h5" if mode in ("fedavg", "fedprox") else f"_client{c}.weights.h5"))
    keras.backend.clear_session()


# ----------------------------------------------------------------------------
# XAI protocol v2: Grad-CAM++ / LIME + baselines + deletion/insertion AUC + model-randomisation sanity check
# ----------------------------------------------------------------------------
def _trapz(y, x):
    y, x = np.asarray(y, float), np.asarray(x, float)
    return float(np.sum((y[1:] + y[:-1]) / 2.0 * np.diff(x)))


def _dice_bin(a, b):
    a, b = a.astype(bool), b.astype(bool)
    return 2.0 * (a & b).sum() / (a.sum() + b.sum() + 1e-8)


def deletion_insertion(model, img_uint8, heatmap, steps=10, clahe=False, bs=22):
    """Deletion/insertion AUC for a segmentation model (Petsiuk et al., adapted).
    Pixels are ranked by the heat-map. Score at every step = Dice between the prediction on the modified image and the
    prediction on the ORIGINAL image. deletion: remove the most important pixels first (LOWER AUC = better explanation);
    insertion: reveal the most important pixels first on a blurred image (HIGHER AUC = better). Returns NaN if the model
    predicts nothing on the original image."""
    H, W = img_uint8.shape[:2]
    base = cv2.GaussianBlur(img_uint8, (0, 0), sigmaX=max(H, W) / 12.0)
    flat = np.asarray(heatmap, float).reshape(-1)
    order = np.argsort(-(flat + 1e-9 * np.random.default_rng(0).random(flat.shape)))
    rank = np.empty_like(order)
    rank[order] = np.arange(H * W)
    fr = np.linspace(0.0, 1.0, steps + 1)
    dele, ins = [], []
    for f in fr:
        m = (rank < int(round(f * H * W))).reshape(H, W, 1)
        dele.append(np.where(m, base, img_uint8))
        ins.append(np.where(m, img_uint8, base))
    probs = predict_probs(model, to_model_input(np.stack(dele + ins).astype(np.uint8), clahe), bs=bs) > 0.5
    ref = probs[0]
    if not ref.any():
        return {"deletion_auc": np.nan, "insertion_auc": np.nan}
    n = len(fr)
    d = [_dice_bin(probs[i], ref) for i in range(n)]
    i_ = [_dice_bin(probs[n + i], ref) for i in range(n)]
    return {"deletion_auc": _trapz(d, fr), "insertion_auc": _trapz(i_, fr)}


def xai_full_eval(model, X_uint8, Y, n_gc=None, n_lime=50, lime_segments=150, lime_samples=1000, clahe=False,
                  with_delins=True, seed=0):
    """ONE protocol for every model (centralised or federated). Images are a seeded random subset of X.
    Returns {method: DataFrame(per-image metrics)}: gradcam++, center_prior, random, lime, gradcam++ deletion/insertion."""
    from lime import lime_image
    n = len(X_uint8) if n_gc is None else min(n_gc, len(X_uint8))
    idx = np.random.default_rng(seed).permutation(len(X_uint8))[:n]
    Xp = to_model_input(X_uint8[idx], clahe)
    expl = lime_image.LimeImageExplainer(random_state=0)
    pf = make_lime_predict_fn(model, clahe)
    H, W = X_uint8.shape[1:3]
    gc, cp, rd, lm, di = [], [], [], [], []
    for j, i in enumerate(idx):
        gt = Y[i]
        cam = gradcam_pp(model, Xp[j])
        gc.append(xai_metrics(cam, gt))
        cp.append(xai_metrics(center_prior(H, W), gt))
        rd.append(xai_metrics(random_map(H, W, int(i)), gt))
        if with_delins:
            di.append(deletion_insertion(model, X_uint8[i], cam, clahe=clahe))
        if j < n_lime:
            lm.append(xai_metrics(lime_heatmap(expl, X_uint8[i], pf, num_samples=lime_samples, n_segments=lime_segments), gt))
    out = {"gradcam++": pd.DataFrame(gc), "center_prior": pd.DataFrame(cp), "random": pd.DataFrame(rd)}
    if lm:
        out["lime"] = pd.DataFrame(lm)
    if di:
        out["gradcam++ deletion/insertion"] = pd.DataFrame(di)
    return out


def xai_report(res, n_boot=2000):
    """mean and 95% bootstrap CI for every metric of every method (output of xai_full_eval)."""
    rows = []
    for method, df in res.items():
        for col in df.columns:
            v = pd.to_numeric(df[col].astype(float), errors="coerce").dropna().values
            if len(v) == 0:
                continue
            lo, hi = bootstrap_ci(v, n_boot=n_boot)
            rows.append({"method": method, "metric": col, "mean": round(float(v.mean()), 4),
                         "ci95_low": round(lo, 4), "ci95_high": round(hi, 4), "n": len(v)})
    return pd.DataFrame(rows)


def randomization_check(trained_model, build_rand_fn, X_uint8, Y, n=20, clahe=False, seed=0):
    """Sanity check (Adebayo et al.): a good explanation must CHANGE when the model weights are randomised.
    random_all     = completely random weights;   random_decoder = trained encoder + randomly initialised decoder.
    Reports Spearman correlation between the trained-model heat-map and the randomised-model heat-map
    (should be LOW) and AUPRC of both heat-maps against the GT (randomised should be clearly lower than trained).
    build_rand_fn = lambda: build_unet_resnet50(IMG_SIZE, weights=None)"""
    from scipy.stats import spearmanr
    rand_all, rand_dec = build_rand_fn(), build_rand_fn()
    for l in trained_model.layers:
        if l.name.startswith(("dec", "final", "seg_out")) or not l.weights:
            continue
        try:
            rand_dec.get_layer(l.name).set_weights(l.get_weights())
        except (ValueError, KeyError):
            pass
    idx = np.random.default_rng(seed).permutation(len(X_uint8))[:n]
    Xp = to_model_input(X_uint8[idx], clahe)
    rows = []
    for j, i in enumerate(idx):
        cam_t = gradcam_pp(trained_model, Xp[j])
        r = {"auprc_trained": xai_metrics(cam_t, Y[i])["auprc"]}
        for name, m in (("random_all", rand_all), ("random_decoder", rand_dec)):
            cam_r = gradcam_pp(m, Xp[j])
            rho = spearmanr(cam_t.ravel(), cam_r.ravel())[0] if cam_r.std() > 0 and cam_t.std() > 0 else np.nan
            r[f"spearman_{name}"] = float(rho)
            r[f"auprc_{name}"] = xai_metrics(cam_r, Y[i])["auprc"]
        rows.append(r)
    return pd.DataFrame(rows)

# ----------------------------------------------------------------------------
# 1. complex architecture (ablation only) — matches the paper's Kvasir branch
# ----------------------------------------------------------------------------
def _cbam(x, f, p, ratio=8):
    """Convolutional Block Attention Module: channel attention (shared MLP on avg+max pool) then
    spatial attention (7x7 conv on channel-pooled maps)."""
    avg = layers.GlobalAveragePooling2D(keepdims=True, name=f"{p}_cbam_avg")(x)
    mx = layers.GlobalMaxPooling2D(keepdims=True, name=f"{p}_cbam_max")(x)
    d1 = layers.Dense(max(1, f // ratio), activation="relu", name=f"{p}_cbam_fc1")
    d2 = layers.Dense(f, name=f"{p}_cbam_fc2")
    ch = layers.Activation("sigmoid", name=f"{p}_cbam_chatt")(layers.Add()([d2(d1(avg)), d2(d1(mx))]))
    x = layers.Multiply(name=f"{p}_cbam_chmul")([x, ch])
    avg_s = layers.Lambda(lambda t: tf.reduce_mean(t, axis=-1, keepdims=True), name=f"{p}_cbam_savg")(x)
    max_s = layers.Lambda(lambda t: tf.reduce_max(t, axis=-1, keepdims=True), name=f"{p}_cbam_smax")(x)
    sp = layers.Conv2D(1, 7, padding="same", activation="sigmoid", name=f"{p}_cbam_spconv")(
        layers.Concatenate(name=f"{p}_cbam_scat")([avg_s, max_s]))
    return layers.Multiply(name=f"{p}_cbam_spmul")([x, sp])


def _attention_gate(x, skip, inter, p):
    """Additive attention gate: suppress irrelevant skip activations before concatenation (Attention U-Net)."""
    theta_x = layers.Conv2D(inter, 1, name=f"{p}_ag_theta")(x)
    phi_g = layers.Conv2D(inter, 1, name=f"{p}_ag_phi")(skip)
    add = layers.ReLU(name=f"{p}_ag_relu")(layers.Add(name=f"{p}_ag_add")([theta_x, phi_g]))
    psi = layers.Conv2D(1, 1, activation="sigmoid", name=f"{p}_ag_psi")(add)
    return layers.Multiply(name=f"{p}_ag_mul")([skip, psi])


def _residual_block(x, f, p):
    sc = layers.Conv2D(f, 1, padding="same", name=f"{p}_sc")(x) if x.shape[-1] != f else x
    y = layers.Conv2D(f, 3, padding="same", use_bias=False, name=f"{p}_conv1")(x)
    y = layers.BatchNormalization(name=f"{p}_bn1")(y)
    y = layers.ReLU(name=f"{p}_relu1")(y)
    y = layers.Conv2D(f, 3, padding="same", use_bias=False, name=f"{p}_conv2")(y)
    y = layers.BatchNormalization(name=f"{p}_bn2")(y)
    y = layers.Add(name=f"{p}_resadd")([sc, y])
    return layers.ReLU(name=f"{p}_relu2")(y)


def _aspp(x, out_ch=256, name="aspp"):
    """Atrous Spatial Pyramid Pooling: global-avg-pool branch + three dilated 3x3 convs (rates 6,12,18) + 1x1."""
    h, w = x.shape[1], x.shape[2]
    gp = layers.GlobalAveragePooling2D(keepdims=True, name=f"{name}_gp")(x)
    gp = layers.Conv2D(out_ch, 1, padding="same", use_bias=False, name=f"{name}_gpconv")(gp)
    gp = layers.BatchNormalization(name=f"{name}_gpbn")(gp)
    gp = layers.ReLU(name=f"{name}_gprelu")(gp)
    gp = layers.Resizing(h, w, interpolation="bilinear", name=f"{name}_gpresize")(gp)
    branches = [gp]
    for i, rate in enumerate((6, 12, 18)):
        b = layers.Conv2D(out_ch, 3, padding="same", dilation_rate=rate, use_bias=False, name=f"{name}_d{i}_conv")(x)
        b = layers.BatchNormalization(name=f"{name}_d{i}_bn")(b)
        b = layers.ReLU(name=f"{name}_d{i}_relu")(b)
        branches.append(b)
    cat = layers.Concatenate(name=f"{name}_cat")(branches)
    out = layers.Conv2D(out_ch, 1, padding="same", use_bias=False, name=f"{name}_proj")(cat)
    out = layers.BatchNormalization(name=f"{name}_projbn")(out)
    return layers.ReLU(name=f"{name}_projrelu")(out)


def build_complex_kvasir(img_size, weights="imagenet", deep_supervision=True,
                         use_cbam=True, use_attention_gate=True, name="complex_kvasir"):
    """ResNet-50 encoder + ASPP bottleneck + (optional attention-gated) residual decoder + (optional)
    CBAM + (optional) deep supervision. This single function builds the FULL model and every
    component-removed ablation variant used in K7 (review 2.3's replication protocol: -CBAM,
    -Attention Gate, -Deep Supervision, each removed one at a time from the same base):
      use_cbam=False            -> "-CBAM"            (skip the CBAM refinement block)
      use_attention_gate=False  -> "-Attention Gate"   (plain concat instead of the gated skip)
      deep_supervision=False    -> "-Deep Supervision" (single output, no auxiliary heads)
    Main output name 'seg_out' (same as the plain model); if deep_supervision=True, also returns 3
    auxiliary sigmoid heads named 'ds1','ds2','ds3' (combo loss weights 1.0/0.4/0.3/0.2 on
    main/ds1/ds2/ds3; tf_utils reads the main head via out_idx=0 everywhere, so every other helper
    — evaluate_arrays, xai_full_eval, gradcam_pp, predict_probs — works unchanged on every variant)."""
    inp = layers.Input((img_size, img_size, 3))
    base = tf.keras.applications.ResNet50(include_top=False, weights=weights, input_tensor=inp)
    s1 = base.get_layer("conv1_relu").output
    s2 = base.get_layer("conv2_block3_out").output
    s3 = base.get_layer("conv3_block4_out").output
    s4 = base.get_layer("conv4_block6_out").output
    x = _aspp(base.output, 256)

    ds_outputs = []
    filters = [256, 128, 64, 32]
    skips = [s4, s3, s2, s1]
    for i, (f, skip) in enumerate(zip(filters, skips), start=1):
        p = f"dec{i}"
        x = layers.UpSampling2D(name=f"{p}_up")(x)
        skip_in = _attention_gate(x, skip, max(f // 2, 8), p) if use_attention_gate else skip
        x = layers.Concatenate(name=f"{p}_cat")([x, skip_in])
        x = _residual_block(x, f, f"{p}_res")              # internal layer names: "{p}_res_*" (no clash below)
        if use_cbam:
            x = _cbam(x, f, f"{p}_cb")                      # internal layer names: "{p}_cb_*"
        x = layers.Activation("linear", name=f"{p}_relu2")(x)   # one clean, PREDICTABLE name per stage;
        # stage 4's tensor is named "dec4_relu2" -- the SAME name the plain model gives its last block,
        # so gradcam_pp's default GRADCAM_LAYER="dec4_relu2" (tf_utils.py) targets every variant identically.
        if deep_supervision and i >= 2:                    # attach aux heads to the 3 shallower levels
            aux = layers.Conv2D(1, 1, activation="sigmoid", name=f"ds{i - 1}_raw")(x)
            aux = layers.Resizing(img_size, img_size, interpolation="bilinear", name=f"ds{i - 1}")(aux)
            ds_outputs.append(aux)
    x = layers.UpSampling2D(name="final_up")(x)
    out = layers.Conv2D(1, 1, activation="sigmoid", name="seg_out")(x)
    outputs = [out] + ds_outputs if deep_supervision else out
    return tf.keras.Model(inp, outputs, name=name)


# ----------------------------------------------------------------------------
# 2. Precision-Recall curve (threshold-free) + headline metrics bar chart
# ----------------------------------------------------------------------------
def pr_curve_and_bar(model, X_uint8, Y, summ, clahe=False, bs=8, out_idx=0, title=""):
    """Draws, side by side:
       (left)  pixel-wise Precision-Recall curve over ALL test pixels + AUPRC (threshold-free;
               the 0.5 operating point used elsewhere is marked with a dot)
       (right) bar chart of the headline per-image-averaged metrics already in `summ`
               (Accuracy/Precision/Recall/Dice/IoU/F1 — same shape as the paper's original Fig. 15/34)
    `summ` = the dict returned by tf_utils.evaluate_arrays's first element (mAccuracy/mPrecision/...).
    Subsamples pixels for the curve (max 2M) so it stays fast on the full test set."""
    probs = predict_probs(model, to_model_input(X_uint8, clahe), bs=bs, out_idx=out_idx)
    y_true = (Y.reshape(-1) > 0.5).astype(int)
    y_score = probs.reshape(-1).astype(np.float32)
    if len(y_true) > 2_000_000:
        rng = np.random.default_rng(0)
        idx = rng.choice(len(y_true), 2_000_000, replace=False)
        y_true, y_score = y_true[idx], y_score[idx]
    prec, rec, _ = precision_recall_curve(y_true, y_score)
    ap = average_precision_score(y_true, y_score)

    fig, axs = plt.subplots(1, 2, figsize=(12, 4.6))
    axs[0].plot(rec, prec, lw=2)
    axs[0].scatter([summ["mRecall"]], [summ["mPrecision"]], color="red", zorder=5,
                   label=f"threshold=0.5 (P={summ['mPrecision']:.3f}, R={summ['mRecall']:.3f})")
    axs[0].set_xlabel("Recall"); axs[0].set_ylabel("Precision")
    axs[0].set_title(f"Precision-Recall curve (AUPRC={ap:.3f})"); axs[0].legend(loc="lower left")
    axs[0].set_xlim(0, 1); axs[0].set_ylim(0, 1.02); axs[0].grid(alpha=0.3)

    keys = ["mAccuracy", "mPrecision", "mRecall", "mDice", "mIoU"]
    labels = ["Accuracy", "Precision", "Recall", "Dice", "IoU"]
    vals = [summ[k] for k in keys]
    # Dice and F1 are mathematically identical for binary segmentation (2TP/(2TP+FP+FN) either way);
    # shown as a separate bar only because the polyp-segmentation literature reports both by convention.
    keys.append("mF1 (= Dice)"); labels.append("F1"); vals.append(summ["mDice"])
    axs[1].bar(labels, vals, color="#4C72B0")
    for i, v in enumerate(vals):
        axs[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
    axs[1].set_ylim(0, 1.05); axs[1].set_title(title or "Headline test-set metrics"); axs[1].grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.show()
    return {"auprc": float(ap)}


def hd95_report(model, X_uint8, Y, clahe=False, bs=8, out_idx=0, thr=0.5, post=False, tta=False, label=""):
    """95th-percentile Hausdorff Distance (pixels; lower = tighter boundaries) — review §3: 'Lead with
    mDice and mIoU; add HD95'. Prints mean +- 95% bootstrap CI and the number of images skipped
    (empty prediction or empty ground truth, where HD95 is undefined and penalised at the image
    diagonal so a degenerate all-background model does not look artificially good)."""
    probs = predict_probs(model, to_model_input(X_uint8, clahe), bs=bs, tta=tta, out_idx=out_idx)
    pred = binarize(probs, thr, post)
    hd, n_empty = hd95_batch(Y, pred)
    lo, hi = bootstrap_ci(hd)
    print(f"{label}HD95 = {hd.mean():.2f} px  (95% CI [{lo:.2f}, {hi:.2f}])"
          f"  | {n_empty}/{len(hd)} images had an empty prediction or mask (penalised at the image diagonal)")
    return {"hd95_mean": float(hd.mean()), "hd95_CI95": (lo, hi), "n_empty": int(n_empty)}


def tune_threshold(model, Xv, Yv, clahe=False, bs=8, out_idx=0, tta=False, post=False, lo=0.2, hi=0.8, steps=31):
    """Grid search over `steps` thresholds in [lo, hi] on the VALIDATION set (same protocol the paper
    itself used: '31 candidate thresholds in [0.2, 0.8]'). Returns (best_threshold, best_val_mDice)."""
    probs = predict_probs(model, to_model_input(Xv, clahe), bs=bs, tta=tta, out_idx=out_idx)
    best_t, best_d = 0.5, -1.0
    for t in np.linspace(lo, hi, steps):
        d = per_image_metrics(Yv, binarize(probs, t, post))["dice"].mean()
        if d > best_d:
            best_t, best_d = float(t), float(d)
    return best_t, best_d


def tta_post_threshold_ablation(model, Xv, Yv, Xt, Yt, clahe=False, bs=8, out_idx=0):
    """Isolates, on the SAME trained weights (no retraining), how much of any Dice difference comes from
    inference-time choices rather than the architecture itself (review §2.3: 'the real difference must
    be TTA, post-processing, threshold, or split, and that needs an honest explanation'):
      (a) default      -- threshold 0.5, no TTA, no post-processing
      (b) + TTA         -- 4-way flip-averaged prediction
      (c) + post-proc   -- largest-connected-component cleanup (threshold still 0.5)
      (d) + full recipe -- TTA + post-processing + a threshold grid-searched on the VALIDATION set
    Returns a DataFrame; also prints the tuned threshold found in (d)."""
    import pandas as pd
    rows = []
    s, _, _ = evaluate_arrays(model, Xt, Yt, clahe=clahe, bs=bs, out_idx=out_idx)
    rows.append({"setting": "(a) default (thr=0.5, no TTA, no post)", "mDice": s["mDice"], "mIoU": s["mIoU"]})
    s, _, _ = evaluate_arrays(model, Xt, Yt, clahe=clahe, bs=bs, out_idx=out_idx, tta=True)
    rows.append({"setting": "(b) + TTA (4-way flip)", "mDice": s["mDice"], "mIoU": s["mIoU"]})
    s, _, _ = evaluate_arrays(model, Xt, Yt, clahe=clahe, bs=bs, out_idx=out_idx, post=True)
    rows.append({"setting": "(c) + largest-component post-processing", "mDice": s["mDice"], "mIoU": s["mIoU"]})
    best_t, val_d = tune_threshold(model, Xv, Yv, clahe=clahe, bs=bs, out_idx=out_idx, tta=True, post=True)
    s, _, _ = evaluate_arrays(model, Xt, Yt, clahe=clahe, bs=bs, out_idx=out_idx, thr=best_t, tta=True, post=True)
    rows.append({"setting": f"(d) + TTA + post + val-tuned threshold (thr={best_t:.3f}, val mDice={val_d:.3f})",
                 "mDice": s["mDice"], "mIoU": s["mIoU"]})
    df = pd.DataFrame(rows).round(4)
    df["Δ vs (a)"] = (df["mDice"] - df.loc[0, "mDice"]).round(4)
    return df


# ----------------------------------------------------------------------------
# 3. OPTIONAL: train the deep-supervision variant (multi-output). Self-contained, bespoke
#    training loop (does NOT touch tf_utils.run_seeds / train_model — those stay single-output
#    for the plain model and the no-deep-supervision complex model, zero risk of breaking them).
# ----------------------------------------------------------------------------
def run_seeds_complex_ds(tag, seeds, img_size, train_arrays, val_arrays, test_sets, epochs, batch,
                         out_dir="results", lr=1e-4, verbose=2, weights="imagenet"):
    """Same bookkeeping as tf_utils.run_seeds (best-val-Dice checkpoint, resume, per-test-set
    dice/iou saved to <tag>_seed<s>.npz) but for the 4-output deep-supervision model."""
    import json
    os.makedirs(out_dir, exist_ok=True)
    to_multi = lambda x, y: (x, {"seg_out": y, "ds1": y, "ds2": y, "ds3": y})
    for s in seeds:
        ck = os.path.join(out_dir, f"{tag}_seed{s}.weights.h5")
        npz = os.path.join(out_dir, f"{tag}_seed{s}.npz")
        if os.path.exists(ck) and os.path.exists(npz):
            print(f"[{tag} seed {s}] pehle se trained -> SKIP"); continue
        set_seed(s)
        tr = make_tf_ds(*train_arrays, batch, training=True, seed=s).map(to_multi)
        va = make_tf_ds(*val_arrays, batch).map(to_multi)
        model = build_complex_kvasir(img_size, weights=weights, deep_supervision=True)
        model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                      loss={"seg_out": bce_dice_loss, "ds1": bce_dice_loss, "ds2": bce_dice_loss, "ds3": bce_dice_loss},
                      loss_weights={"seg_out": 1.0, "ds1": 0.4, "ds2": 0.3, "ds3": 0.2},
                      metrics={"seg_out": [dice_coef]})
        mon = "val_seg_out_dice_coef"
        cbs = [tf.keras.callbacks.ModelCheckpoint(ck, monitor=mon, mode="max", save_best_only=True, save_weights_only=True),
               tf.keras.callbacks.ReduceLROnPlateau(monitor=mon, mode="max", factor=0.5, patience=5, min_lr=1e-6)]
        model.fit(tr, validation_data=va, epochs=epochs, callbacks=cbs, verbose=verbose)
        model.load_weights(ck)
        pack = {}
        for k, (Xt, Yt) in test_sets.items():
            summ, per, _ = evaluate_arrays(model, Xt, Yt, bs=batch * 2, out_idx=0)   # out_idx=0 -> main 'seg_out' head
            pack[f"dice__{k}"], pack[f"iou__{k}"] = per["dice"], per["iou"]
            print(f"[{tag} seed {s}] {k:8s} mDice={summ['mDice']:.4f}")
        np.savez(npz, **pack)
        tf.keras.backend.clear_session()

# ================================================================================================================
# ================================================================================================================
#                                   NEW HELPERS  (added for the final critical review)
# ================================================================================================================
# ================================================================================================================
UNSEEN = ("cvc300", "colondb", "etis")          # sites never used for training -> pooled into "unseen (image-weighted)" (S1)


# ----------------------------------------------------------------------------------------------------------------
# M3 / S1 : ONE statistic for the p-value AND the interval (both are about the MEAN per-image difference)
# ----------------------------------------------------------------------------------------------------------------
def sign_flip_p(d, n_perm=20000, seed=0):
    """Two-sided paired sign-flip (permutation) test on the MEAN of d = a - b. Exact when n <= 16, Monte-Carlo otherwise.
    It tests exactly the quantity whose bootstrap interval we report -> test and interval can no longer disagree (review M3)."""
    d = np.asarray(d, dtype=np.float64)
    d = d[np.isfinite(d)]
    n = len(d)
    if n == 0 or np.allclose(d, 0.0):
        return 1.0
    obs = abs(d.mean())
    if n <= 16:
        signs = np.array(list(itertools.product((-1.0, 1.0), repeat=n)))
        return float((np.abs((signs * d).mean(1)) >= obs - 1e-12).mean())
    rng = np.random.default_rng(seed)
    hit, done = 0, 0
    while done < n_perm:
        k = min(2000, n_perm - done)
        s = rng.integers(0, 2, (k, n)) * 2.0 - 1.0
        hit += int((np.abs((s * d).mean(1)) >= obs - 1e-12).sum())
        done += k
    return (hit + 1.0) / (n_perm + 1.0)


def hodges_lehmann(d):
    """Hodges-Lehmann pseudo-median of the paired differences (the effect size that goes with the Wilcoxon test)."""
    d = np.asarray(d, dtype=np.float64)
    iu = np.triu_indices(len(d))
    return float(np.median((d[:, None] + d[None, :])[iu] / 2.0))


def hl_bootstrap_ci(d, n_boot=500, alpha=0.05, seed=0):
    d = np.asarray(d, dtype=np.float64)
    rng = np.random.default_rng(seed)
    n = len(d)
    iu = np.triu_indices(n)
    vals = np.empty(n_boot)
    for b in range(n_boot):
        x = d[rng.integers(0, n, n)]
        vals[b] = np.median((x[:, None] + x[None, :])[iu] / 2.0)
    return float(np.percentile(vals, 100 * alpha / 2)), float(np.percentile(vals, 100 * (1 - alpha / 2)))


def compare_v2(per_image, ref, n_boot=10000, n_perm=20000, hl_boot=500, seed=0):
    """per_image: {config: [array(N,) per seed]} (SAME image order everywhere). Statistical unit = IMAGE (seed-averaged).
    Columns: mean mDice, SD over seeds, mean difference vs ref, UNADJUSTED 95% bootstrap CI of the mean difference,
    Bonferroni-ADJUSTED CI (level 1-0.05/k), sign-flip p on the mean difference (raw + Holm), Wilcoxon p (raw + Holm),
    Hodges-Lehmann median difference with its own bootstrap CI, and an agreement flag (raw sign-flip p < 0.05  <=>  unadjusted CI excludes 0). Holm p and Bonferroni CI are the multiplicity-adjusted pair."""
    names = [k for k in per_image if k != ref]
    avg = {k: np.mean(np.stack(per_image[k]), 0) for k in per_image}
    seed_means = lambda k: np.array([a.mean() for a in per_image[k]])
    sd = lambda k: float(seed_means(k).std(ddof=1)) if len(per_image[k]) > 1 else float("nan")
    rows = [{"config": ref, "mDice_mean": float(seed_means(ref).mean()), "std_over_seeds": sd(ref), "diff_vs_ref": 0.0,
             "CI95_unadj": "-", "CI95_bonf": "-", "p_perm": np.nan, "p_perm_holm": np.nan, "p_wilcoxon": np.nan,
             "p_wilcoxon_holm": np.nan, "HL_median_diff": np.nan, "HL_CI95": "-", "agree": ""}]
    if not names:
        return pd.DataFrame(rows)
    k_comp = len(names)
    tmp = []
    for k in names:
        d = avg[k] - avg[ref]
        lo, hi = bootstrap_ci(d, n_boot=n_boot, alpha=0.05, seed=seed)
        lo_a, hi_a = bootstrap_ci(d, n_boot=n_boot, alpha=0.05 / k_comp, seed=seed)
        hl = hodges_lehmann(d)
        hlo, hhi = hl_bootstrap_ci(d, n_boot=hl_boot, seed=seed)
        tmp.append(dict(k=k, d=d, ci=(lo, hi), ci_a=(lo_a, hi_a), p=sign_flip_p(d, n_perm, seed),
                        pw=paired_wilcoxon(avg[ref], avg[k]), hl=hl, hl_ci=(hlo, hhi)))
    ph = holm([t["p"] for t in tmp])
    pwh = holm([t["pw"] for t in tmp])
    for t, a, b in zip(tmp, ph, pwh):
        sig = t["p"] < 0.05                                             # raw test and raw interval are about the SAME statistic (the mean)
        excl = (t["ci"][0] > 0) or (t["ci"][1] < 0)
        rows.append({"config": t["k"], "mDice_mean": float(seed_means(t["k"]).mean()), "std_over_seeds": sd(t["k"]),
                     "diff_vs_ref": float(t["d"].mean()), "CI95_unadj": f"[{t['ci'][0]:+.3f}, {t['ci'][1]:+.3f}]",
                     "CI95_bonf": f"[{t['ci_a'][0]:+.3f}, {t['ci_a'][1]:+.3f}]", "p_perm": t["p"], "p_perm_holm": float(a),
                     "p_wilcoxon": t["pw"], "p_wilcoxon_holm": float(b), "HL_median_diff": t["hl"],
                     "HL_CI95": f"[{t['hl_ci'][0]:+.3f}, {t['hl_ci'][1]:+.3f}]", "agree": "yes" if sig == excl else "NO"})
    return pd.DataFrame(rows)


def compare_table(per, ref, **kw):
    """notebook-compatible name: now returns the consistent (M3) table."""
    t = compare_v2(per, ref, **kw)
    num = [c for c in t.columns if c.startswith(("p_", "mDice", "std", "diff", "HL_median"))]
    t[num] = t[num].astype(float).round(4)
    return t


def load_per_image(tag, folder, key, seeds=None):
    """[array per seed] of per-image Dice for a test set key, or key='unseen' = CVC-300 + CVC-ColonDB + ETIS concatenated (636 images)."""
    seeds = SEEDS if seeds is None else seeds
    if key == "unseen":
        return [np.concatenate([np.load(os.path.join(folder, f"{tag}_seed{s}.npz"))[f"dice__{k}"] for k in UNSEEN]) for s in seeds]
    return load_dice(tag, seeds, key, folder)


def has_results(tag, folder, seeds=None, keys=None):
    seeds = SEEDS if seeds is None else seeds
    for s in seeds:
        p = os.path.join(folder, f"{tag}_seed{s}.npz")
        if not os.path.exists(p):
            return False
        if keys:
            z = np.load(p)
            if not all(f"dice__{k}" in z.files for k in keys):
                return False
    return True


# ----------------------------------------------------------------------------------------------------------------
# S6 / M8 : field-of-view hashing, SSIM confirmation, hash-graph groups (= video-sequence proxy)
# ----------------------------------------------------------------------------------------------------------------
def crop_fov(img_bgr, thr=12, min_frac=0.10):
    """Crop to the endoscopic field of view (drop the black border) so that the hash is not dominated by the black frame."""
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    m = g > thr
    if m.mean() < 0.2:
        return img_bgr
    ys, xs = np.where(m.mean(1) > min_frac)[0], np.where(m.mean(0) > min_frac)[0]
    if len(ys) < 16 or len(xs) < 16:
        return img_bgr
    return img_bgr[ys[0]:ys[-1] + 1, xs[0]:xs[-1] + 1]


def fov_hash_bits(path):
    return phash_bits(crop_fov(cv2.imread(path)))


_GRAY_CACHE = {}


def fov_gray(path, size=128):
    if path not in _GRAY_CACHE:
        g = cv2.cvtColor(crop_fov(cv2.imread(path)), cv2.COLOR_BGR2GRAY)
        _GRAY_CACHE[path] = cv2.resize(g, (size, size), interpolation=cv2.INTER_AREA)
    return _GRAY_CACHE[path]


def ssim_gray(a, b):
    from skimage.metrics import structural_similarity
    return float(structural_similarity(a, b, data_range=255))


def audit_near_duplicates(train_paths, test_paths, max_dist=8, min_ssim=0.9, topk=5):
    """Per test image: nearest train image by FOV-cropped pHash; if within max_dist, SSIM (128x128 FOV crops) against the top-k
    hash neighbours. hash_flag = hash distance <= max_dist ; confirmed = hash_flag AND best SSIM >= min_ssim."""
    ha = np.stack([fov_hash_bits(p) for p in train_paths])
    hb = np.stack([fov_hash_bits(p) for p in test_paths])
    dist = (ha[:, None, :] != hb[None, :, :]).sum(2)
    rows = []
    for j, p in enumerate(test_paths):
        col = dist[:, j]
        order = np.argsort(col, kind="stable")[:topk]
        i0 = int(order[0])
        best_i, best_s = i0, float("nan")
        if col[i0] <= max_dist:
            gj, best_s = fov_gray(p), -1.0
            for i in order:
                if col[i] > max_dist:
                    break
                s = ssim_gray(gj, fov_gray(train_paths[int(i)]))
                if s > best_s:
                    best_s, best_i = s, int(i)
        flag = bool(col[i0] <= max_dist)
        rows.append({"test_image": os.path.basename(p), "nearest_train": os.path.basename(train_paths[best_i]),
                     "test_path": p, "train_path": train_paths[best_i], "hash_dist": int(col[i0]), "ssim": best_s,
                     "hash_flag": flag, "confirmed": bool(flag and best_s >= min_ssim)})
    return pd.DataFrame(rows)


def flagged_pairs_montage(audit, name, per_page=12, max_pages=3, cols=4):
    """Montage of flagged pairs (test | nearest train) with hash distance and SSIM -> supplementary figure (review S6)."""
    sel = audit[audit["hash_flag"]].sort_values(["confirmed", "hash_dist"], ascending=[False, True]).reset_index(drop=True)
    if sel.empty:
        print(f"[{name}] no flagged pairs -> no montage")
        return
    for page in range(min(max_pages, int(np.ceil(len(sel) / per_page)))):
        part = sel.iloc[page * per_page:(page + 1) * per_page]
        rws = int(np.ceil(len(part) / cols))
        fig, axs = plt.subplots(rws, cols * 2, figsize=(3.2 * cols * 2 / 2 * 1.0, 1.9 * rws), squeeze=False)
        for ax in axs.ravel():
            ax.axis("off")
        for n, (_, r) in enumerate(part.iterrows()):
            rr, cc = divmod(n, cols)
            for off, path, lab in ((0, r["test_path"], "test"), (1, r["train_path"], "train")):
                im = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
                axs[rr, cc * 2 + off].imshow(im)
                axs[rr, cc * 2 + off].set_title(f"{lab}" + (f"  d={r['hash_dist']} S={r['ssim']:.2f}" if off == 0 else ""), fontsize=8)
        fig.suptitle(f"{name}: flagged near-duplicate pairs (page {page + 1}); confirmed = SSIM >= 0.9", fontsize=9)
        plt.tight_layout()
        save_fig_named(fig, f"S6_montage_{_slug(name)}_p{page + 1}")


def hash_groups(paths, thr=8, max_group_frac=0.20, fallback=(6, 4, 2, 0)):
    """Connected components of the FOV-hash graph (edge if Hamming distance <= thr) = proxy for video sequences.
    If one component would exceed max_group_frac of all images the threshold is lowered automatically."""
    from scipy.sparse import csr_matrix
    from scipy.sparse.csgraph import connected_components
    H = np.stack([fov_hash_bits(p) for p in paths])
    dist = (H[:, None, :] != H[None, :, :]).sum(2)
    used = thr
    for t in [thr] + [c for c in fallback if c < thr]:
        used = t
        _, lab = connected_components(csr_matrix(dist <= t), directed=False)
        if np.bincount(lab).max() <= max_group_frac * len(paths):
            break
    return lab, used, dist


def group_split(labels, n_target, seed=42, tol=0.10):
    """Pick WHOLE groups (shuffled with `seed`) until ~n_target images are chosen. Returns a boolean mask over images."""
    rng = np.random.default_rng(seed)
    gids = np.unique(labels)
    rng.shuffle(gids)
    sizes = {g: int((labels == g).sum()) for g in gids}
    chosen, cur = [], 0
    for g in gids:
        if cur >= n_target:
            break
        if cur + sizes[g] <= n_target * (1 + tol):
            chosen.append(g)
            cur += sizes[g]
    for g in sorted((g for g in gids if g not in chosen), key=lambda g: sizes[g]):          # 2nd pass: fill up with the smallest remaining groups
        if cur >= 0.95 * n_target:
            break
        if cur + sizes[g] <= n_target * (1 + tol):
            chosen.append(g)
            cur += sizes[g]
    if cur < 0.8 * n_target:
        raise RuntimeError(f"group_split: could only collect {cur} of {n_target} images; lower hash threshold / group size cap")
    return np.isin(labels, chosen)


# ----------------------------------------------------------------------------------------------------------------
# context switch (M8 re-uses EVERY training/analysis function on a different split / other result folders)
# ----------------------------------------------------------------------------------------------------------------
@contextlib.contextmanager
def use_context(ctx):
    g = globals()
    old = {k: g[k] for k in ctx if k in g}
    g.update(ctx)
    try:
        yield
    finally:
        g.update(old)


def make_pool(Dx):
    tr = (np.concatenate([Dx["kvasir"]["train"][0], Dx["cvc"]["train"][0]]), np.concatenate([Dx["kvasir"]["train"][1], Dx["cvc"]["train"][1]]))
    va = (np.concatenate([Dx["kvasir"]["val"][0], Dx["cvc"]["val"][0]]), np.concatenate([Dx["kvasir"]["val"][1], Dx["cvc"]["val"][1]]))
    return tr, va


def regime_ckpt(name, seed, dataset=None):
    """weights path of a regime in the CURRENT context. FedBN has one file per client (Kvasir -> client 0, CVC -> client 1)."""
    tag, folder = CFG[name]
    if name == "FedBN":
        return f"{folder}/{tag}_seed{seed}_client{ {'kvasir': 0, 'cvc': 1}.get(dataset, 0) }.weights.h5"
    return f"{folder}/{tag}_seed{seed}.weights.h5"


# ----------------------------------------------------------------------------------------------------------------
# Federated training v2  (= your 'B4 REPLACEMENT (FIXED)' + options for review M2 / S2 / S4)
#   default options reproduce B4 EXACTLY (so existing fl_fedavg / fl_fedprox / fl_fedbn results stay valid)
#   agg="equal"            : aggregation weights 0.5/0.5 instead of client-size weights               (S2)
#   lr_decay_rounds=(10,15): learning rate x0.5 after rounds 10 and 15                                   (M2 b)
#   select_best=True       : keep the round with the best COMBINED per-image validation Dice             (M2 b)
#   keep_opt=True          : each client keeps its own Adam state across rounds (default: reset)         (M2 optional)
#   aug="strong"           : stronger augmentation (normally set globally by step aug2 via TRAIN_AUG)    (S4)
# ----------------------------------------------------------------------------------------------------------------
FLCFG_DEFAULT = dict(agg="size", lr_decay_rounds=(), select_best=False, keep_opt=False, aug=None, mu=0.01, lr=1e-4)   # aug=None -> global TRAIN_AUG
_FL_MAX_GPU_STALLS = 2


def _fl_clean():
    tf.keras.backend.clear_session()
    gc.collect()


def _fl_gpu_error(error):
    if isinstance(error, tf.errors.ResourceExhaustedError):
        return True
    message = str(error).lower()
    return any(t in message for t in ("out of memory", "oom when", "failed to allocate", "cuda_error", "cuda error", "cudnn",
                                      "cublas", "no algorithm worked", "dnn library", "streamexecutor", "failed call to cuinit"))


def _gpu_mem_mib():
    try:
        return tf.config.experimental.get_memory_info("GPU:0")["current"] / 2.0 ** 20
    except Exception:
        return None


def fl_done(tag, mode, seed, folder=None):
    folder = FED if folder is None else folder
    prefix = os.path.join(folder, f"{tag}_seed{seed}")
    w = [prefix + f"_client{c}.weights.h5" for c in range(2)] if mode == "fedbn" else [prefix + ".weights.h5"]
    return os.path.isfile(prefix + ".npz") and all(os.path.isfile(p) for p in w)


def _fl_train(mode, seed, device, state, cfg):
    set_seed(seed)
    clients = ("kvasir", "cvc")
    sizes = [len(D[k]["train"][0]) for k in clients]
    steps = [max(1, n // BATCH) for n in sizes]
    fractions = [0.5, 0.5] if cfg["agg"] == "equal" else [n / float(sum(sizes)) for n in sizes]
    with tf.device("/CPU:0"):
        train_sets = [make_tf_ds(*D[k]["train"], BATCH, training=True, repeat=True, seed=seed, aug=cfg["aug"]) for k in clients]
        val_sets = [make_tf_ds(*D[k]["val"], BATCH) for k in clients]
        val_pre = [(to_model_input(D[k]["val"][0]), D[k]["val"][1]) for k in clients] if cfg["select_best"] else None
    with tf.device(device):
        worker = build()
        bn_idx = bn_weight_indices(worker)
        bn_set = set(bn_idx)
        if not state:
            initial = worker.get_weights()
            state.update(glob=initial, client_bn=[{i: initial[i].copy() for i in bn_idx} for _ in clients], bn_idx=bn_idx,
                         history=[], client_opt=[None, None], best=None)
        if bn_idx != state["bn_idx"]:
            raise ValueError("FL: rebuilt model has a different BatchNorm weight order")
        optimizer = tf.keras.optimizers.Adam(learning_rate=cfg["lr"], jit_compile=False)
        optimizer.build(worker.trainable_variables)
        opt_vars = optimizer.variables
        opt_vars = list(opt_vars() if callable(opt_vars) else opt_vars)
        initial_opt = [v.numpy().copy() for v in opt_vars]
        refs, train_idx = [], []
        loss_fn = bce_dice_loss
        if mode == "fedprox":
            widx = {id(v): i for i, v in enumerate(worker.weights)}
            train_idx = [widx[id(v)] for v in worker.trainable_variables]
            refs = [tf.Variable(state["glob"][i], trainable=False, dtype=v.dtype) for v, i in zip(worker.trainable_variables, train_idx)]
            train_vars = list(worker.trainable_variables)

            def loss_fn(y_true, y_pred):                                       # (mu/2) * ||w - w_global||^2
                penalty = tf.add_n([tf.reduce_sum(tf.square(v - r)) for v, r in zip(train_vars, refs)])
                return bce_dice_loss(y_true, y_pred) + 0.5 * cfg["mu"] * penalty
        worker.compile(optimizer=optimizer, loss=loss_fn, metrics=[dice_coef], jit_compile=False)

        for round_id in range(len(state["history"]), FL_ROUNDS):
            lr_now = cfg["lr"] * (0.5 ** sum(1 for r in cfg["lr_decay_rounds"] if round_id >= r))
            global_weights = state["glob"]
            local_weights = []
            for ref, index in zip(refs, train_idx):
                ref.assign(global_weights[index])
            for client in range(len(clients)):
                worker.set_weights(client_weights(global_weights, state["client_bn"], bn_idx, client, mode))
                start = (state["client_opt"][client] if cfg["keep_opt"] and state["client_opt"][client] is not None else initial_opt)
                for variable, value in zip(opt_vars, start):
                    variable.assign(value)
                tf.keras.backend.set_value(optimizer.learning_rate, lr_now)
                worker.reset_metrics()
                worker.fit(train_sets[client], epochs=FL_LOCAL_EP, steps_per_epoch=steps[client], verbose=0)
                if cfg["keep_opt"]:
                    state["client_opt"][client] = [v.numpy().copy() for v in opt_vars]
                local_weights.append(worker.get_weights())
            merged = [global_weights[i] if mode == "fedbn" and i in bn_set else sum(fractions[c] * local_weights[c][i] for c in range(len(clients)))
                      for i in range(len(global_weights))]
            client_bn = ([{i: local_weights[c][i].copy() for i in bn_idx} for c in range(len(clients))] if mode == "fedbn" else state["client_bn"])
            val_dice, per_img = [], []
            for client in range(len(clients)):
                worker.set_weights(client_weights(merged, client_bn, bn_idx, client, mode))
                val_dice.append(float(worker.evaluate(val_sets[client], verbose=0, return_dict=True)["dice_coef"]))
                if cfg["select_best"]:
                    pr = predict_probs(worker, val_pre[client][0], bs=BATCH)
                    per_img.append(per_image_metrics(val_pre[client][1], binarize(pr, 0.5))["dice"])
            row = {"round": round_id + 1, "val_dice_per_client": val_dice, "val_dice_mean": float(np.mean(val_dice)), "lr": lr_now}
            if cfg["select_best"]:
                comb = float(np.concatenate(per_img).mean())                       # image-weighted over BOTH clients' validation images
                row["val_dice_img_combined"] = comb
                if state["best"] is None or comb > state["best"]["val"]:
                    state["best"] = {"val": comb, "round": round_id + 1, "glob": [w.copy() for w in merged],
                                     "client_bn": [{i: v.copy() for i, v in d.items()} for d in client_bn]}
            state["glob"], state["client_bn"] = merged, client_bn
            state["history"].append(row)
            mem = _gpu_mem_mib() if device.startswith("/GPU") else None
            print(f"[{mode} seed {seed}] {device} | round {round_id + 1}/{FL_ROUNDS} | lr {lr_now:.2e} | val Dice: "
                  f"{[round(x, 4) for x in val_dice]}" + (f" | combined/img {row['val_dice_img_combined']:.4f}" if cfg['select_best'] else "")
                  + (f" | GPU live {mem:.0f} MiB" if mem is not None else ""), flush=True)
            del local_weights
            gc.collect()


def _fl_finish(tag, mode, seed, state, has_gpu, cfg, folder):
    if cfg["select_best"] and state.get("best") is not None:                          # evaluate / save the SELECTED round
        state["glob"], state["client_bn"] = state["best"]["glob"], state["best"]["client_bn"]
        with open(os.path.join(folder, f"{tag}_seed{seed}_selected_round.json"), "w") as fh:
            json.dump({"selected_round": state["best"]["round"], "val_dice_img_combined": state["best"]["val"]}, fh)
    plan = ([("/GPU:0", 4)] if has_gpu else []) + [("/CPU:0", 8)]
    result = None
    for device, bs in plan:
        _fl_clean()
        try:
            with tf.device(device):
                result = eval_federated(build, state, mode, TEST, client_for_set={"kvasir": 0, "cvc": 1}, bs=bs)
            break
        except tf.errors.OpError as error:
            if device == "/CPU:0" or not _fl_gpu_error(error):
                raise
            print(f"[{mode} seed {seed}] eval failed on GPU -> CPU (slow, training result is safe).", flush=True)
    _fl_clean()
    with tf.device("/CPU:0"):
        save_fed_models(build, state, mode, os.path.join(folder, f"{tag}_seed{seed}"))
    _fl_clean()
    with open(os.path.join(folder, f"{tag}_seed{seed}_history.json"), "w") as fh:
        json.dump(state["history"], fh)
    save_fed_result(tag, seed, result, folder)                                       # npz = completion marker, written LAST
    return result


def fl_run(tag, mode, seed, cfg=None, folder=None):
    """Train ONE federated run (tag e.g. 'fl_fedavg', 'fl_fedavg_eq' ...), evaluate on all TEST sets, save weights/npz/history."""
    cfg = {**FLCFG_DEFAULT, **(cfg or {})}
    folder = FED if folder is None else folder
    os.makedirs(folder, exist_ok=True)
    if fl_done(tag, mode, seed, folder):
        print(f"[{tag} seed {seed}] already trained -> SKIP")
        return
    has_gpu = bool(tf.config.list_physical_devices("GPU"))
    state, use_cpu, stalls = {}, not has_gpu, 0
    while True:
        device = "/CPU:0" if use_cpu else "/GPU:0"
        before = len(state.get("history", []))
        _fl_clean()
        try:
            _fl_train(mode, seed, device, state, cfg)
            break
        except tf.errors.OpError as error:
            if use_cpu or not _fl_gpu_error(error):
                raise
            reason = (str(error).strip().splitlines() or ["?"])[0][:140]
        done_rounds = len(state.get("history", []))
        stalls = 0 if done_rounds > before else stalls + 1
        if stalls >= _FL_MAX_GPU_STALLS:
            use_cpu = True
            print(f"[{tag} seed {seed}] GPU failed {stalls}x without progress ({reason}) -> CPU from round {done_rounds + 1} (very slow).", flush=True)
        else:
            print(f"[{tag} seed {seed}] GPU error ({reason}); cleanup + retry on GPU from round {done_rounds + 1}.", flush=True)
    _fl_clean()
    res = _fl_finish(tag, mode, seed, state, has_gpu, cfg, folder)
    print(f"[{tag} seed {seed}] DONE:", {NAMES[k]: round(float(v[0].mean()), 4) for k, v in res.items()}, flush=True)
    del state, res
    _fl_clean()


def fl_run_seeds(tag, mode, seeds, cfg=None, folder=None):
    for s in seeds:
        fl_run(tag, mode, s, cfg, folder)


# ----------------------------------------------------------------------------------------------------------------
# M6 : FedBN as ONE deployable model
# ----------------------------------------------------------------------------------------------------------------
def _scores(Y, probs):
    per = per_image_metrics(Y, binarize(probs, 0.5))
    return per["dice"], per["iou"]


def fedbn_ensemble_eval(folder, tag, seed, test_sets, bs=4):
    """(a) average the PROBABILITY MAPS of the two client models, threshold once, score once."""
    m = build()
    acc = {k: 0.0 for k in test_sets}
    for c in (0, 1):
        m.load_weights(f"{folder}/{tag}_seed{seed}_client{c}.weights.h5")
        for k, (X, Y) in test_sets.items():
            acc[k] = acc[k] + predict_probs(m, to_model_input(X), bs=bs) / 2.0
    res = {k: _scores(test_sets[k][1], acc[k]) for k in test_sets}
    tf.keras.backend.clear_session()
    return res


def adabn_reestimate(model, Xp, bs=8):
    """AdaBN: recompute BatchNorm moving mean/var on the (unlabelled) target images. Cumulative average of the batch statistics:
    batch i is blended with momentum i/(i+1) -> after the last batch the moving statistics are the plain mean over all batches."""
    bns = [l for l in model.layers if isinstance(l, layers.BatchNormalization)]
    old = [l.momentum for l in bns]
    try:
        for i, s in enumerate(range(0, len(Xp), bs)):
            for l in bns:
                l.momentum = i / (i + 1.0)
            model(tf.convert_to_tensor(Xp[s:s + bs]), training=True)
    finally:
        for l, mo in zip(bns, old):
            l.momentum = mo


def fedbn_adabn_eval(folder, tag, seed, test_sets, bs=8):
    """(b) shared weights = average of the two FedBN client models (identical non-BN weights; BN affine params averaged),
    BN statistics re-estimated on each target test set (AdaBN, transductive, no labels)."""
    m = build()
    ws = []
    for c in (0, 1):
        m.load_weights(f"{folder}/{tag}_seed{seed}_client{c}.weights.h5")
        ws.append(m.get_weights())
    shared = [(a + b) / 2.0 for a, b in zip(ws[0], ws[1])]
    res = {}
    for k, (X, Y) in test_sets.items():
        m.set_weights(shared)
        Xp = to_model_input(X)
        adabn_reestimate(m, Xp, bs=bs)
        res[k] = _scores(Y, predict_probs(m, Xp, bs=4))
    tf.keras.backend.clear_session()
    return res


def eval_existing(tag, folder, seed, test_sets, build_fn=None, out_folder=None, out_tag=None):
    """Evaluate already-trained global weights on (other) test sets and store the per-image npz (no training)."""
    build_fn = build_fn or build
    m = build_fn()
    m.load_weights(f"{folder}/{tag}_seed{seed}.weights.h5")
    pack = {}
    for k, (X, Y) in test_sets.items():
        _, per, _ = evaluate_arrays(m, X, Y, bs=4)
        pack[f"dice__{k}"], pack[f"iou__{k}"] = per["dice"], per["iou"]
    os.makedirs(out_folder, exist_ok=True)
    np.savez(os.path.join(out_folder, f"{out_tag or tag}_seed{seed}.npz"), **pack)
    tf.keras.backend.clear_session()


# ================================================================================================================
# ================================================================================================================
#                                   CONFIG  +  DATA  +  STEP REGISTRY
# ================================================================================================================
# ================================================================================================================
try:
    for _g in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(_g, True)
except RuntimeError as _e:
    print("memory growth already set / GPU already initialised:", _e)

# ======================= SWITCHES (same names as in your notebooks) =======================
QUICK_TEST = bool(ARGS.quick)         # True = 1 seed, 2 epochs, separate *_quicktest folders
RUN_ABLATION = True                   # frozen / no-ImageNet (CVC + Kvasir), CLAHE, complex-decoder components (Kvasir)
RUN_KVASIR_AND_FED = True             # Kvasir-only, pooled, FedAvg / FedProx / FedBN
RUN_COMPLEX_DS = False                # + deep-supervision variant of the complex model (extra time)
RUN_ABLATION_NOSKIP = False           # optional: U-Net without skip connections
# --- NEW switches (review) ---
RUN_M2 = True                         # protocol-control arms (10 runs)
M2_KEEP_OPT = False                   # + FedAvg whose clients keep their Adam state across rounds (5 extra runs)
RUN_S2 = True                         # FedAvg with equal weights (5 runs)
RUN_AUG2 = True                       # YOUR stronger-augmentation recipe (aug2 notebook): all six regimes x 5 seeds -> results_*_aug2 (30 runs; skipped if already on disk)
RUN_M8 = True                         # CVC-ClinicDB sequence-group split (20 runs + evaluation of Kvasir-only)
M8_MODES = ("fedavg", "fedbn")        # federated modes retrained in M8 (add "fedprox" if you have the time)
M8_HASH_THR = 8                       # hash-graph edge threshold (auto-lowered if one group becomes > 20% of all frames)
PRANET_ROOT = ARGS.root

IMG_SIZE, BATCH = 352, 8
SEEDS = [42] if QUICK_TEST else [42, 123, 2024, 7, 99]
if ARGS.seeds.strip():
    SEEDS = [int(x) for x in ARGS.seeds.split(",") if x.strip()]
EPOCHS = 2 if QUICK_TEST else 40                     # SAME total epochs in every configuration (matched budget)
EP_CVC = EP_KV = EP_POOL = EPOCHS
FL_LOCAL_EP = 1 if QUICK_TEST else 2
FL_ROUNDS = EPOCHS // FL_LOCAL_EP
assert FL_ROUNDS * FL_LOCAL_EP == EPOCHS, "FL rounds x local epochs must equal EPOCHS"
N_LIME = 3 if QUICK_TEST else 50
N_GC = 5 if QUICK_TEST else 100
SUFFIX = "_quicktest" if QUICK_TEST else ""
CVC_DIR, KV_DIR, FED = "results_cvc" + SUFFIX, "results_kvasir" + SUFFIX, "results_fed" + SUFFIX
OUT = CVC_DIR
for _d in (CVC_DIR, KV_DIR, FED):
    os.makedirs(_d, exist_ok=True)

NAMES = {"kvasir": "Kvasir-SEG", "cvc": "CVC-ClinicDB", "cvc300": "CVC-300", "colondb": "CVC-ColonDB", "etis": "ETIS"}
build = lambda: build_unet_resnet50(IMG_SIZE)          # PLAIN ResNet-50 U-Net, ImageNet weights, full fine-tuning
D = TEST = POOL_TR = POOL_VA = None                    # filled by init_data()
CFG = {"CVC-only": ("cvc_main", CVC_DIR), "Kvasir-only": ("kvasir_main", KV_DIR), "Pooled (central)": ("pooled", FED),
       "FedAvg": ("fl_fedavg", FED), "FedProx": ("fl_fedprox", FED), "FedBN": ("fl_fedbn", FED)}
FL_MODES = {"FedAvg": "fedavg", "FedProx": "fedprox", "FedBN": "fedbn"}
# new arms (tags are fixed; they live next to the originals)
TAG_POOL_CONST = "pooled_constlr_last"          # M2 (a)
TAG_FEDAVG_TUNED = "fl_fedavg_tuned"            # M2 (b)
TAG_FEDAVG_KEEPOPT = "fl_fedavg_tuned_keepopt"  # M2 optional
TAG_FEDAVG_EQ = "fl_fedavg_eq"                  # S2
TAG_FEDBN_ENS = "fl_fedbn_ens"                  # M6 (a)
TAG_FEDBN_ADABN = "fl_fedbn_adabn"              # M6 (b)
SEQ_ROOT = "results_seqsplit" + SUFFIX          # M8 result folders (cvc / kvasir / fed live below it)
M2_TUNED_CFG = dict(select_best=True, lr_decay_rounds=(FL_ROUNDS // 2, (3 * FL_ROUNDS) // 4))     # rounds 10 and 15 of 20

# ======================= step registry =======================
STEPS = []


def step(name, group, desc):
    def deco(fn):
        STEPS.append({"name": name, "group": group, "desc": desc, "fn": fn})
        return fn
    return deco


def init_data():
    global D, TEST, POOL_TR, POOL_VA
    D, TEST = prepare_pranet(PRANET_ROOT, IMG_SIZE, val_frac=0.1, seed=42)
    POOL_TR, POOL_VA = make_pool(D)
    print("TEST sets:", {NAMES[k]: v[0].shape[0] for k, v in TEST.items()})
    print("QUICK_TEST =", QUICK_TEST, "| seeds =", SEEDS, "| folders:", CVC_DIR, KV_DIR, FED)


def have(tag, folder=None, seeds=None):
    seeds = SEEDS if seeds is None else seeds
    folder = FED if folder is None else folder
    return all(os.path.exists(f"{folder}/{tag}_seed{s}.npz") for s in seeds)


def avg_dice(tag, folder, key):
    return np.mean(load_dice(tag, SEEDS, key, folder), 0)


def show_model_results(tag, folder, title):
    """mean +- SD table + violin plot (per-image Dice of every test set) -- from disk, no training"""
    if not have(tag, folder):
        print(f"[{title}] results not found ({folder}/{tag}_seed*.npz) -> run its training step first")
        return
    print(f"=== {title}: mean +- SD over {len(SEEDS)} seeds ===")
    t = ms_table(tag, SEEDS, list(TEST), folder)
    t["test_set"] = t["test_set"].map(NAMES)
    display(t, name=f"results_{_slug(title)}")
    violin_plot({NAMES[k]: avg_dice(tag, folder, k) for k in TEST}, f"{title}: per-image Dice (seed-averaged; red dot = mean)", rotate=15)
    plt.show()


def free_mem():
    free_gpu_memory()
    gc.collect()


def fmt_ms(vals, nd=3):
    v = np.asarray(vals, float)
    v = v[np.isfinite(v)]
    return f"{v.mean():.{nd}f} \u00b1 {(v.std(ddof=1) if len(v) > 1 else 0.0):.{nd}f}" if len(v) else "-"


def fb_label(name, key=None):
    """FedBN is NOT one deployable model: seen sites = personalised client model, unseen sets = mean of the two client models' scores (review M6)."""
    if name != "FedBN":
        return name
    return "FedBN (personalised)" if key in ("kvasir", "cvc") else "FedBN (mean of 2 client models)"


# ================================================================================================================
# ================================================================================================================
#                                   CORE STEPS  (every cell of your two notebooks)
# ================================================================================================================
# ================================================================================================================
@step("env", "core", "Environment / GPU check (notebook CELL 2)")
def step_env():
    print("python", sys.version.split()[0], "| tensorflow", tf.__version__, "| numpy", np.__version__)
    gpus = tf.config.list_physical_devices("GPU")
    print("GPUs:", [g.name for g in gpus] if gpus else "NONE (training will be very slow)")
    if gpus:
        with tf.device("/GPU:0"):
            y = tf.keras.layers.Conv2D(8, 3)(tf.random.normal((2, 352, 352, 3)))
        print("GPU conv test OK on", y.device)
    for mod in ("scipy", "pandas", "sklearn", "skimage", "cv2", "lime", "h5py"):
        try:
            m = __import__(mod)
            print(f"  OK   {mod:8s} {getattr(m, '__version__', '')}")
        except Exception as e:
            print(f"  FAIL {mod}: {e}  -> pip install it")


@step("data", "core", "Split lists (CSV), train/val overlap check, client-heterogeneity plots (C0, C1, D5, D4, K11, K12)")
def step_data():
    for k in ("cvc", "kvasir"):
        assert not set(D[k]["paths_train"]) & set(D[k]["paths_val"]), f"{k}: train/val overlap!"
        print(f"{NAMES[k]}: train {len(D[k]['train'][0])} | val {len(D[k]['val'][0])} (checkpoint selection only) | official test {len(D[k]['test'][0])}")
    print("Augmentation: on-the-fly, ONLY on training batches (after the split). Val/test: none.")
    rows = []
    for k in ("kvasir", "cvc"):
        for sp in ("train", "val"):
            rows += [{"dataset": NAMES[k], "split": sp, "image_id": os.path.basename(p)} for p in D[k][f"paths_{sp}"]]
    for k in TEST:
        rows += [{"dataset": NAMES[k], "split": "test", "image_id": os.path.basename(p)} for p in D[k]["paths_test"]]
    split_df = pd.DataFrame(rows)
    display(split_df.groupby(["dataset", "split"]).size().unstack(fill_value=0), name="split_sizes")
    display(split_df, name="split_image_ids", show=False)
    # client heterogeneity (D4, panels 1-2) and Kvasir-vs-test-set heterogeneity (K11)
    cl = {"Kvasir-SEG": D["kvasir"]["train"], "CVC-ClinicDB": D["cvc"]["train"]}
    fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.6))
    violin_plot({n: (Y > 0.5).reshape(len(Y), -1).mean(1) for n, (X, Y) in cl.items()}, "Polyp size (mask/image area)", ylabel="area fraction", ax=axs[0])
    violin_plot({n: X.reshape(len(X), -1).mean(1) / 255.0 for n, (X, Y) in cl.items()}, "Image brightness", ylabel="mean pixel / 255", ax=axs[1])
    plt.tight_layout(); plt.show()
    sets = {"Kvasir-SEG (train)": D["kvasir"]["train"], **{NAMES[k]: TEST[k] for k in ("cvc", "colondb", "etis") if k in TEST}}
    fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.8))
    violin_plot({n: (Y > 0.5).reshape(len(Y), -1).mean(1) for n, (X, Y) in sets.items()}, "Polyp size", ylabel="area fraction", ax=axs[0], rotate=15)
    violin_plot({n: X.reshape(len(X), -1).mean(1) / 255.0 for n, (X, Y) in sets.items()}, "Brightness", ylabel="mean pixel / 255", ax=axs[1], rotate=15)
    plt.tight_layout(); plt.show()


# ----------------------------------------------------------------------------------------------------------------
@step("leak", "core", "Leakage audit: pHash, exact copies, thresholds, clean subset stats, control analysis, FOV+SSIM+montage (C1,D6,V4,Q1,T1,K12 + S6)")
def step_leak():
    train_all = D["kvasir"]["paths_train"] + D["kvasir"]["paths_val"] + D["cvc"]["paths_train"] + D["cvc"]["paths_val"]
    display(leakage_report(train_all, {NAMES[k]: D[k]["paths_test"] for k in TEST}, max_dist=8), name="leakage_phash_all_sets")
    PH = 8
    # ---- original audit (full-image pHash <= 8) on the CVC-ClinicDB test set ----
    nd = near_duplicates(train_all, D["cvc"]["paths_test"], max_dist=64)
    dist = np.asarray(nd["nearest_dist"])
    flag_orig = dist <= PH
    rows = []
    for th in (-1, 0, 2, 4, 8, 12):                                          # V4
        keep = dist > th
        row = {"pHash threshold": "full set" if th < 0 else f"<= {th} removed", "removed": int((~keep).sum()), "left": int(keep.sum())}
        for name in ("CVC-only", "Pooled (central)", "FedAvg", "FedProx", "FedBN"):
            if has_results(*CFG[name], keys=["cvc"]):
                row[name] = pm([a[keep].mean() for a in load_dice(CFG[name][0], SEEDS, "cvc", CFG[name][1])]) if keep.sum() >= 5 else "n<5"
        rows.append(row)
    display(pd.DataFrame(rows), name="leak_threshold_sensitivity")
    # ---- D6: every test set, full vs without near-duplicates ----
    flags = {k: near_duplicates(train_all, D[k]["paths_test"], max_dist=PH)["nearest_dist"] <= PH for k in TEST}
    print(f"Near-duplicate test images (pHash <= {PH}):", {NAMES[k]: f"{int(v.sum())}/{len(v)}" for k, v in flags.items()})
    rows = []
    for name, (tag, folder) in CFG.items():
        if not has_results(tag, folder):
            continue
        for k in TEST:
            if not flags[k].any() or (~flags[k]).sum() < 5:
                continue
            arrs = load_dice(tag, SEEDS, k, folder)
            full, clean = [a.mean() for a in arrs], [a[~flags[k]].mean() for a in arrs]
            rows.append({"model": name, "test set": NAMES[k], "n": len(flags[k]), "n near-dup": int(flags[k].sum()), "mDice full": pm(full),
                         "mDice w/o near-dup": pm(clean), "difference": round(float(np.mean(clean) - np.mean(full)), 4)})
    display(pd.DataFrame(rows), name="leak_full_vs_clean_all_sets")

    # ---- S6: FIELD-OF-VIEW hashing + SSIM >= 0.9 confirmation + montage ----
    audits = {}
    for k in TEST:
        a = audit_near_duplicates(train_all, D[k]["paths_test"], max_dist=PH, min_ssim=0.9)
        audits[k] = a
        flagged_pairs_montage(a, NAMES[k])
    summ = pd.DataFrame([{"test set": NAMES[k], "n": len(a), "hash-flagged (FOV pHash<=8)": int(a["hash_flag"].sum()),
                          "SSIM-confirmed (>=0.9)": int(a["confirmed"].sum()),
                          "exact copies (d=0)": int((a["hash_dist"] == 0).sum())} for k, a in audits.items()])
    display(summ, name="S6_fov_hash_ssim_summary")
    for k, a in audits.items():                                           # hash lists for the repository (review week-3 plan)
        display(a.drop(columns=["test_path", "train_path"]), name=f"S6_{k}_pairs", show=False)
    hrows = []
    for split, plist in (("train+val", train_all), *((f"test:{NAMES[k]}", D[k]["paths_test"]) for k in TEST)):
        for p in plist:
            bits = fov_hash_bits(p)
            hrows.append({"split": split, "image_id": os.path.basename(p), "fov_phash_hex": "%016x" % int("".join("1" if b else "0" for b in bits), 2)})
    display(pd.DataFrame(hrows), name="S6_fov_phash_all_images", show=False)
    rows = []
    for name, (tag, folder) in CFG.items():
        if not has_results(tag, folder):
            continue
        for k in TEST:
            a = audits[k]
            full_f = [x.mean() for x in load_dice(tag, SEEDS, k, folder)]
            r = {"model": name, "test set": NAMES[k], "mDice full": pm(full_f)}
            for lab, m in (("w/o hash-flagged", ~a["hash_flag"].values), ("w/o SSIM-confirmed", ~a["confirmed"].values),
                           ("hash-flagged but NOT confirmed", a["hash_flag"].values & ~a["confirmed"].values)):
                if m.sum() >= 5 and (lab != "w/o hash-flagged" or (~m).any()):
                    r[lab] = pm([x[m].mean() for x in load_dice(tag, SEEDS, k, folder)])
            rows.append(r)
    display(pd.DataFrame(rows), name="S6_dice_by_flag_definition")
    print("If removing flagged images RAISES Dice (Kvasir, ColonDB) the flagged images were harder than average -> many hash matches are false;"
          " use the SSIM-confirmed definition in the paper.")

    # ---- Q1: clean-subset statistics (both flag definitions), consistent M3 tests ----
    n_kv_train = len(D["kvasir"]["paths_train"]) + len(D["kvasir"]["paths_val"])
    ha = np.stack([phash_bits(cv2.imread(p)) for p in train_all])
    hb = np.stack([phash_bits(cv2.imread(p)) for p in D["cvc"]["paths_test"]])
    dm = (ha[:, None, :] != hb[None, :, :]).sum(2)
    nearest, arg = dm.min(0), dm.argmin(0)
    exact = np.where(nearest == 0)[0]
    if len(exact):
        display(pd.DataFrame([{"test image": os.path.basename(D["cvc"]["paths_test"][i]), "nearest train image": os.path.basename(train_all[arg[i]]),
                               "train dataset": "CVC-ClinicDB" if arg[i] >= n_kv_train else "Kvasir-SEG"} for i in exact]), name="leak_exact_copies")
    bins = pd.cut(nearest, [-1, 0, 2, 5, 8, 1000], labels=["0 (exact)", "1-2", "3-5", "6-8", "> 8 (clean)"])
    display(bins.value_counts().sort_index().rename("n test images").to_frame(), name="leak_distance_bins")
    models = [n for n in CFG if has_results(*CFG[n], keys=["cvc"])]
    REF = "Pooled (central)"
    conf = audits["cvc"]["confirmed"].values
    for lab, flag in (("pHash<=8 (original)", flag_orig), ("FOV pHash + SSIM>=0.9", conf)):
        keep = ~flag
        rows = [{"model": n, f"full (n={len(flag)})": fmt_ms([a.mean() for a in load_dice(CFG[n][0], SEEDS, "cvc", CFG[n][1])]),
                 f"near-dup (n={int(flag.sum())})": fmt_ms([a[flag].mean() for a in load_dice(CFG[n][0], SEEDS, "cvc", CFG[n][1])]) if flag.any() else "-",
                 f"CLEAN (n={int(keep.sum())})": fmt_ms([a[keep].mean() for a in load_dice(CFG[n][0], SEEDS, "cvc", CFG[n][1])])} for n in models]
        display(pd.DataFrame(rows), name=f"leak_subsets_{_slug(lab)}")
        # identity check: weighted subset means must reproduce the full mean
        err = max(abs((flag.sum() * np.mean([a[flag].mean() for a in load_dice(CFG[n][0], SEEDS, 'cvc', CFG[n][1])]) +
                       keep.sum() * np.mean([a[keep].mean() for a in load_dice(CFG[n][0], SEEDS, 'cvc', CFG[n][1])])) / len(flag) -
                      np.mean([a.mean() for a in load_dice(CFG[n][0], SEEDS, 'cvc', CFG[n][1])])) for n in models) if flag.any() and models else 0.0
        print(f"[{lab}] table-consistency check (weighted subset means == full mean): max abs error = {err:.2e}")
        if keep.sum() >= 8 and REF in models:
            per_image = {n: [a[keep] for a in load_dice(CFG[n][0], SEEDS, "cvc", CFG[n][1])] for n in models}
            display(compare_table(per_image, REF), name=f"leak_clean_subset_stats_{_slug(lab)}")

    # ---- T1: leakage CONTROL analysis (Kvasir-only never saw CVC frames) ----
    CONTROL, TRAINED = "Kvasir-only", ["CVC-only", "Pooled (central)", "FedAvg", "FedProx", "FedBN"]
    cvc_train = D["cvc"]["paths_train"] + D["cvc"]["paths_val"]
    nd2 = near_duplicates(cvc_train, D["cvc"]["paths_test"], max_dist=PH)["nearest_dist"] <= PH
    if has_results(*CFG[CONTROL], keys=["cvc"]):
        for lab, flag in (("pHash<=8 vs CVC train", nd2), ("FOV+SSIM confirmed", conf)):
            if not flag.any() or flag.all():
                continue
            avg = lambda name: np.mean(np.stack(load_dice(CFG[name][0], SEEDS, "cvc", CFG[name][1])), 0)
            ctrl, rng, rows = avg(CONTROL), np.random.default_rng(0), []
            boot = lambda x, B=10000: x[rng.integers(0, len(x), (B, len(x)))].mean(1)
            for name in TRAINED:
                if not has_results(*CFG[name], keys=["cvc"]):
                    continue
                gain = avg(name) - ctrl
                g_dup, g_cl = gain[flag], gain[~flag]
                lo, hi = np.percentile(boot(g_dup) - boot(g_cl), [2.5, 97.5])
                rows.append({"model": name, "mDice near-dup": round(float(avg(name)[flag].mean()), 3), "mDice clean": round(float(avg(name)[~flag].mean()), 3),
                             "gain vs control, near-dup": round(float(g_dup.mean()), 3), "gain vs control, clean": round(float(g_cl.mean()), 3),
                             "(near-dup gain)-(clean gain)": round(float(g_dup.mean() - g_cl.mean()), 3), "CI95": f"[{lo:+.3f}, {hi:+.3f}]"})
            display(pd.DataFrame(rows), name=f"leak_control_{_slug(lab)}")
            print(f"[{lab}] control (Kvasir-only): near-dup {ctrl[flag].mean():.3f} | clean {ctrl[~flag].mean():.3f}  (n near-dup {int(flag.sum())}, clean {int((~flag).sum())})")
    OLD_AUG = "Augmented/PNG/Original"
    if os.path.exists(OLD_AUG):
        files = sorted(os.listdir(OLD_AUG))
        tr_f, tmp_f = train_test_split(files, test_size=0.3, random_state=42)
        _, te_f = train_test_split(tmp_f, test_size=0.5, random_state=42)
        base = lambda f: f.rsplit("_", 1)[0]
        tr_b = {base(f) for f in tr_f}
        print("OLD split: test images whose original frame was in train:", sum(base(f) in tr_b for f in te_f), "/", len(te_f))


# ----------------------------------------------------------------------------------------------------------------
@step("train_central", "core", "Train CVC-only, Kvasir-only, pooled (A1, B1, B2, K2)")
def step_train_central():
    free_mem()
    run_seeds("cvc_main", SEEDS, build, D["cvc"]["train"], D["cvc"]["val"], TEST, epochs=EP_CVC, batch=BATCH, out_dir=CVC_DIR)
    if not RUN_KVASIR_AND_FED:
        print("RUN_KVASIR_AND_FED = False -> Kvasir-only / pooled skipped")
        return
    free_mem()
    run_seeds("kvasir_main", SEEDS, build, D["kvasir"]["train"], D["kvasir"]["val"], TEST, epochs=EP_KV, batch=BATCH, out_dir=KV_DIR)
    free_mem()
    print("pooled train/val:", POOL_TR[0].shape[0], POOL_VA[0].shape[0])
    run_seeds("pooled", SEEDS, build, POOL_TR, POOL_VA, TEST, epochs=EP_POOL, batch=BATCH, out_dir=FED)


@step("train_fl", "core", "Train FedAvg / FedProx / FedBN, 2 institutions, no warm start (B4)")
def step_train_fl():
    if not RUN_KVASIR_AND_FED:
        print("RUN_KVASIR_AND_FED = False -> skip")
        return
    print(f"FL: batch={BATCH}, size={IMG_SIZE}, rounds={FL_ROUNDS}, local epochs={FL_LOCAL_EP}")
    for seed in SEEDS:
        for mode in ("fedavg", "fedprox", "fedbn"):
            fl_run(f"fl_{mode}", mode, seed)


# ----------------------------------------------------------------------------------------------------------------
@step("ablation", "core", "Ablations: frozen / no-ImageNet (CVC + Kvasir), CLAHE, complex decoder components, XAI plain vs complex, collapse check (A6, K7, K7b, V2)")
def step_ablation():
    if not RUN_ABLATION:
        print("RUN_ABLATION = False -> skip")
        return
    free_mem()
    # ---------------- CVC-ClinicDB (A6) ----------------
    only = {"cvc": TEST["cvc"]}
    run_seeds("cvc_frozen", SEEDS, lambda: build_unet_resnet50(IMG_SIZE, freeze_encoder=True), D["cvc"]["train"], D["cvc"]["val"], only, epochs=EP_CVC, batch=BATCH, out_dir=CVC_DIR)
    run_seeds("cvc_scratch", SEEDS, lambda: build_unet_resnet50(IMG_SIZE, weights=None), D["cvc"]["train"], D["cvc"]["val"], only, epochs=EP_CVC, batch=BATCH, out_dir=CVC_DIR)
    if RUN_ABLATION_NOSKIP:
        run_seeds("cvc_noskip", SEEDS, lambda: build_unet_resnet50(IMG_SIZE, skip_connections=False), D["cvc"]["train"], D["cvc"]["val"], only, epochs=EP_CVC, batch=BATCH, out_dir=CVC_DIR)
    per = {"Proposed (fine-tuned)": load_dice("cvc_main", SEEDS, "cvc", CVC_DIR), "Frozen encoder": load_dice("cvc_frozen", SEEDS, "cvc", CVC_DIR),
           "No ImageNet (scratch)": load_dice("cvc_scratch", SEEDS, "cvc", CVC_DIR)}
    if RUN_ABLATION_NOSKIP and have("cvc_noskip", CVC_DIR):
        per["No skip connections"] = load_dice("cvc_noskip", SEEDS, "cvc", CVC_DIR)
    display(compare_table(per, "Proposed (fine-tuned)"), name="ablation_cvc_clinicdb")
    violin_plot({n: np.mean(np.stack(v), 0) for n, v in per.items()}, "Ablation on CVC-ClinicDB: per-image Dice", rotate=10); plt.show()

    # ---------------- V2: collapse check ----------------
    variants = {"Proposed (fine-tuned)": ("cvc_main", build), "Frozen encoder": ("cvc_frozen", lambda: build_unet_resnet50(IMG_SIZE, freeze_encoder=True)),
                "No ImageNet (scratch)": ("cvc_scratch", lambda: build_unet_resnet50(IMG_SIZE, weights=None))}
    Xt, Yt = D["cvc"]["test"]
    gt = Yt[..., 0] > 0.5
    rows = []
    for name, (tag, bf) in variants.items():
        st = []
        for s in SEEDS:
            ck = f"{CVC_DIR}/{tag}_seed{s}.weights.h5"
            if not os.path.exists(ck):
                continue
            m = load_trained(bf, ck)
            _, pr_, probs = evaluate_arrays(m, Xt, Yt)
            pr = probs[..., 0] > 0.5
            tp, fp, fn = int((pr & gt).sum()), int((pr & ~gt).sum()), int((~pr & gt).sum())
            st.append((pr_["dice"].mean(), tp / max(tp + fp, 1), tp / max(tp + fn, 1), 100 * pr.mean(), 100 * gt.mean()))
            del m
            free_mem()
        if st:
            a = np.array(st)
            rows.append({"model": name, "seeds": len(st), "mDice": pm(a[:, 0]), "precision (pixel-pooled)": pm(a[:, 1]), "recall (pixel-pooled)": pm(a[:, 2]),
                         "predicted polyp area (%)": pm(a[:, 3], 1), "GT polyp area (%)": f"{a[0, 4]:.1f}"})
    display(pd.DataFrame(rows), name="ablation_collapse_check")

    # ---------------- Kvasir-SEG (K7) ----------------
    only = {"kvasir": TEST["kvasir"]}
    Xte, Yte = TEST["kvasir"]
    run_seeds("kvasir_frozen", SEEDS, lambda: build_unet_resnet50(IMG_SIZE, freeze_encoder=True), D["kvasir"]["train"], D["kvasir"]["val"], only, epochs=EPOCHS, batch=BATCH, out_dir=KV_DIR)
    run_seeds("kvasir_scratch", SEEDS, lambda: build_unet_resnet50(IMG_SIZE, weights=None), D["kvasir"]["train"], D["kvasir"]["val"], only, epochs=EPOCHS, batch=BATCH, out_dir=KV_DIR)
    run_seeds("kvasir_clahe", SEEDS, build, D["kvasir"]["train"], D["kvasir"]["val"], only, epochs=EPOCHS, batch=BATCH, out_dir=KV_DIR, clahe=True)
    CV = {"kvasir_complex": dict(use_cbam=True, use_attention_gate=True), "kvasir_complex_no_cbam": dict(use_cbam=False, use_attention_gate=True),
          "kvasir_complex_no_attn": dict(use_cbam=True, use_attention_gate=False)}
    for tag, kw in CV.items():
        run_seeds(tag, SEEDS, lambda kw=kw: build_complex_kvasir(IMG_SIZE, deep_supervision=False, **kw), D["kvasir"]["train"], D["kvasir"]["val"], only,
                  epochs=EPOCHS, batch=BATCH, out_dir=KV_DIR)
    if RUN_COMPLEX_DS:
        run_seeds_complex_ds("kvasir_complex_ds", SEEDS, IMG_SIZE, D["kvasir"]["train"], D["kvasir"]["val"], only, epochs=EPOCHS, batch=BATCH, out_dir=KV_DIR)
    if RUN_ABLATION_NOSKIP:
        run_seeds("kvasir_noskip", SEEDS, lambda: build_unet_resnet50(IMG_SIZE, skip_connections=False), D["kvasir"]["train"], D["kvasir"]["val"], only, epochs=EPOCHS, batch=BATCH, out_dir=KV_DIR)
    tag_of = {"Proposed (plain, fine-tuned)": "kvasir_main", "Frozen encoder": "kvasir_frozen", "No ImageNet (scratch)": "kvasir_scratch", "+ CLAHE": "kvasir_clahe",
              "Complex: Full (ASPP+Attn+CBAM)": "kvasir_complex", "Complex: -CBAM": "kvasir_complex_no_cbam", "Complex: -Attention Gate": "kvasir_complex_no_attn"}
    build_of = {"Proposed (plain, fine-tuned)": build, "Frozen encoder": build, "No ImageNet (scratch)": build, "+ CLAHE": build,
                "Complex: Full (ASPP+Attn+CBAM)": lambda: build_complex_kvasir(IMG_SIZE, deep_supervision=False, **CV["kvasir_complex"]),
                "Complex: -CBAM": lambda: build_complex_kvasir(IMG_SIZE, deep_supervision=False, **CV["kvasir_complex_no_cbam"]),
                "Complex: -Attention Gate": lambda: build_complex_kvasir(IMG_SIZE, deep_supervision=False, **CV["kvasir_complex_no_attn"])}
    if RUN_COMPLEX_DS and have("kvasir_complex_ds", KV_DIR):
        tag_of["Complex: Full + Deep Supervision"] = "kvasir_complex_ds"
        build_of["Complex: Full + Deep Supervision"] = lambda: build_complex_kvasir(IMG_SIZE, deep_supervision=True)
    if RUN_ABLATION_NOSKIP and have("kvasir_noskip", KV_DIR):
        tag_of["No skip connections"] = "kvasir_noskip"
        build_of["No skip connections"] = lambda: build_unet_resnet50(IMG_SIZE, skip_connections=False)
    per = {n: load_dice(t, SEEDS, "kvasir", KV_DIR) for n, t in tag_of.items() if have(t, KV_DIR)}
    tab = compare_table(per, "Proposed (plain, fine-tuned)")
    hd = []
    for row_name in tab["config"]:
        mdl = load_trained(build_of[row_name], f"{KV_DIR}/{tag_of[row_name]}_seed{SEEDS[0]}.weights.h5")
        h, _ = hd95_batch(Yte, binarize(predict_probs(mdl, to_model_input(Xte), out_idx=0), 0.5))
        hd.append(round(float(h.mean()), 2))
        del mdl
        free_mem()
    tab["HD95 (px, seed0)"] = hd
    display(tab, name="ablation_kvasir_seg")
    violin_plot({n: np.mean(np.stack(v), 0) for n, v in per.items()}, "Ablation on Kvasir-SEG: per-image Dice", rotate=15); plt.show()
    print("If no Complex row is significantly better than the plain model (Holm p >= 0.05), the plain ResNet-50 U-Net is the justified main architecture.")

    # ---------------- K7b: XAI plain vs complex (same layer, same n) ----------------
    if have("kvasir_complex", KV_DIR) and have("kvasir_main", KV_DIR):
        plain = load_trained(build, f"{KV_DIR}/kvasir_main_seed{SEEDS[0]}.weights.h5")
        r_plain = xai_full_eval(plain, Xte, Yte, n_gc=N_GC, n_lime=N_LIME, lime_segments=150, lime_samples=1000)
        del plain
        free_mem()
        cplx = load_trained(lambda: build_complex_kvasir(IMG_SIZE, deep_supervision=False), f"{KV_DIR}/kvasir_complex_seed{SEEDS[0]}.weights.h5")
        r_cplx = xai_full_eval(cplx, Xte, Yte, n_gc=N_GC, n_lime=N_LIME, lime_segments=150, lime_samples=1000)
        del cplx
        free_mem()
        a, b = xai_report(r_plain), xai_report(r_cplx)
        a.insert(0, "model", "Plain"); b.insert(0, "model", "Complex")
        display(pd.concat([a, b]).pivot_table(index=["method", "metric"], columns="model", values="mean").round(3), name="ablation_xai_plain_vs_complex")


# ----------------------------------------------------------------------------------------------------------------
PUBLISHED = {   # mDice [Kvasir, ClinicDB, CVC-300, ColonDB, ETIS] verified from the original papers (see C1 cell of your notebook)
    "PraNet (MICCAI'20)": [0.898, 0.899, 0.871, 0.709, 0.628], "HarDNet-MSEG (arXiv'21)": [0.912, 0.932, 0.887, 0.731, 0.677],
    "UACANet-S (ACM MM'21)": [0.905, 0.916, 0.902, 0.783, 0.694], "UACANet-L (ACM MM'21)": [0.912, 0.926, 0.910, 0.751, 0.766],
    "Polyp-PVT (arXiv'21)": [0.917, 0.937, 0.900, 0.808, 0.787]}
PUB_EFF = {"PraNet (MICCAI'20)": (32.55, 13.11), "HarDNet-MSEG (arXiv'21)": (33.34, 11.38), "UACANet-S (ACM MM'21)": (None, None),
           "UACANet-L (ACM MM'21)": (None, None), "Polyp-PVT (arXiv'21)": (25.1, 10.0)}   # SAME numbers in every table (CVC notebook G1)
SET_ORDER = ["kvasir", "cvc", "cvc300", "colondb", "etis"]


@step("tables", "core", "All headline tables: mean+-SD, per-site tests vs pooled (M3 consistent stats), unseen image-weighted (S1), matrix, mIoU, SOTA, bars, curves")
def step_tables():
    for tag, folder, title in (("cvc_main", CVC_DIR, "CVC-only model"), ("kvasir_main", KV_DIR, "Kvasir-only model"), ("pooled", FED, "Pooled (central) model")):
        show_model_results(tag, folder, title)
    avail = {n: v for n, v in CFG.items() if has_results(*v)}
    REF = "Pooled (central)"
    if REF not in avail:
        print("Pooled results not found -> run train_central first")
        return
    keys = list(TEST) + ["unseen"]
    labels = {**NAMES, "unseen": "Unseen (image-weighted, 636 imgs)"}
    fig, axs = plt.subplots(2, 3, figsize=(7.2, 6.4))
    axs = axs.ravel()
    for ax, k in zip(axs, keys):
        per = {fb_label(n, k): load_per_image(t, d, k) for n, (t, d) in avail.items()}
        kind = "seen client site" if k in ("kvasir", "cvc") else "UNSEEN (zero-shot)"
        print(f"\n--- {labels[k]}  [{kind}] ---")
        display(compare_table(per, REF), name=f"B5_vs_pooled_{k}", latex=True)
        violin_plot({n.replace(" (central)", ""): np.mean(np.stack(v), 0) for n, v in per.items()}, labels[k], ax=ax, rotate=30, compact=True)
    plt.tight_layout(); plt.show()
    # ---- S1 / M1 evidence: image-weighted unseen mean per regime ----
    rows = []
    for n, (t, d) in avail.items():
        ps = [a.mean() for a in load_per_image(t, d, "unseen")]
        w = {k: len(TEST[k][0]) for k in UNSEEN}
        row = {"regime": fb_label(n, "unseen"), "unseen mDice (image-weighted)": fmt_ms(ps)}
        for k in UNSEEN:
            row[NAMES[k]] = fmt_ms([a.mean() for a in load_dice(t, SEEDS, k, d)])
        row["_mean"] = float(np.mean(ps))
        rows.append(row)
    ut = pd.DataFrame(rows).sort_values("_mean", ascending=False).drop(columns="_mean")
    display(ut, name="M1_unseen_image_weighted", latex=True)
    print("weights:", {NAMES[k]: len(TEST[k][0]) for k in UNSEEN}, "-> if FedAvg/FedBN fall below CVC-only the title must NOT claim robustness / domain generalisation.")

    # ---- C2 generalisation matrix ----
    rows, means = [], {}
    for name, (tag, folder) in CFG.items():
        if not has_results(tag, folder):
            continue
        r = {"trained on \\ tested on": name}
        means[name] = []
        for k in TEST:
            m = [d.mean() for d in load_dice(tag, SEEDS, k, folder)]
            r[NAMES[k]] = pm(m)
            means[name].append(float(np.mean(m)))
        r["Unseen (image-weighted, 636 imgs)"] = fmt_ms([a.mean() for a in load_per_image(tag, folder, "unseen")])
        rows.append(r)
    display(pd.DataFrame(rows), name="C2_generalisation_matrix", latex=True)
    print("FedBN row: Kvasir-SEG / CVC-ClinicDB = personalised client models; unseen sets = mean of the two client models (not one deployable model; see step m6).")
    M = np.array(list(means.values()), float)
    fig, ax = plt.subplots(figsize=(7.2, 0.5 * len(means) + 1.8))
    im = ax.imshow(M, vmin=0.5, vmax=1.0, cmap="viridis")
    ax.set_xticks(range(len(TEST))); ax.set_xticklabels([NAMES[k] for k in TEST], rotation=20)
    ax.set_yticks(range(len(means))); ax.set_yticklabels(list(means))
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i, j]:.3f}", ha="center", va="center", color="w", fontsize=8)
    plt.colorbar(im, label="mDice"); ax.set_title("mDice: trained on (rows) vs tested on (columns)"); plt.tight_layout(); plt.show()

    # ---- W1 mIoU ----
    rows = []
    for name, (tag, folder) in CFG.items():
        if not has_results(tag, folder):
            continue
        r = {"regime": name, **{NAMES[k]: fmt_ms([float(np.mean(np.load(f"{folder}/{tag}_seed{s}.npz")[f"iou__{k}"])) for s in SEEDS]) for k in TEST}}
        r["Unseen (image-weighted)"] = fmt_ms([float(np.concatenate([np.load(f"{folder}/{tag}_seed{s}.npz")[f"iou__{k}"] for k in UNSEEN]).mean()) for s in SEEDS])
        rows.append(r)
    display(pd.DataFrame(rows), name="W1_mIoU_all_regimes", latex=True)

    # ---- SOTA table (C1) ----
    cols = SET_ORDER
    rows = [{"method": k, "seeds": "-", **{NAMES[c]: f"{x:.3f}" for c, x in zip(cols, v)}} for k, v in PUBLISHED.items()]
    for name, label in (("Pooled (central)", "Ours: pooled Kvasir+CVC (central) [like-for-like]"), ("FedAvg", "Ours: FedAvg [federated, NOT comparable]"),
                        ("FedProx", "Ours: FedProx [federated, NOT comparable]"), ("FedBN", "Ours: FedBN [federated, NOT comparable]"),
                        ("Kvasir-only", "Ours: Kvasir-only [NOT like-for-like]"), ("CVC-only", "Ours: CVC-only [NOT like-for-like]")):
        if name in avail:
            t, d = avail[name]
            rows.append({"method": label, "seeds": len(SEEDS), **{NAMES[c]: fmt_ms([a.mean() for a in load_dice(t, SEEDS, c, d)]) for c in cols}})
    display(pd.DataFrame(rows), name="C1_sota_comparison", latex=True)
    print("TABLE NOTE: all methods use the PraNet protocol (900 Kvasir + 550 ClinicDB train; test 100/62/60/380/196). Published values are copied from the papers (original resolution);"
          " ours are mean+-SD over seeds at 352x352. We do NOT claim state of the art; federated rows are compared with the pooled row only.")

    # ---- E1 bar chart + training curves ; F1 FL curves ; D4 panel 3 ----
    names_ = list(avail)
    fig, ax = plt.subplots(figsize=(7.2, 3.6))
    w = 0.8 / len(names_)
    x = np.arange(len(TEST))
    for j, n in enumerate(names_):
        t, d = avail[n]
        ps = {k: [a.mean() for a in load_dice(t, SEEDS, k, d)] for k in TEST}
        ax.bar(x + (j - (len(names_) - 1) / 2) * w, [np.mean(ps[k]) for k in TEST], w, yerr=[np.std(ps[k], ddof=1) if len(ps[k]) > 1 else 0 for k in TEST], capsize=2, label=n)
    ax.set_xticks(x); ax.set_xticklabels([NAMES[k] for k in TEST]); ax.set_ylim(0, 1.05); ax.set_ylabel("mDice (mean \u00b1 SD over seeds)")
    ax.legend(ncol=3, fontsize=8, loc="lower left"); ax.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

    fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.2))
    for name, (tag, folder) in {"CVC-only": ("cvc_main", CVC_DIR), "Kvasir-only": ("kvasir_main", KV_DIR), "Pooled": ("pooled", FED)}.items():
        runs = [json.load(open(p)) for p in (f"{folder}/{tag}_seed{s}.json" for s in SEEDS) if os.path.exists(p)]
        for ax, key, lab in ((axs[0], "val_dice_coef", "val Dice"), (axs[1], "val_loss", "val loss")):
            arr = np.array([r[key] for r in runs if key in r], float)
            if arr.size:
                ep = np.arange(1, arr.shape[1] + 1)
                m, s_ = arr.mean(0), (arr.std(0, ddof=1) if len(arr) > 1 else np.zeros(arr.shape[1]))
                ax.plot(ep, m, label=f"{name} (n={len(arr)})"); ax.fill_between(ep, m - s_, m + s_, alpha=0.2); ax.set_xlabel("epoch"); ax.set_ylabel(lab); ax.grid(alpha=0.3)
    axs[0].legend(fontsize=8); plt.tight_layout(); plt.show()

    fig, axs = plt.subplots(1, 3, figsize=(7.2, 2.8))
    rows = {}
    for name, mode in FL_MODES.items():
        runs = [json.load(open(p)) for p in (f"{FED}/fl_{mode}_seed{s}_history.json" for s in SEEDS) if os.path.exists(p)]
        if not runs:
            continue
        last = np.array([[r[-1]["val_dice_per_client"][0], r[-1]["val_dice_per_client"][1], r[-1]["val_dice_mean"]] for r in runs], float)
        rows[name] = {c: fmt_ms(last[:, i]) for i, c in enumerate(["final val Dice Kvasir", "final val Dice CVC", "final val Dice mean"])}
        for ax, (title, get) in zip(axs, (("mean of clients", lambda h: h["val_dice_mean"]), ("Kvasir client", lambda h: h["val_dice_per_client"][0]), ("CVC client", lambda h: h["val_dice_per_client"][1]))):
            arr = np.array([[get(h) for h in r] for r in runs], float)
            xs = np.arange(1, arr.shape[1] + 1)
            m, sd = arr.mean(0), (arr.std(0, ddof=1) if len(arr) > 1 else np.zeros(arr.shape[1]))
            ax.plot(xs, m, label=name); ax.fill_between(xs, m - sd, m + sd, alpha=0.2); ax.set_title(title); ax.set_xlabel("round"); ax.set_ylabel("val Dice"); ax.grid(alpha=0.3)
    if rows:
        axs[0].legend(fontsize=8); plt.tight_layout(); plt.show(); display(pd.DataFrame(rows).T, name="F1_fl_final_val_dice")
    names3 = [n for n in CFG if has_results(*CFG[n], keys=["kvasir", "cvc"])]
    kv = [[d.mean() for d in load_dice(CFG[n][0], SEEDS, "kvasir", CFG[n][1])] for n in names3]
    cv = [[d.mean() for d in load_dice(CFG[n][0], SEEDS, "cvc", CFG[n][1])] for n in names3]
    display(pd.DataFrame({"model": names3, "Kvasir mDice": np.round([np.mean(v) for v in kv], 3), "CVC mDice": np.round([np.mean(v) for v in cv], 3),
                          "|gap| between clients": np.round(np.abs(np.array([np.mean(v) for v in kv]) - np.array([np.mean(v) for v in cv])), 3)}), name="D4_client_gap")


# ----------------------------------------------------------------------------------------------------------------
@step("size_fail", "core", "Polyp-size stratification + failure cases (A3, K4)")
def step_size_fail():
    for tag, folder, key, title in (("cvc_main", CVC_DIR, "cvc", "CVC-ClinicDB (CVC-only)"), ("kvasir_main", KV_DIR, "kvasir", "Kvasir-SEG (Kvasir-only)")):
        ck = f"{folder}/{tag}_seed{SEEDS[0]}.weights.h5"
        if not os.path.exists(ck):
            print(f"[{title}] weights missing -> skip")
            continue
        model = load_trained(build, ck)
        Xte, Yte = TEST[key]
        _, per, probs = evaluate_arrays(model, Xte, Yte)
        lab, frac, th = size_bins(Yte)
        bins = dice_by_bin(per["dice"], lab)
        display(pd.DataFrame({b: {"mean Dice": round(v[0], 3), "n images": v[1]} for b, v in bins.items()}).T, name=f"size_bins_{key}")
        tiny = frac < 0.025
        print(f"{title}: tiny (<2.5% area): {int(tiny.sum())} images" + (f" | mean Dice {per['dice'][tiny].mean():.3f}" if tiny.any() else ""),
              "| per-image Dice mean/median/std:", round(per["dice"].mean(), 3), round(float(np.median(per["dice"])), 3), round(per["dice"].std(), 3))
        violin_plot({f"{b} (n={v[1]})": per["dice"][lab == b] for b, v in bins.items()}, f"{title}: Dice by polyp size"); plt.show()
        worst = np.argsort(per["dice"])[:6]
        fig, ax = plt.subplots(3, len(worst), figsize=(1.6 * len(worst), 5), squeeze=False)
        for j, i in enumerate(worst):
            ax[0, j].imshow(Xte[i]); ax[0, j].set_title(f"#{i} D={per['dice'][i]:.2f}", fontsize=8)
            ax[1, j].imshow(Yte[i, ..., 0], cmap="gray"); ax[2, j].imshow(probs[i, ..., 0] > 0.5, cmap="gray")
            for r in range(3):
                ax[r, j].axis("off")
        fig.suptitle("6 worst cases: image / ground truth / prediction"); plt.tight_layout(); plt.show()
        del model
        free_mem()


# ----------------------------------------------------------------------------------------------------------------
ROB_MODELS = ["CVC-only", "Kvasir-only", "Pooled (central)", "FedAvg", "FedProx", "FedBN"]


def robustness_for(names, seeds, sets, cfgmap, rob_dir, label):
    """robustness (5 corruptions x severity 0..5, FULL test sets) for every regime x seed; resumable pickles; returns {(name,seed): {set: {corr: [6 dice]}}}"""
    os.makedirs(rob_dir, exist_ok=True)
    m = build()
    res = {}
    for seed in seeds:
        for name in names:
            p = os.path.join(rob_dir, f"{_slug(name)}_seed{seed}.pkl")
            if os.path.exists(p):
                res[(name, seed)] = pickle.load(open(p, "rb"))
                continue
            tag, folder = cfgmap[name]
            out = {}
            for ds in sets:
                ck = (f"{folder}/{tag}_seed{seed}_client{ {'kvasir': 0, 'cvc': 1}.get(ds, 0) }.weights.h5" if name == "FedBN" else f"{folder}/{tag}_seed{seed}.weights.h5")
                if not os.path.exists(ck):
                    out = None
                    break
                m.load_weights(ck)
                out[ds] = robustness_eval(m, *TEST[ds])
            if out is None:
                print(f"[{label}: {name} seed {seed}] weights missing -> skip")
                continue
            pickle.dump(out, open(p, "wb"))
            res[(name, seed)] = out
            print(f"[{label}: {name} seed {seed}] robustness done", flush=True)
    free_mem()
    return res


def robustness_report(res, names, seeds, sets, title):
    present = [n for n in names if any((n, s) in res for s in seeds)]
    for ds in sets:
        rows = []
        for n in present:
            rr = [relative_robustness(res[(n, s)][ds]) for s in seeds if (n, s) in res]
            clean = [float(np.mean([v[0] for v in res[(n, s)][ds].values()])) for s in seeds if (n, s) in res]
            row = {"model": n, "seeds": len(rr), "clean mDice": fmt_ms(clean), "overall relative robustness": fmt_ms([r["overall"] for r in rr])}
            row.update({c: fmt_ms([r[c] for r in rr]) for c in rr[0] if c != "overall"})
            rows.append(row)
        display(pd.DataFrame(rows), name=f"{title}_{ds}", latex=True)
    fig, axs = plt.subplots(1, len(sets), figsize=(3.6 * len(sets), 3.2), squeeze=False)
    for ax, ds in zip(axs[0], sets):
        for n in present:
            cur = np.array([np.mean([v for v in res[(n, s)][ds].values()], axis=0) for s in seeds if (n, s) in res])
            mu, sd = cur.mean(0), (cur.std(0, ddof=1) if len(cur) > 1 else np.zeros(cur.shape[1]))
            ax.plot(range(cur.shape[1]), mu, marker="o", ms=3, label=n); ax.fill_between(range(cur.shape[1]), mu - sd, mu + sd, alpha=0.15)
        ax.set_title(NAMES[ds]); ax.set_xlabel("severity (0 = clean)"); ax.set_ylabel("mean Dice"); ax.grid(alpha=0.3)
    axs[0][0].legend(fontsize=8); plt.tight_layout(); plt.show()


@step("robust", "core", "Robustness: all six regimes x 3 seeds, full test sets, severity 0..5 (A4, D2, M1, K5)")
def step_robust():
    seeds, sets = SEEDS[:3], ["kvasir", "cvc"]
    res = robustness_for(ROB_MODELS, seeds, sets, CFG, os.path.join(FED, "robust_multiseed"), "robust")
    if res:
        robustness_report(res, ROB_MODELS, seeds, sets, "robustness")
        print("Check: do federated models lose MORE under corruption than pooled? (relative robustness lower than pooled) -> then 'robust' cannot be in the title.")


# ----------------------------------------------------------------------------------------------------------------
XAI_MODELS = ["CVC-only", "Kvasir-only", "Pooled (central)", "FedAvg", "FedProx", "FedBN"]


def _xai_pkl(name, seed):
    return os.path.join(FED, "xai_multiseed", _slug(name) + f"_seed{seed}.pkl")


@step("xai", "core", "XAI: Grad-CAM++/LIME, 6 regimes x 3 seeds on the SAME 162 images, dataset-wise, paired stats, model-randomisation check, figure (A5, B6, D1, H1, I1, J1, O1, K6)")
def step_xai():
    os.makedirs(os.path.join(FED, "xai_multiseed"), exist_ok=True)
    seeds = SEEDS[:3]
    xx = np.concatenate([TEST["kvasir"][0], TEST["cvc"][0]])
    yx = np.concatenate([TEST["kvasir"][1], TEST["cvc"][1]])
    n_kv = len(TEST["kvasir"][0])
    kw = dict(n_gc=5 if QUICK_TEST else len(xx), n_lime=N_LIME, lime_segments=150, lime_samples=1000)
    print(f"XAI: {len(xx)} images (Kvasir {n_kv} + CVC {len(xx) - n_kv}) | LIME n={N_LIME} | seeds {seeds}")
    per_seed = {}
    for seed in seeds:
        for name in XAI_MODELS:
            p = _xai_pkl(name, seed)
            if os.path.exists(p):
                per_seed[(name, seed)] = pickle.load(open(p, "rb"))
                continue
            tag, folder = CFG[name]
            if name != "FedBN":
                ck = f"{folder}/{tag}_seed{seed}.weights.h5"
                if not os.path.exists(ck):
                    continue
                m = load_trained(build, ck)
                r = xai_full_eval(m, xx, yx, **kw)
                del m
            else:
                cks = [f"{folder}/{tag}_seed{seed}_client{c}.weights.h5" for c in (0, 1)]
                if not all(os.path.exists(c) for c in cks):
                    continue
                rs = []
                for ck in cks:
                    m = load_trained(build, ck)
                    rs.append(xai_full_eval(m, xx, yx, **kw))
                    del m
                    free_mem()
                order = np.random.default_rng(0).permutation(len(xx))
                r = {}
                for method, df0 in rs[0].items():
                    from_kv = order[:len(df0)] < n_kv
                    r[method] = pd.DataFrame(np.where(from_kv[:, None], df0.values, rs[1][method].values), columns=df0.columns)
            free_mem()
            pickle.dump(r, open(p, "wb"))
            per_seed[(name, seed)] = r
            print(f"[XAI {name} seed {seed}] done", flush=True)
    if not per_seed:
        print("no XAI results (weights missing)")
        return
    rows = []
    for (name, seed), r in per_seed.items():
        for method, df in r.items():
            for col in df.columns:
                v = pd.to_numeric(df[col], errors="coerce").dropna()
                if len(v):
                    rows.append({"model": name, "seed": seed, "method": method, "metric": col, "value": float(v.mean())})
    agg = pd.DataFrame(rows)
    order_models = [m for m in XAI_MODELS if m in agg["model"].unique()]
    g = agg.groupby(["method", "metric", "model"])["value"]
    summ = pd.DataFrame({"mean": g.mean(), "sd": g.std(ddof=1)}).reset_index()
    summ["txt"] = [f"{m:.3f} \u00b1 {0 if pd.isna(s) else s:.3f}" for m, s in zip(summ["mean"], summ["sd"])]
    display(summ.pivot_table(index=["method", "metric"], columns="model", values="txt", aggfunc="first").reindex(columns=order_models), name="XAI_mean_sd_over_seeds", latex=True)

    def per_image(name, method, col, subset=None):
        arrs = [np.asarray(pd.to_numeric(per_seed[(name, sd)][method][col], errors="coerce"), float) for sd in seeds
                if (name, sd) in per_seed and method in per_seed[(name, sd)] and col in per_seed[(name, sd)][method].columns]
        return arrs
    # dataset-wise table (I1) + paired statistics vs pooled (J1, M3-consistent)
    order = np.random.default_rng(0).permutation(len(xx))
    is_kv_all = order < n_kv
    ref = "Pooled (central)"
    for metric in ("iou_topg", "auprc"):
        full = {m: per_image(m, "gradcam++", metric) for m in order_models}
        full = {m: a for m, a in full.items() if a}
        if ref not in full:
            continue
        L = min(len(a) for arrs in full.values() for a in arrs)
        is_kv = is_kv_all[:L]
        for sname, sel in (("ALL 162", np.ones(L, bool)), ("Kvasir-SEG", is_kv), ("CVC-ClinicDB", ~is_kv)):
            valid = sel.copy()
            for arrs in full.values():
                for a in arrs:
                    valid &= np.isfinite(a[:L])
            display(compare_table({m: [a[:L][valid] for a in arrs] for m, arrs in full.items()}, ref), name=f"XAI_gradcam_{metric}_{_slug(sname)}_vs_pooled")
    # model-randomisation sanity check (O1)
    rows = []
    for name in XAI_MODELS:
        for ds in ("kvasir", "cvc"):
            tag, folder = CFG[name]
            ck = regime_ckpt(name, SEEDS[0], ds)
            if not os.path.exists(ck):
                continue
            m = load_trained(build, ck)
            rc = randomization_check(m, lambda: build_unet_resnet50(IMG_SIZE, weights=None), *TEST[ds], n=5 if QUICK_TEST else 20)
            rows.append({"model": name, "test set": NAMES[ds], **{k: round(float(v), 3) for k, v in rc.mean().items()}})
            del m
            free_mem()
    display(pd.DataFrame(rows), name="XAI_model_randomisation_check")
    # qualitative figures (best / median / worst case): CVC-only on CVC-ClinicDB (Fig. 7) and Kvasir-only on Kvasir-SEG (Fig. 8)
    for tag, folder, key in (("cvc_main", CVC_DIR, "cvc"), ("kvasir_main", KV_DIR, "kvasir")):
        ck = f"{folder}/{tag}_seed{SEEDS[0]}.weights.h5"
        if not os.path.exists(ck):
            continue
        m = load_trained(build, ck)
        Xc, Yc = TEST[key]
        _, per, _ = evaluate_arrays(m, Xc, Yc)
        o = np.argsort(per["dice"])
        xai_figure(m, Xc, Yc, [o[-1], o[len(o) // 2], o[0]], lime_samples=300)
        del m
        free_mem()


# ----------------------------------------------------------------------------------------------------------------
def measure_efficiency():
    m = build()
    try:
        return efficiency_report(m, IMG_SIZE)
    finally:
        del m
        free_mem()


@step("efficiency", "core", "Params / GFLOPs / latency / FPS, accuracy-vs-efficiency plots (A7, K8, G1, L1, V3, K13)")
def step_efficiency():
    rep = measure_efficiency()
    gm = rep["gflops"] / 2 if rep.get("gflops") is not None else None
    rows = [{"method": k, "params (M)": p if p is not None else "-", "GMac @352": g if g is not None else "-"} for k, (p, g) in PUB_EFF.items()]
    rows.append({"method": "Ours: ResNet50 U-Net (all six regimes)", "params (M)": round(rep["params_M"], 2), "GMac @352": round(gm, 1) if gm else "-"})
    display(pd.DataFrame(rows), name="G1_efficiency", latex=True)
    print(f"Ours: {rep['params_M']:.2f} M params, {gm and round(gm, 1)} GMac (TF profiler FLOPs / 2), latency {rep['latency_ms']:.1f} ms ({rep['fps']:.1f} FPS). "
          "Published numbers = third-party measurements; UACANet S/L need measuring with the official code.")
    # L1: inference time
    def bench(fn, n_warm, n_run):
        for _ in range(n_warm):
            fn()
        t = []
        for _ in range(n_run):
            t0 = time.perf_counter(); fn(); t.append((time.perf_counter() - t0) * 1000.0)
        return np.array(t)
    stats_ = lambda t: {"mean \u00b1 SD (ms)": f"{t.mean():.1f} \u00b1 {t.std(ddof=1):.1f}", "median (ms)": round(float(np.median(t)), 1), "p95 (ms)": round(float(np.percentile(t, 95)), 1)}
    out = []
    devs = (["/GPU:0"] if tf.config.list_physical_devices("GPU") else []) + ["/CPU:0"]
    for dev in devs:
        n_run = 200 if dev.startswith("/GPU") else 15
        with tf.device(dev):
            m = build()
            ck = f"{FED}/pooled_seed{SEEDS[0]}.weights.h5"
            if os.path.exists(ck):
                m.load_weights(ck)
            x8 = TEST["cvc"][0][:8]
            xp1, xp8 = to_model_input(x8[:1]), to_model_input(x8)
            t = bench(lambda: m.predict_on_batch(xp1), 20, n_run); out.append({"device": dev, "setting": "model only, batch 1", **stats_(t), "FPS": round(1000 / t.mean(), 1)})
            t = bench(lambda: m.predict_on_batch(xp8), 5, max(10, n_run // 4)); out.append({"device": dev, "setting": "model only, batch 8 (per image)", **stats_(t / 8.0), "FPS": round(8000 / t.mean(), 1)})
            t = bench(lambda: binarize(m.predict_on_batch(to_model_input(x8[:1])), 0.5, False), 20, n_run); out.append({"device": dev, "setting": "end-to-end, batch 1", **stats_(t), "FPS": round(1000 / t.mean(), 1)})
            tf.keras.backend.clear_session()
    display(pd.DataFrame(out), name="L1_inference_time")
    print(f"Report GPU name, batch size, resolution {IMG_SIZE}x{IMG_SIZE}, TF {tf.__version__} in the paper (latency is hardware specific).")
    # V3 / K13: accuracy-vs-efficiency
    if have("pooled", FED) and gm:
        ours = [float(np.mean([a.mean() for a in load_dice("pooled", SEEDS, k, FED)])) for k in TEST]
        fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.3))
        for ax, idx, title in ((axs[0], None, "mean mDice over 5 test sets"), (axs[1], 1, "mDice on CVC-ClinicDB")):
            for name, (p, g) in PUB_EFF.items():
                if p is None:
                    continue
                d = PUBLISHED[name]
                y = float(np.mean(d)) if idx is None else d[idx]
                ax.scatter(g, y, s=p * 6, alpha=0.6); ax.annotate(name.split(" (")[0], (g, y), textcoords="offset points", xytext=(5, 4), fontsize=8)
            y = float(np.mean(ours)) if idx is None else ours[idx]
            ax.scatter(gm, y, s=rep["params_M"] * 6, marker="*", color="red"); ax.annotate("Ours (pooled)", (gm, y), textcoords="offset points", xytext=(5, -12), fontsize=8)
            ax.set_xlabel("GMac @352"); ax.set_ylabel("mDice"); ax.set_title(title); ax.grid(alpha=0.3)
        plt.tight_layout(); plt.show()


# ----------------------------------------------------------------------------------------------------------------
@step("pr_hd95", "core", "Precision/recall (empties handled, S5), PR curve, HD95 + S-measure for all regimes incl. FedBN (K1, X1, D3, N1, K3)")
def step_pr_hd95():
    avail = {n: v for n, v in CFG.items() if has_results(*v)}
    m = build()
    pr_dir = os.path.join(FED, "pr_metrics")
    os.makedirs(pr_dir, exist_ok=True)

    def per_image_pr(mdl, Xt, Yt):
        _, per, probs = evaluate_arrays(mdl, Xt, Yt, bs=4)
        tp, fp, fn = per["tp"], per["fp"], per["fn"]
        return (np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), np.nan), np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), np.nan),
                per["dice"], (tp + fp == 0).astype(float), probs)

    def seed_eval(name, seed, ds_keys):
        """per-set (prec, rec, dice, empty, hd95, s_measure, n_hd_empty); FedBN: in-domain set with own client BN, other sets: mean over both clients"""
        tag, folder = CFG[name]
        clients = {k: ([{"kvasir": 0, "cvc": 1}[k]] if (name == "FedBN" and k in ("kvasir", "cvc")) else ([0, 1] if name == "FedBN" else [None])) for k in ds_keys}
        out = {}
        for k in ds_keys:
            acc = []
            for c in clients[k]:
                ck = f"{folder}/{tag}_seed{seed}_client{c}.weights.h5" if c is not None else f"{folder}/{tag}_seed{seed}.weights.h5"
                if not os.path.exists(ck):
                    return None
                m.load_weights(ck)
                X, Y = TEST[k]
                p, r, d, e, probs = per_image_pr(m, X, Y)
                hd, ne = hd95_batch(Y, probs)
                sm = np.array([s_measure(probs[i, ..., 0], Y[i, ..., 0]) for i in range(len(X))])
                acc.append((p, r, d, e, hd, sm, ne))
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                out[k] = tuple(np.nanmean(np.stack([a[i] for a in acc]), 0) if i < 6 else float(np.mean([a[6] for a in acc])) for i in range(7))
        return out

    rows = []
    for name in avail:
        for seed in SEEDS:
            cp = os.path.join(pr_dir, f"{_slug(name)}_seed{seed}.pkl")
            if os.path.exists(cp):
                got = pickle.load(open(cp, "rb"))
            else:
                got = seed_eval(name, seed, list(TEST))
                if got is None:
                    continue
                pickle.dump(got, open(cp, "wb"))
            for k, (p, r, d, e, hd, sm, ne) in got.items():
                valid = np.isfinite(p)
                rows.append({"model": name, "seed": seed, "set": k, "precision (empties dropped)": float(np.nanmean(p)) if valid.any() else np.nan,
                             "precision (empties = 0)": float(np.where(valid, p, 0.0).mean()), "recall": float(np.nanmean(r)), "dice": float(np.mean(d)),
                             "n empty predictions": float(np.sum(e > 0.5)), "empty (%)": 100 * float(np.mean(e > 0.5)), "HD95 (px)": float(np.mean(hd)),
                             "S-measure": float(np.mean(sm)), "HD95 diag-penalised n": ne})
    free_mem()
    df = pd.DataFrame(rows)
    if df.empty:
        print("no results")
        return
    order = [n for n in avail if n in df["model"].unique()]
    for k in TEST:
        sub = df[df["set"] == k]
        tab = pd.DataFrame([{"model": n + (" (personalised BN on seen sites)" if n == "FedBN" and k in ("kvasir", "cvc") else ""),
                             "precision (empties dropped)": fmt_ms(sub[sub.model == n]["precision (empties dropped)"]),
                             "precision (empties = 0)": fmt_ms(sub[sub.model == n]["precision (empties = 0)"]), "recall": fmt_ms(sub[sub.model == n]["recall"]),
                             "n empty pred (mean/seed)": f"{sub[sub.model == n]['n empty predictions'].mean():.1f} of {len(TEST[k][0])}",
                             "HD95 px": fmt_ms(sub[sub.model == n]["HD95 (px)"], 1), "S-measure": fmt_ms(sub[sub.model == n]["S-measure"])} for n in order])
        display(tab, name=f"K1_precision_recall_hd95_{k}", latex=True)
    # PR curve + bar chart for CVC-only (X1) and Kvasir-only (K3)
    for tag, folder, key, title in (("cvc_main", CVC_DIR, "cvc", "CVC-ClinicDB (CVC-only)"), ("kvasir_main", KV_DIR, "kvasir", "Kvasir-SEG (Kvasir-only)")):
        ck = f"{folder}/{tag}_seed{SEEDS[0]}.weights.h5"
        if os.path.exists(ck):
            mm = load_trained(build, ck)
            Xt, Yt = TEST[key]
            summ, _, _ = evaluate_arrays(mm, Xt, Yt)
            pr_curve_and_bar(mm, Xt, Yt, summ, title=f"{title}: headline metrics"); plt.show()
            hd95_report(mm, Xt, Yt, label=f"{title}: ")
            del mm
            free_mem()


def s_measure(pred, gt, eps=1e-8):
    """Structure-measure S_alpha (Fan et al., ICCV 2017; alpha = 0.5). pred: probability map (H,W); gt: binary mask (H,W)."""
    pred = np.asarray(pred, np.float64)
    gt = (np.asarray(gt) > 0.5).astype(np.float64)
    y = gt.mean()
    if y == 0:
        return float(1.0 - pred.mean())
    if y == 1:
        return float(pred.mean())

    def s_obj(p, g):
        x = p[g == 1].mean()
        sd = p[g == 1].std(ddof=1) if (g == 1).sum() > 1 else 0.0
        return 2 * x / (x ** 2 + 1 + sd + eps)
    s_object = y * s_obj(pred * gt, gt) + (1 - y) * s_obj((1 - pred) * (1 - gt), 1 - gt)
    h, w = gt.shape
    rows, cols = np.arange(1, h + 1), np.arange(1, w + 1)
    cx = int(np.clip(round((gt.sum(0) * cols).sum() / gt.sum()), 1, w - 1))
    cy = int(np.clip(round((gt.sum(1) * rows).sum() / gt.sum()), 1, h - 1))

    def ssim(p, g):
        if p.size == 0:
            return 0.0
        n = p.size
        mp, mg = p.mean(), g.mean()
        vp, vg, cpg = ((p - mp) ** 2).sum() / (n - 1 + eps), ((g - mg) ** 2).sum() / (n - 1 + eps), ((p - mp) * (g - mg)).sum() / (n - 1 + eps)
        a, b = 4 * mp * mg * cpg, (mp ** 2 + mg ** 2) * (vp + vg)
        return a / (b + eps) if a != 0 else (1.0 if b == 0 else 0.0)
    area = h * w
    parts = [(slice(0, cy), slice(0, cx), cx * cy / area), (slice(0, cy), slice(cx, w), (w - cx) * cy / area),
             (slice(cy, h), slice(0, cx), cx * (h - cy) / area), (slice(cy, h), slice(cx, w), (w - cx) * (h - cy) / area)]
    return float(max(0.0, 0.5 * s_object + 0.5 * sum(wt * ssim(pred[r, c], gt[r, c]) for r, c, wt in parts)))


# ----------------------------------------------------------------------------------------------------------------
@step("valtest", "core", "Val vs test Dice (V1, P1) + exact per-image validation Dice vs Keras batch-average (S5)")
def step_valtest():
    ms = lambda x: fmt_ms(x)
    n_kv, n_cvc = len(TEST["kvasir"][0]), len(TEST["cvc"][0])

    def test_dice(tag, folder, seed, keys, w=None):
        z = np.load(f"{folder}/{tag}_seed{seed}.npz")
        vals = [float(np.mean(z[f"dice__{k}"])) for k in keys]
        return float(np.average(vals, weights=w)) if w else float(np.mean(vals))
    rows = []
    for name, tag, folder, keys, w in (("CVC-only", "cvc_main", CVC_DIR, ["cvc"], None), ("Kvasir-only", "kvasir_main", KV_DIR, ["kvasir"], None),
                                       ("Pooled (central)", "pooled", FED, ["kvasir", "cvc"], [n_kv, n_cvc])):
        v, t = [], []
        for s in SEEDS:
            jp, zp = f"{folder}/{tag}_seed{s}.json", f"{folder}/{tag}_seed{s}.npz"
            if os.path.exists(jp) and os.path.exists(zp):
                v.append(best_val_dice(json.load(open(jp)))); t.append(test_dice(tag, folder, s, keys, w))
        if v:
            rows.append({"model": name, "seeds": len(v), "val Dice (best ckpt / last round)": ms(v), "test mDice": ms(t), "val - test": ms(np.array(v) - np.array(t))})
    for name, mode in FL_MODES.items():
        v, t = [], []
        for s in SEEDS:
            jp, zp = f"{FED}/fl_{mode}_seed{s}_history.json", f"{FED}/fl_{mode}_seed{s}.npz"
            if os.path.exists(jp) and os.path.exists(zp):
                v.append(float(json.load(open(jp))[-1]["val_dice_mean"])); t.append(test_dice(f"fl_{mode}", FED, s, ["kvasir", "cvc"]))
        if v:
            rows.append({"model": name, "seeds": len(v), "val Dice (best ckpt / last round)": ms(v), "test mDice": ms(t), "val - test": ms(np.array(v) - np.array(t))})
    display(pd.DataFrame(rows), name="P1_val_vs_test")
    # S5: exact per-image validation Dice of the SAVED checkpoints vs the Keras number that selected them
    rows = []
    for name, tag, folder, keys in (("CVC-only", "cvc_main", CVC_DIR, ["cvc"]), ("Kvasir-only", "kvasir_main", KV_DIR, ["kvasir"]), ("Pooled (central)", "pooled", FED, ["kvasir", "cvc"])):
        diffs, exact = [], []
        for s in SEEDS:
            ck, jp = f"{folder}/{tag}_seed{s}.weights.h5", f"{folder}/{tag}_seed{s}.json"
            if not (os.path.exists(ck) and os.path.exists(jp)):
                continue
            mdl = load_trained(build, ck)
            Xv = np.concatenate([D[k]["val"][0] for k in keys])
            Yv = np.concatenate([D[k]["val"][1] for k in keys])
            _, per, _ = evaluate_arrays(mdl, Xv, Yv)
            exact.append(float(per["dice"].mean()))
            diffs.append(exact[-1] - best_val_dice(json.load(open(jp))))
            del mdl
            free_mem()
        if exact:
            rows.append({"model": name, "per-image val mDice (exact)": ms(exact), "exact - logged val Dice at best ckpt": ms(diffs)})
    display(pd.DataFrame(rows), name="S5_val_dice_per_image_vs_keras")
    print("The Keras metric is a batch average of per-image Dice (last batch weighted equally); exact per-image values differ only marginally -> report the exact value.")
    # S5 (selection): would the exact per-image metric have picked another epoch than the Keras batch-average? (runs trained with this script version log val_dice_img every epoch)
    rows = []
    for name, tag, folder in (("CVC-only", "cvc_main", CVC_DIR), ("Kvasir-only", "kvasir_main", KV_DIR), ("Pooled (central)", "pooled", FED)):
        same, gap, used = [], [], set()
        for s in SEEDS:
            jp = f"{folder}/{tag}_seed{s}.json"
            if not os.path.exists(jp):
                continue
            h = json.load(open(jp))
            if not h.get("val_dice_img"):
                continue
            ek, ei = int(np.argmax(h["val_dice_coef"])), int(np.argmax(h["val_dice_img"]))
            same.append(ek == ei)
            gap.append(h["val_dice_img"][ei] - h["val_dice_img"][ek])
            used.add("per-image" if (h.get("val_select_per_image") or [0])[0] > 0.5 else "Keras batch-avg")
        if same:
            rows.append({"model": name, "runs with per-image log": len(same), "checkpoint selected by": "/".join(sorted(used)), "same best epoch (both metrics)": f"{sum(same)}/{len(same)}",
                         "exact val Dice lost by Keras-selected epoch": fmt_ms(gap, 4)})
    if rows:
        display(pd.DataFrame(rows), name="S5_selection_agreement")
        print("If 'same best epoch' is (nearly) always full and the lost Dice is ~0, the batch-average metric is harmless -> one sentence in the paper. Otherwise retrain ALL centralised regimes with --val-per-image.")
    else:
        print("S5 selection check: no per-image validation log yet (models trained before this version have none). New runs (aug2, m8, retrained models) create it automatically.")


@step("tta", "core", "TTA / post-processing / threshold isolation on the Kvasir-only model (K7c)")
def step_tta():
    ck = f"{KV_DIR}/kvasir_main_seed{SEEDS[0]}.weights.h5"
    if not os.path.exists(ck):
        print("kvasir_main weights missing -> skip")
        return
    m = load_trained(build, ck)
    display(tta_post_threshold_ablation(m, *D["kvasir"]["val"], *TEST["kvasir"]), name="K7c_tta_post_threshold")
    del m
    free_mem()


# ================================================================================================================
# ================================================================================================================
#                                   NEW REVIEW STEPS  (M2, M6, M8, S2, S3, S4, repro)
# ================================================================================================================
# ================================================================================================================
def _gap_table(pairs, keys, title):
    """pairs: [(label, (tagA, folderA), (tagB, folderB))] -> gap = A - B per test set (positive = A better), sign-flip p (Holm over sets), unadjusted CI."""
    rows = []
    for label, (ta, fa), (tb, fb) in pairs:
        if not (has_results(ta, fa) and has_results(tb, fb)):
            rows.append({"comparison": label, "note": "results missing"})
            continue
        ds, ps, cis = {}, [], {}
        for k in keys:
            d = np.mean(np.stack(load_per_image(ta, fa, k)), 0) - np.mean(np.stack(load_per_image(tb, fb, k)), 0)
            ds[k] = d
            ps.append(sign_flip_p(d))
            cis[k] = bootstrap_ci(d)
        ph = holm(ps)
        for k, p, a in zip(keys, ps, ph):
            rows.append({"comparison": label, "test set": {**NAMES, "unseen": "Unseen (image-weighted)"}[k], "gap (A-B)": round(float(ds[k].mean()), 4),
                         "CI95 (unadj.)": f"[{cis[k][0]:+.3f}, {cis[k][1]:+.3f}]", "p_perm": round(p, 4), "p_perm_holm": round(float(a), 4)})
    t = pd.DataFrame(rows)
    display(t, name=title, latex=True)
    return t


@step("m2", "review", "M2: protocol-control arms - pooled (const LR, last epoch) and FedAvg (best round, LR decay at rounds 10/15) [+ optional keep-Adam-state arm]")
def step_m2():
    if not RUN_M2:
        print("RUN_M2 = False -> skip")
        return
    free_mem()
    run_seeds(TAG_POOL_CONST, SEEDS, build, POOL_TR, POOL_VA, TEST, epochs=EP_POOL, batch=BATCH, out_dir=FED, use_plateau=False, select_best=False)
    print("M2 FedAvg-tuned config:", M2_TUNED_CFG)
    fl_run_seeds(TAG_FEDAVG_TUNED, "fedavg", SEEDS, M2_TUNED_CFG)
    if M2_KEEP_OPT:
        fl_run_seeds(TAG_FEDAVG_KEEPOPT, "fedavg", SEEDS, {**M2_TUNED_CFG, "keep_opt": True})
    arms = {"Pooled (main: LR schedule + best ckpt)": ("pooled", FED), "Pooled (a: constant LR, last epoch)": (TAG_POOL_CONST, FED),
            "FedAvg (main: constant LR, last round)": ("fl_fedavg", FED), "FedAvg (b: best round + LR decay)": (TAG_FEDAVG_TUNED, FED)}
    if M2_KEEP_OPT:
        arms["FedAvg (b + client Adam state kept)"] = (TAG_FEDAVG_KEEPOPT, FED)
    keys = list(TEST) + ["unseen"]
    rows = [{"arm": n, **{({**NAMES, 'unseen': 'Unseen (image-weighted)'})[k]: fmt_ms([a.mean() for a in load_per_image(t, f, k)]) for k in keys}}
            for n, (t, f) in arms.items() if has_results(t, f)]
    display(pd.DataFrame(rows), name="M2_arms_mdice", latex=True)
    sel = [json.load(open(p))["selected_round"] for p in (f"{FED}/{TAG_FEDAVG_TUNED}_seed{s}_selected_round.json" for s in SEEDS) if os.path.exists(p)]
    print("FedAvg (b): selected round per seed:", sel)
    P, Pa, F, Fb = arms["Pooled (main: LR schedule + best ckpt)"], arms["Pooled (a: constant LR, last epoch)"], arms["FedAvg (main: constant LR, last round)"], arms["FedAvg (b: best round + LR decay)"]
    t = _gap_table([("main protocols: Pooled(main) - FedAvg(main)", P, F), ("matched, constant LR / last model: Pooled(a) - FedAvg(main)", Pa, F),
                    ("matched, schedule + best ckpt: Pooled(main) - FedAvg(b)", P, Fb)], keys, "M2_pooled_minus_fedavg_gaps")
    cv = t[(t.get("test set") == "CVC-ClinicDB")] if "test set" in t else pd.DataFrame()
    if len(cv):
        for _, r in cv.iterrows():
            print(f"   {r['comparison']}: gap on CVC-ClinicDB = {r['gap (A-B)']:+.3f}  ->", "conclusion holds (>= 0.03)" if r["gap (A-B)"] >= 0.03 else "BELOW 0.03: the protocol confound explains part of the gap - say so in the paper")


@step("s2", "review", "S2: FedAvg with equal aggregation weights (0.5/0.5) - tests 'the smaller client pays most of the cost'")
def step_s2():
    if not RUN_S2:
        print("RUN_S2 = False -> skip")
        return
    fl_run_seeds(TAG_FEDAVG_EQ, "fedavg", SEEDS, {"agg": "equal"})
    keys = list(TEST) + ["unseen"]
    arms = {"Pooled": ("pooled", FED), "FedAvg (size weights, default)": ("fl_fedavg", FED), "FedAvg (equal weights 0.5/0.5)": (TAG_FEDAVG_EQ, FED)}
    display(pd.DataFrame([{"arm": n, **{({**NAMES, 'unseen': 'Unseen (image-weighted)'})[k]: fmt_ms([a.mean() for a in load_per_image(t, f, k)]) for k in keys}}
                          for n, (t, f) in arms.items() if has_results(t, f)]), name="S2_equal_weights_mdice", latex=True)
    _gap_table([("Pooled - FedAvg(size)", arms["Pooled"], arms["FedAvg (size weights, default)"]), ("Pooled - FedAvg(equal)", arms["Pooled"], arms["FedAvg (equal weights 0.5/0.5)"]),
                ("FedAvg(equal) - FedAvg(size)", arms["FedAvg (equal weights 0.5/0.5)"], arms["FedAvg (size weights, default)"])], keys, "S2_gaps")
    print("If the CVC-ClinicDB gap to pooled SHRINKS with equal weights (and the Kvasir gap grows) the 'smaller client pays most of the cost' explanation is confirmed; otherwise drop it.")


@step("m6", "review", "M6: FedBN as ONE deployable model - probability-map ensemble and AdaBN (seen-site FedBN is personalised)")
def step_m6():
    todo = [s for s in SEEDS if not (os.path.exists(f"{FED}/{TAG_FEDBN_ENS}_seed{s}.npz") and os.path.exists(f"{FED}/{TAG_FEDBN_ADABN}_seed{s}.npz"))]
    for s in todo:
        if not all(os.path.exists(f"{FED}/fl_fedbn_seed{s}_client{c}.weights.h5") for c in (0, 1)):
            print(f"[seed {s}] FedBN client weights missing -> run train_fl first")
            continue
        for tag, fn in ((TAG_FEDBN_ENS, fedbn_ensemble_eval), (TAG_FEDBN_ADABN, fedbn_adabn_eval)):
            if os.path.exists(f"{FED}/{tag}_seed{s}.npz"):
                continue
            try:
                res = fn(FED, "fl_fedbn", s, TEST)
                save_fed_result(tag, s, res, FED)
                print(f"[{tag} seed {s}]", {NAMES[k]: round(float(v[0].mean()), 4) for k, v in res.items()}, flush=True)
            except Exception as e:
                print(f"[{tag} seed {s}] FAILED: {type(e).__name__}: {str(e)[:200]}")
            free_mem()
    arms = {"Pooled (central)": ("pooled", FED), "FedAvg": ("fl_fedavg", FED), "FedProx": ("fl_fedprox", FED),
            "FedBN (per-client BN: seen = personalised, unseen = mean of 2 scores)": ("fl_fedbn", FED),
            "FedBN-ensemble (probability-map average)": (TAG_FEDBN_ENS, FED), "FedBN-AdaBN (shared weights, BN re-estimated on target)": (TAG_FEDBN_ADABN, FED)}
    arms = {n: v for n, v in arms.items() if has_results(*v)}
    keys = list(TEST) + ["unseen"]
    display(pd.DataFrame([{"regime": n, **{({**NAMES, 'unseen': 'Unseen (image-weighted)'})[k]: fmt_ms([a.mean() for a in load_per_image(t, f, k)]) for k in keys}}
                          for n, (t, f) in arms.items()]), name="M6_fedbn_deployable_mdice", latex=True)
    ref = "Pooled (central)"
    if ref in arms:
        for k in keys:
            display(compare_table({n: load_per_image(t, f, k) for n, (t, f) in arms.items()}, ref), name=f"M6_vs_pooled_{k}")
    print("Report the ensemble or AdaBN row as 'FedBN (deployable)'; keep the per-client row only as 'FedBN (personalised)' on the two seen sites.")


def _val_summary(name, tag, folder, seeds):
    """best validation Dice (centralised, Keras batch-average) or final-round mean client validation Dice (federated) per seed"""
    out = []
    for s in seeds:
        if name in FL_MODES:
            p = f"{folder}/{tag}_seed{s}_history.json"
            if os.path.exists(p):
                out.append(float(json.load(open(p))[-1]["val_dice_mean"]))
        else:
            p = f"{folder}/{tag}_seed{s}.json"
            if os.path.exists(p):
                out.append(best_val_dice(json.load(open(p))))
    return out


@step("aug2", "core", "Stronger augmentation (YOUR aug2 recipe): retrain all 6 regimes x 5 seeds into *_aug2 folders, Table XI, Supp. S4/S5, corruption test (review S4)")
def step_aug2():
    if not RUN_AUG2:
        print("RUN_AUG2 = False -> skip")
        return
    d_cvc, d_kv, d_fed = (f"results_{n}_aug2{SUFFIX}" for n in ("cvc", "kvasir", "fed"))
    for d in (d_cvc, d_kv, d_fed):
        os.makedirs(d, exist_ok=True)
    cfg2 = {"CVC-only": ("cvc_main", d_cvc), "Kvasir-only": ("kvasir_main", d_kv), "Pooled (central)": ("pooled", d_fed),
            "FedAvg": ("fl_fedavg", d_fed), "FedProx": ("fl_fedprox", d_fed), "FedBN": ("fl_fedbn", d_fed)}
    print("aug2 folders:", d_cvc, d_kv, d_fed, "| training augmentation = _augment_strong (your v2: zoom 0.70-1.0, brightness/contrast, saturation, hue, light noise)")
    with use_context(dict(CVC_DIR=d_cvc, KV_DIR=d_kv, FED=d_fed, CFG=cfg2, TRAIN_AUG="strong")):
        step_train_central()
        step_train_fl()
    ok = [n for n in cfg2 if has_results(*cfg2[n]) and has_results(*CFG[n])]
    if not ok:
        print("no aug2 results yet")
        return
    keys = list(TEST)
    # ---- Supplementary Table S4: full mDice (aug2) ; mIoU ; validation ----
    display(pd.DataFrame([{"regime": n, **{NAMES[k]: fmt_ms([a.mean() for a in load_dice(cfg2[n][0], SEEDS, k, cfg2[n][1])]) for k in keys}} for n in ok]), name="aug2_S4_mdice_stronger", latex=True)
    display(pd.DataFrame([{"regime": n, **{NAMES[k]: fmt_ms([float(np.mean(np.load(f"{cfg2[n][1]}/{cfg2[n][0]}_seed{s}.npz")[f"iou__{k}"])) for s in SEEDS]) for k in keys}} for n in ok]), name="aug2_S5_miou_stronger", latex=True)
    # ---- Table XI: change in mDice (stronger minus milder) + the paper's '4 of 30' rule + validation ----
    rows, exceed, total, mean_changes, val_changes = [], 0, 0, [], []
    for n in ok:
        r = {"regime": n}
        deltas = []
        for k in keys:
            old = np.array([a.mean() for a in load_dice(CFG[n][0], SEEDS, k, CFG[n][1])])
            new = np.array([a.mean() for a in load_dice(cfg2[n][0], SEEDS, k, cfg2[n][1])])
            d = float(new.mean() - old.mean())
            sd = max(old.std(ddof=1) if len(old) > 1 else 0.0, new.std(ddof=1) if len(new) > 1 else 0.0)
            exceed += int(abs(d) > sd)
            total += 1
            deltas.append(d)
            r[NAMES[k]] = f"{d:+.3f}"
        r["Mean"] = f"{np.mean(deltas):+.3f}"
        mean_changes.append(float(np.mean(deltas)))
        vo, vn = _val_summary(n, CFG[n][0], CFG[n][1], SEEDS), _val_summary(n, cfg2[n][0], cfg2[n][1], SEEDS)
        if vo and vn:
            val_changes.append(float(np.mean(vn) - np.mean(vo)))
            r["validation Dice change"] = f"{val_changes[-1]:+.3f}"
        rows.append(r)
    display(pd.DataFrame(rows), name="aug2_table_XI_change_in_mdice", latex=True)
    print(f"mean change over the five test sets ranges {min(mean_changes):+.3f} .. {max(mean_changes):+.3f}; {exceed} of {total} comparisons exceed the larger of the two seed-to-seed SDs"
          + (f"; validation Dice change {min(val_changes):+.3f} .. {max(val_changes):+.3f}" if val_changes else ""))
    # ---- federated vs pooled under the stronger augmentation (does the conclusion change? FedProx-ETIS check) ----
    ref = "Pooled (central)"
    if ref in ok:
        for k in keys + ["unseen"]:
            display(compare_table({n: load_per_image(cfg2[n][0], cfg2[n][1], k) for n in ok}, ref), name=f"aug2_vs_pooled_{k}")
    # ---- corruption test (review S4): can augmentation fix the Gaussian-noise weakness? ----
    seeds, sets = SEEDS[:3], ["kvasir", "cvc"]
    res_new = robustness_for(ok, seeds, sets, cfg2, os.path.join(d_fed, "robust_multiseed"), "aug2")
    res_old = robustness_for(ok, seeds, sets, CFG, os.path.join(FED, "robust_multiseed"), "main")
    if res_new:
        robustness_report(res_new, ok, seeds, sets, "S4_robustness_aug2")
        rows = []
        for n in ok:
            for ds in sets:
                o = [res_old[(n, s)][ds]["Gaussian Noise"] for s in seeds if (n, s) in res_old]
                w = [res_new[(n, s)][ds]["Gaussian Noise"] for s in seeds if (n, s) in res_new]
                if o and w:
                    o, w = np.array(o), np.array(w)
                    rows.append({"regime": n, "test set": NAMES[ds], "clean Dice milder": fmt_ms(o[:, 0]), "clean Dice stronger": fmt_ms(w[:, 0]),
                                 "noise rel. robustness milder": fmt_ms(o[:, 1:].mean(1) / o[:, 0]), "noise rel. robustness stronger": fmt_ms(w[:, 1:].mean(1) / w[:, 0]),
                                 "Dice @ noise sev 5 milder": fmt_ms(o[:, 5]), "Dice @ noise sev 5 stronger": fmt_ms(w[:, 5])})
        display(pd.DataFrame(rows), name="S4_noise_main_vs_aug2", latex=True)
        print("If 'noise rel. robustness stronger' is still clearly below the other corruptions / pooled, augmentation does NOT fix the noise weakness - say so in the paper.")


# ----------------------------------------------------------------------------------------------------------------
def _topk_iou(cam, mask):
    mask = np.asarray(mask).squeeze() > 0.5
    k = int(mask.sum())
    if k == 0 or k == mask.size:
        return float("nan")
    flat = np.asarray(cam).ravel()
    top = np.zeros(flat.shape, bool)
    top[np.argpartition(-flat + 1e-9 * np.random.default_rng(0).random(flat.shape), k - 1)[:k]] = True
    top = top.reshape(mask.shape)
    return float((top & mask).sum() / ((top | mask).sum() + 1e-12))


@step("xai_layers", "review", "S3: explanation top-g IoU vs Dice correlation + decoder (dec4_relu2) vs ENCODER (conv5_block3_out) Grad-CAM++ for all six regimes x 3 seeds")
def step_xai_layers():
    from scipy.stats import spearmanr
    seeds = SEEDS[:3]
    LAYERS = ("dec4_relu2", "conv5_block3_out")
    out_dir = os.path.join(FED, "xai_layers")
    os.makedirs(out_dir, exist_ok=True)
    data = {}
    for name in XAI_MODELS:
        for seed in seeds:
            p = os.path.join(out_dir, f"{_slug(name)}_seed{seed}.pkl")
            if os.path.exists(p):
                data[(name, seed)] = pickle.load(open(p, "rb"))
                continue
            rec = {l: {"iou_topg": [], "auprc": [], "cam_pred_iou": []} for l in LAYERS}
            dice = []
            ok = True
            for ds in ("kvasir", "cvc"):
                ck = regime_ckpt(name, seed, ds)
                if not os.path.exists(ck):
                    ok = False
                    break
                m = load_trained(build, ck)
                X, Y = TEST[ds]
                Xp = to_model_input(X)
                pred = predict_probs(m, Xp, bs=4) > 0.5
                gms = {l: make_gradcam_model(m, l) for l in LAYERS}
                for j in range(len(X)):
                    for l in LAYERS:
                        cam = gradcam_pp(m, Xp[j], layer_name=l, gm=gms[l])
                        mt = xai_metrics(cam, Y[j])
                        rec[l]["iou_topg"].append(mt["iou_topg"]); rec[l]["auprc"].append(mt["auprc"]); rec[l]["cam_pred_iou"].append(_topk_iou(cam, pred[j]))
                z = np.load(f"{CFG[name][1]}/{CFG[name][0]}_seed{seed}.npz")
                dice.append(z[f"dice__{ds}"])
                del m, gms
                free_mem()
            if not ok:
                continue
            rec["dice"] = np.concatenate(dice)
            pickle.dump(rec, open(p, "wb"))
            data[(name, seed)] = rec
            print(f"[xai_layers {name} seed {seed}] done", flush=True)
    if not data:
        print("no results (weights missing)")
        return
    rows, regime_pts = [], {}
    for (name, seed), rec in data.items():
        for l in LAYERS:
            iou = np.asarray(rec[l]["iou_topg"], float)
            ok = np.isfinite(iou)
            rho = spearmanr(iou[ok], rec["dice"][ok])[0]
            cp = np.asarray(rec[l]["cam_pred_iou"], float)
            rows.append({"model": name, "seed": seed, "layer": l, "mean top-g IoU": np.nanmean(iou), "mean AUPRC": np.nanmean(np.asarray(rec[l]["auprc"], float)),
                         "Spearman(per-image IoU, Dice)": rho, "mean IoU(CAM top-k, predicted mask)": np.nanmean(cp), "mean Dice (162 imgs)": float(rec["dice"].mean())})
    df = pd.DataFrame(rows)
    tab = df.groupby(["layer", "model"], sort=False).agg(**{c: (c, "mean") for c in df.columns if c not in ("model", "seed", "layer")}).round(3)
    display(tab.reset_index(), name="S3_layer_analysis", latex=True)
    for l in LAYERS:
        rhos = []
        for seed in seeds:
            sub = df[(df.layer == l) & (df.seed == seed)]
            if len(sub) >= 4:
                rhos.append(spearmanr(sub["mean top-g IoU"], sub["mean Dice (162 imgs)"])[0])
        if rhos:
            print(f"[{l}] Spearman rho across regimes between mean explanation IoU and mean seen-site Dice: {fmt_ms(rhos, 2)} (seeds {seeds}) "
                  "-> high rho means regime ranking by explanation quality simply follows accuracy.")


# ----------------------------------------------------------------------------------------------------------------
@step("m8", "review", "M8: split CVC-ClinicDB by video-sequence group (hash-graph components), retrain CVC-only / pooled / FedAvg / FedBN, compare with the official split")
def step_m8():
    if not RUN_M8:
        print("RUN_M8 = False -> skip")
        return
    cvc_paths = D["cvc"]["paths_train"] + D["cvc"]["paths_val"] + D["cvc"]["paths_test"]
    Xall = np.concatenate([D["cvc"]["train"][0], D["cvc"]["val"][0], D["cvc"]["test"][0]])
    Yall = np.concatenate([D["cvc"]["train"][1], D["cvc"]["val"][1], D["cvc"]["test"][1]])
    lab, thr_used, dist = hash_groups(cvc_paths, M8_HASH_THR)
    sizes = np.bincount(lab)
    print(f"CVC-ClinicDB frames: {len(cvc_paths)} | groups: {len(sizes)} (hash threshold used: {thr_used}) | group sizes min/median/max: {sizes.min()}/{int(np.median(sizes))}/{sizes.max()}")
    n_test = len(D["cvc"]["test"][0])
    test_m = group_split(lab, n_test, seed=42)
    rest = np.where(~test_m)[0]
    val_m = group_split(lab[rest], int(round(0.1 * len(rest))), seed=43)
    i_test, i_val, i_train = np.where(test_m)[0], rest[val_m], rest[~val_m]
    near = dist[np.ix_(i_test, np.concatenate([i_train, i_val]))].min(1)
    print(f"new split: train {len(i_train)} | val {len(i_val)} | test {len(i_test)} | test images with a train/val neighbour at hash distance <= 8: {int((near <= 8).sum())} of {len(i_test)}"
          f" (official split: see leakage step; was 45/62 by full-image pHash)")
    rows = [{"image_id": os.path.basename(p), "group": int(g), "split": ("test" if test_m[i] else ("val" if i in set(i_val) else "train"))} for i, (p, g) in enumerate(zip(cvc_paths, lab))]
    display(pd.DataFrame(rows), name="M8_cvc_sequence_split_ids", show=False)
    pick = lambda idx: (Xall[idx], Yall[idx])
    D2 = dict(D)
    D2["cvc"] = {"train": pick(i_train), "val": pick(i_val), "test": pick(i_test), "paths_train": [cvc_paths[i] for i in i_train],
                 "paths_val": [cvc_paths[i] for i in i_val], "paths_test": [cvc_paths[i] for i in i_test]}
    TEST2 = dict(TEST)
    TEST2["cvc"] = D2["cvc"]["test"]
    sq = {"cvc": os.path.join(SEQ_ROOT, "cvc"), "kvasir": os.path.join(SEQ_ROOT, "kvasir"), "fed": os.path.join(SEQ_ROOT, "fed")}
    for p in sq.values():
        os.makedirs(p, exist_ok=True)
    tr2, va2 = make_pool(D2)
    cfg2 = {"CVC-only": ("cvc_main", sq["cvc"]), "Kvasir-only": ("kvasir_main", sq["kvasir"]), "Pooled (central)": ("pooled", sq["fed"]),
            "FedAvg": ("fl_fedavg", sq["fed"]), "FedProx": ("fl_fedprox", sq["fed"]), "FedBN": ("fl_fedbn", sq["fed"])}
    orig_kv, orig_cfg = KV_DIR, dict(CFG)
    ctx = dict(D=D2, TEST=TEST2, CVC_DIR=sq["cvc"], KV_DIR=sq["kvasir"], FED=sq["fed"], CFG=cfg2, POOL_TR=tr2, POOL_VA=va2)
    with use_context(ctx):
        free_mem()
        run_seeds("cvc_main", SEEDS, build, D["cvc"]["train"], D["cvc"]["val"], TEST, epochs=EP_CVC, batch=BATCH, out_dir=CVC_DIR)
        free_mem()
        run_seeds("pooled", SEEDS, build, POOL_TR, POOL_VA, TEST, epochs=EP_POOL, batch=BATCH, out_dir=FED)
        for seed in SEEDS:
            for mode in M8_MODES:
                fl_run(f"fl_{mode}", mode, seed)
        for seed in SEEDS:                                   # Kvasir-only never saw CVC frames -> reuse, only re-score on the new CVC test set
            if not os.path.exists(f"{KV_DIR}/kvasir_main_seed{seed}.npz") and os.path.exists(f"{orig_kv}/kvasir_main_seed{seed}.weights.h5"):
                eval_existing("kvasir_main", orig_kv, seed, TEST, out_folder=KV_DIR)
        # ---- analysis inside the new context ----
        present = {n: v for n, v in CFG.items() if has_results(*v) and (n in ("CVC-only", "Kvasir-only", "Pooled (central)") or FL_MODES.get(n) in M8_MODES)}
        ref = "Pooled (central)"
        if ref not in present:
            print("M8 results incomplete")
            return
        for k in ("cvc", "unseen"):
            display(compare_table({n: load_per_image(t, f, k) for n, (t, f) in present.items()}, ref), name=f"M8_seqsplit_vs_pooled_{k}", latex=True)
        new_means = {n: [a.mean() for a in load_dice(t, SEEDS, "cvc", f)] for n, (t, f) in present.items()}
    rows = []
    for n, nm in new_means.items():
        o = orig_cfg[n]
        om = [a.mean() for a in load_dice(o[0], SEEDS, "cvc", o[1])] if has_results(*o, keys=["cvc"]) else [np.nan]
        rows.append({"regime": n, "official CVC test (near-dups)": fmt_ms(om), "sequence-split CVC test": fmt_ms(nm), "change (new - official)": round(float(np.mean(nm) - np.mean(om)), 3)})
    t = pd.DataFrame(rows)
    display(t, name="M8_official_vs_sequence_split", latex=True)
    g = {r["regime"]: float(np.mean(new_means[r["regime"]])) for r in rows}
    if "Pooled (central)" in g:
        print("Seen-site gap to pooled on the SEQUENCE-SPLIT test set:", {n: round(g["Pooled (central)"] - v, 3) for n, v in g.items() if n not in ("Pooled (central)",)})
    print("The drop 'official -> sequence split' is the MEASURED optimism of the near-duplicate test set (images differ between the two test sets, so it is an unpaired comparison).")


# ----------------------------------------------------------------------------------------------------------------
@step("repro", "core", "Reproducibility package + numbers_for_paper.md + SUBMISSION_CHECKLIST.md (R1, M7)")
def step_repro():
    PKG = "repro_package"
    os.makedirs(os.path.join(PKG, "splits"), exist_ok=True)
    names = ["SEEDS", "EPOCHS", "FL_ROUNDS", "FL_LOCAL_EP", "BATCH", "IMG_SIZE", "QUICK_TEST", "RUN_ABLATION", "RUN_KVASIR_AND_FED", "RUN_M2", "RUN_S2", "RUN_AUG2", "RUN_M8",
             "M8_HASH_THR", "N_LIME", "N_GC", "M2_TUNED_CFG", "VAL_PER_IMAGE"]
    cfg = {n: globals()[n] for n in names if n in globals()}
    cfg.update({
        "datasets": {NAMES[k]: len(D[k]["paths_test"]) for k in TEST}, "train_val_sizes": {NAMES[k]: {"train": len(D[k]["paths_train"]), "val": len(D[k]["paths_val"])} for k in ("kvasir", "cvc")},
        "protocol": "PraNet: train = 900 Kvasir-SEG + 550 CVC-ClinicDB (val = random 10% of each, seed 42, checkpoint selection only); test = five official PraNet test sets",
        "model": "ResNet50 (ImageNet) encoder + U-Net decoder (BN), 352x352, sigmoid", "preprocessing": "keras.applications.resnet50.preprocess_input (one function everywhere)",
        "optimizer": "Adam lr 1e-4; centralised: ReduceLROnPlateau(val Dice, 0.5, patience 5) + best-val checkpoint; federated: constant lr, Adam reset every client-round, last-round model",
        "loss": "BCE + Dice", "threshold": 0.5, "augmentation": "on the fly, training batches only (after the split); main = mild _augment; stronger = _augment_strong (aug2: zoom 0.70-1.0, brightness/contrast, saturation, hue, light noise) -> results_*_aug2",
        "federated": {"clients": ["Kvasir-SEG", "CVC-ClinicDB"], "aggregation": "size-weighted (equal-weight variant: S2)", "init": "ImageNet encoder, no warm start", "FedProx_mu": 0.01,
                      "FedBN": "BN params + statistics local; deployable variants: probability-map ensemble, AdaBN (M6)"},
        "statistics": "per-image Dice averaged over seeds; paired sign-flip test on the MEAN difference (20000), Holm; 95% bootstrap CI (10000) unadjusted + Bonferroni-adjusted; Wilcoxon + Hodges-Lehmann reported alongside",
        "near_duplicate_audit": "FOV-cropped pHash (Hamming <= 8) + SSIM >= 0.9 confirmation; sequence groups = connected components of the hash graph (M8)",
        "created": datetime.datetime.now().isoformat(timespec="seconds")})
    json.dump(cfg, open(os.path.join(PKG, "run_config.json"), "w"), indent=2, default=str)
    env = [f"python {sys.version.split()[0]} | {platform.platform()}", f"tensorflow {tf.__version__}"]
    try:
        env.append("GPU: " + subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], capture_output=True, text=True, timeout=30).stdout.strip())
    except Exception:
        env.append("GPU: (nvidia-smi not available)")
    try:
        freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, timeout=180).stdout
    except Exception:
        freeze = "(pip freeze failed)"
    open(os.path.join(PKG, "environment.txt"), "w").write("\n".join(env) + "\n\n# pip freeze\n" + freeze)
    keep = ("tensorflow", "numpy", "scipy", "pandas", "scikit-learn", "scikit-image", "opencv", "matplotlib", "lime", "h5py", "tqdm")
    open(os.path.join(PKG, "requirements.txt"), "w").write("\n".join(l for l in freeze.splitlines() if l.lower().startswith(keep)) + "\n")
    for src in glob.glob(os.path.join(TAB_DIR, "split_image_ids.csv")) + glob.glob(os.path.join(TAB_DIR, "M8_cvc_sequence_split_ids.csv")) + glob.glob(os.path.join(TAB_DIR, "S6_*_pairs.csv")) + glob.glob(os.path.join(TAB_DIR, "S6_fov_phash_all_images.csv")):
        shutil.copy(src, os.path.join(PKG, "splits", os.path.basename(src)))
    try:
        shutil.copy(os.path.abspath(__file__), os.path.join(PKG, "polyp_fl_complete_study.py"))
    except NameError:
        pass
    man = {lab: {"folder": f, "npz": len(glob.glob(os.path.join(f, "*.npz"))), "weights": len(glob.glob(os.path.join(f, "*.weights.h5")))}
           for lab, f in (("CVC-only / ablation", CVC_DIR), ("Kvasir-only / ablation", KV_DIR), ("pooled + federated + new arms", FED), ("M8 sequence split (fed)", os.path.join(SEQ_ROOT, "fed")))
           if os.path.isdir(f)}
    json.dump(man, open(os.path.join(PKG, "results_manifest.json"), "w"), indent=2)
    open(os.path.join(PKG, "README.md"), "w").write(f"""# Federated vs pooled polyp segmentation (leakage-aware): code and configuration

Repository: [FILL: public URL]   |   License: [FILL]

* `polyp_fl_complete_study.py` : the single script (all experiments, tables, figures)
* `run_config.json`, `environment.txt`, `requirements.txt` : seeds {SEEDS}, epochs, protocol, exact package versions
* `splits/` : train/val/test image IDs, CVC-ClinicDB near-duplicate pairs (FOV pHash + SSIM), sequence-group split (M8)

Reproduce: download the five public datasets in the PraNet layout (`PraNet_data/TrainDataset`, `PraNet_data/TestDataset`), install `requirements.txt`, then
`python polyp_fl_complete_study.py --root PraNet_data` (training resumes automatically; finished runs are skipped).
""")
    open(os.path.join(PKG, "DATA_AND_ETHICS.md"), "w").write("""# Data availability, code availability, ethics (DRAFT - complete the [FILL] items and verify citations before submission)

**Data availability.** All five datasets are public: Kvasir-SEG (Jha et al., MMM 2020), CVC-ClinicDB (Bernal et al., Comput Med Imaging Graph 2015),
CVC-ColonDB (Tajbakhsh, Gurudu, Liang, IEEE Trans Med Imaging 35(2):630-644, 2016), ETIS-Larib (Silva et al., IJCARS 2014), CVC-300/EndoScene (Vazquez et al., J Healthc Eng 2017).
Partition: PraNet (Fan et al., MICCAI 2020). All split lists are in `splits/`.

**Code availability.** Code, configuration, seeds, hash lists and the federated implementation: [FILL: repository URL] (archive DOI: [FILL]).

**Ethics.** Only public, de-identified images were used; no new patient data. [FILL: wording required by the journal / institution]

**Funding:** [FILL]   **Competing interests:** [FILL]   **Author contributions:** [FILL]
""")
    # ---- numbers_for_paper.md: every table that the abstract / title / conclusions depend on ----
    want = ["M1_unseen_image_weighted", "B5_vs_pooled_cvc", "B5_vs_pooled_unseen", "M2_arms_mdice", "M2_pooled_minus_fedavg_gaps", "M6_fedbn_deployable_mdice", "M8_official_vs_sequence_split",
            "S2_gaps", "aug2_table_XI_change_in_mdice", "S4_noise_main_vs_aug2", "S3_layer_analysis", "S6_fov_hash_ssim_summary", "leak_subsets_FOV_pHash_SSIM_0.9", "robustness_cvc"]
    with open(os.path.join(REP_DIR, "numbers_for_paper.md"), "w") as f:
        f.write("# Numbers for the paper (auto-generated from outputs/tables; re-derive every abstract number from here)\n\n")
        for n in want:
            p = os.path.join(TAB_DIR, n + ".csv")
            if os.path.exists(p):
                f.write(f"## {n}\n```\n{pd.read_csv(p).to_string(index=False)}\n```\n\n")
            else:
                f.write(f"## {n}\n(not produced yet - run the corresponding step)\n\n")
    # ---- checklist ----
    st = lambda ok: "DONE (code ran)" if ok else "TODO (run the step)"
    ok_m2 = has_results(TAG_POOL_CONST, FED) and has_results(TAG_FEDAVG_TUNED, FED)
    ok_m6 = has_results(TAG_FEDBN_ENS, FED) and has_results(TAG_FEDBN_ADABN, FED)
    ok_m8 = has_results("pooled", os.path.join(SEQ_ROOT, "fed")) and has_results("cvc_main", os.path.join(SEQ_ROOT, "cvc"))
    ok_s2, ok_s4 = has_results(TAG_FEDAVG_EQ, FED), has_results("fl_fedavg", f"results_fed_aug2{SUFFIX}") and has_results("pooled", f"results_fed_aug2{SUFFIX}")
    open(os.path.join(REP_DIR, "SUBMISSION_CHECKLIST.md"), "w").write(f"""# Submission checklist (Review 3) - experiments are code; the rest is YOUR writing / admin work

## Experiments
- M2 protocol-control arms ........ {st(ok_m2)}   -> outputs/tables/M2_*.csv
- M3 consistent test + interval ... built into every comparison table (columns p_perm_holm, CI95_unadj, CI95_bonf, HL_*, agree)
- M6 FedBN deployable ............. {st(ok_m6)}   -> M6_*.csv
- M8 CVC-ClinicDB sequence split .. {st(ok_m8)}   -> M8_*.csv
- S1 unseen image-weighted ........ in B5_vs_pooled_unseen.csv and M1_unseen_image_weighted.csv
- S2 equal-weight FedAvg .......... {st(ok_s2)}   -> S2_*.csv
- S3 explanation layers / correlation: run step xai_layers -> S3_layer_analysis.csv
- S4 strong augmentation + corruption: {st(ok_s4)}   -> S4_*.csv
- S5 empties + per-image val Dice . K1_precision_recall_hd95_*.csv, S5_val_dice_per_image_vs_keras.csv, S5_selection_agreement.csv (needs runs trained with this version; see --val-per-image)
- S6 FOV hash + SSIM + montage .... S6_*.csv and figures/S6_montage_*.png

## Writing / admin (cannot be automated)
- [ ] M1  Title: remove "Robust", "Domain Generalized", "Deep Network". Check M1_unseen_image_weighted.csv before choosing; suggested titles are in the review.
- [ ] M4  Related work: cite pFLFE (2024), Fan et al. (Math. Biosci. Eng. 2024), FL-MedSegBench (2026), QFedPolyp (2026) [as listed in the review - verify details] and NARROW the novelty claim.
- [ ] M5  In Section III-A cite CVC-ColonDB as [53] (Tajbakhsh, Gurudu, Liang, IEEE T-MI 35(2):630-644 - already in your reference list) instead of [35] (Bernal 2012 = a different 300-image database); add one sentence: ClinicDB, CVC-300 and ColonDB share the Barcelona origin, ETIS is the most independent unseen set.
- [ ] M7  Fill repository URL, ethics, funding, competing interests, author contributions ([FILL] in repro_package/); IEEE equal-contribution footnote; cite the companion classification paper as "under review" and disclose it in the cover letter; confirm affiliations.
- [ ] M7/S8 Figures: >= 8 pt fonts (script already sets this); keep ONE of Fig. S1/S2; move Fig. 4, Fig. 7 and Tables XII-XIV to the supplement (page budget).
- [ ] S7  Discussion: shorten the FedProx-ETIS opening; abstract: use the clean-subset result (federated trails pooled on the de-duplicated CVC-ClinicDB images).
- [ ] PDF edits that follow from the NEW results (check each against outputs/tables):
    * Section III-E: broken cross-reference "Fig. ??" -> the architecture figure (Fig. 2).
    * Wilcoxon wording must change to the sign-flip permutation test on the mean (Sec. I-E item 1, II-D, III-F, Table V caption); say intervals are unadjusted, Holm p and Bonferroni CI are the adjusted pair; keep Wilcoxon + Hodges-Lehmann as a robustness column.
    * Tables II / IV: add the "unseen (image-weighted)" column (M1_unseen_image_weighted.csv, B5_vs_pooled_unseen.csv).
    * Table XIV (precision/recall): add the empty-prediction count column (K1_precision_recall_hd95_*.csv).
    * Section IV-G / Table XI: regenerate from aug2_table_XI_change_in_mdice.csv (the printed line gives 'x of 30 comparisons', ranges and validation change); re-check the FedProx-ETIS sentence with aug2_vs_pooled_etis.csv.
    * Section IV-H: add the stronger-augmentation corruption result (S4_noise_main_vs_aug2.csv) - answers "can augmentation fix noise?".
    * Section IV-D + Limitations: the "we did not split by video sequence" bullet becomes the M8 result (M8_official_vs_sequence_split.csv); Limitations "FedBN deployment" bullet becomes the M6 result; add the M2 control and S2 result to Sections IV-C / V.
    * Abstract: delete the Gaussian-noise sentence, add the clean-subset result (S7); update the numbers from outputs/reports/numbers_for_paper.md.
- [ ] Mock review: a co-author re-derives every abstract number from outputs/reports/numbers_for_paper.md.
- [ ] Do not submit the segmentation and the classification paper to the same journal at the same time.
""")
    print(f"'{PKG}/' and {REP_DIR}/ written. Complete every [FILL] before submission.")


# ================================================================================================================
#                                                   MAIN
# ================================================================================================================
ORDER = ["env", "data", "train_central", "train_fl", "ablation", "m2", "s2", "m6", "aug2", "m8", "leak", "tables", "size_fail", "robust",
         "xai", "xai_layers", "efficiency", "pr_hd95", "valtest", "tta", "repro"]


def _select(spec, by_name):
    sel = []
    for tok in [t.strip() for t in spec.split(",") if t.strip()]:
        if tok == "all":
            sel += [s["name"] for s in STEPS]
        elif tok in ("core", "review"):
            sel += [s["name"] for s in STEPS if s["group"] == tok]
            if tok == "review":
                sel += ["repro", "aug2"]
        elif tok in by_name:
            sel.append(tok)
        else:
            raise SystemExit(f"unknown step/group '{tok}'. Use --list.")
    return set(sel)


def main():
    by_name = {s["name"]: s for s in STEPS}
    assert set(ORDER) == set(by_name), f"ORDER and STEPS differ: {set(ORDER) ^ set(by_name)}"
    if ARGS.list:
        for n in ORDER:
            print(f"{n:14s} [{by_name[n]['group']:6s}] {by_name[n]['desc']}")
        return
    sel = _select(ARGS.steps, by_name) - {t.strip() for t in ARGS.skip.split(",") if t.strip()}
    print("steps to run:", [n for n in ORDER if n in sel])
    if sel - {"env"}:
        init_data()
    failures, t_all = [], time.time()
    for n in ORDER:
        if n not in sel:
            continue
        set_step(n)
        t0 = time.time()
        print("\n" + "=" * 110 + f"\n== STEP {n}: {by_name[n]['desc']}\n" + "=" * 110, flush=True)
        try:
            by_name[n]["fn"]()
        except Exception as e:
            import traceback
            traceback.print_exc()
            failures.append((n, f"{type(e).__name__}: {str(e)[:200]}"))
            try:
                free_mem()
            except Exception:
                pass
        print(f"-- step {n} finished in {(time.time() - t0) / 60:.1f} min", flush=True)
    print("\n" + "=" * 110)
    print(f"ALL DONE in {(time.time() - t_all) / 3600:.2f} h | tables: {TAB_DIR} | figures: {FIG_DIR} | reports: {REP_DIR}")
    if failures:
        print("STEPS WITH ERRORS (fix, then re-run only these with --steps):")
        for n, m in failures:
            print(f"   {n}: {m}")


if __name__ == "__main__":
    main()


In [ ]:
import subprocess, sys, time

def run_step(steps):
    t0 = time.time()
    print("=" * 100 + f"\n>>> STEP(S): {steps}\n" + "=" * 100, flush=True)
    with open(f"log_{steps.replace(',', '_')}.txt", "a") as log:
        p = subprocess.Popen([sys.executable, "-u", "polyp_fl_complete_study.py", "--steps", steps],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end="")
            log.write(line); log.flush()
        p.wait()
    print(f"\n<<< {steps}: exit code {p.returncode} | {(time.time() - t0) / 60:.1f} min", flush=True)


In [ ]:
run_step("env,data")

In [ ]:
run_step("train_central")

In [ ]:
run_step("train_fl")

In [ ]:
run_step("m2")

In [ ]:
run_step("s2")

In [ ]:
run_step("m6")

In [ ]:
run_step("aug2")

In [ ]:
run_step("m8")

In [ ]:
run_step("leak")

In [ ]:
run_step("tables")

In [ ]:
run_step("pr_hd95")

In [ ]:
run_step("valtest")

In [ ]:
run_step("xai_layers")

In [ ]:
run_step("repro")

In [ ]:
import os, numpy as np, pandas as pd
from IPython.display import display

ROOT = "."   # notebook wale folder mein hi results_* folders hain
SEEDS = [42, 123, 2024, 7, 99]
MODELS = [("CVC-only", "results_cvc", "cvc_main"),
          ("Kvasir-only", "results_kvasir", "kvasir_main"),
          ("Pooled (central)", "results_fed", "pooled")]
SETS = [("kvasir", "Kvasir-SEG"), ("cvc", "CVC-ClinicDB"), ("cvc300", "CVC-300"), ("colondb", "CVC-ColonDB"), ("etis", "ETIS")]
UNSEEN = ["cvc300", "colondb", "etis"]

def ms(x):
    x = np.asarray(x, float)
    return f"{x.mean():.4f} ± {x.std(ddof=1) if len(x) > 1 else float('nan'):.4f}"

def build(metric):                     # metric = "dice" ya "iou"
    rows = []
    for name, folder, tag in MODELS:
        per_seed = {k: [] for k, _ in SETS}
        per_seed["unseen"] = []
        used = []
        for s in SEEDS:
            p = os.path.join(ROOT, folder, f"{tag}_seed{s}.npz")
            if not os.path.exists(p):
                print(f"  (nahi mila) {p}")
                continue
            z = np.load(p)
            used.append(s)
            for k, _ in SETS:
                per_seed[k].append(z[f"{metric}__{k}"].mean())
            per_seed["unseen"].append(np.concatenate([z[f"{metric}__{k}"] for k in UNSEEN]).mean())   # image-weighted
        if not used:
            continue
        row = {"model": name, "seeds": len(used)}
        for k, lab in SETS:
            row[lab] = ms(per_seed[k])
        row["Unseen (image-weighted, 636 imgs)"] = ms(per_seed["unseen"])
        rows.append(row)
    return pd.DataFrame(rows)

print("===== mDice (mean ± SD over seeds) =====")
display(build("dice"))
print("\n===== mIoU (mean ± SD over seeds) =====")
display(build("iou"))

In [ ]:
run_step("efficiency")

In [ ]:
run_step("size_fail")

In [ ]:
run_step("robust")

In [ ]:
run_step("xai")

In [ ]:
run_step("tta")

In [ ]:
import os, re, glob
import pandas as pd
from IPython.display import display, Image, Markdown

pd.set_option("display.max_columns", None, "display.max_colwidth", 80, "display.width", 250)

STEPS = ["env,data", "train_central", "train_fl", "m2", "s2", "m6", "aug2", "m8", "leak",
         "tables", "pr_hd95", "valtest", "xai_layers", "repro",
         "efficiency", "size_fail", "robust", "xai", "tta"]       # jo steps aap ne chalaye hon, wohi dikhenge
SHOW_LOG = False       # True: har step ka poora (saaf kiya hua) log bhi dikhao;  False: sirf chhoti status lines
SHOW_EPOCHS = False    # True: log mein har epoch ki line bhi
TAB_DIR, FIG_DIR = os.path.join("outputs", "tables"), os.path.join("outputs", "figures")

# kaun si table/figure kis step ki hai (file ke naam ke shuru se)
PAT = {"data": ["split_*", "D4_*", "C0_*", "C1_*", "D5_*"], "train_central": [], "train_fl": ["F1_*"],
       "m2": ["M2_*"], "s2": ["S2_*"], "m6": ["M6_*"], "aug2": ["aug2_*", "S4_*"], "m8": ["M8_*"],
       "leak": ["S6_*", "leak_*", "leakage_*"], "tables": ["B5_*", "C1_*", "C2_*", "M1_*", "W1_*"],
       "pr_hd95": ["K1_*"], "valtest": ["P1_*", "S5_*"], "xai_layers": ["S3_*"], "xai": ["XAI_*"],
       "efficiency": ["G1_*", "L1_*"], "size_fail": ["size_bins_*"], "tta": ["K7c_*"], "ablation": ["ablation_*"]}
_shown = set()

def _clean(text):
    i = text.rfind("GPU free memory")
    if i > 0:
        text = text[i:]
    out, skip = [], False
    for ln in text.splitlines():
        if skip:
            if ln.startswith("version:"):
                skip = False
            continue
        if "None of the algorithms provided by cuDNN" in ln:
            skip = True
            continue
        if "W tensorflow" in ln or ln.startswith(("Relying on driver", "Modify $PATH", "This message will be only")):
            continue
        if not SHOW_EPOCHS and (re.match(r"^Epoch \d+/\d+$", ln) or re.match(r"^\d+/\d+ - \d+s - loss", ln)):
            continue
        out.append(ln)
    return re.sub(r"\n{3,}", "\n\n", "\n".join(out)).strip()

def _status(text):
    keep = [l for l in text.splitlines() if l.startswith(("== STEP", "-- step", "ALL DONE", "Traceback", "FAILED", "failed"))
            or "Error" in l or "failed" in l.lower()]
    return "\n".join(keep)

def _read(p):
    df = pd.read_csv(p)
    if len(df.columns) and str(df.columns[0]).startswith("Unnamed"):
        df = df.set_index(df.columns[0]); df.index.name = None
    return df

def _files(folder, step, ext):
    pats = [f"{s}_table*" if ext == "csv" else f"{s}_fig*" for s in step.split(",")]
    for s in step.split(","):
        pats += PAT.get(s, [])
    got = []
    for pat in pats:
        got += sorted(glob.glob(os.path.join(folder, pat + "." + ext)))
    return list(dict.fromkeys(got))

def show_step(step, width=700):
    display(Markdown(f"## ▶ STEP: `{step}`"))
    log = f"log_{step.replace(',', '_')}.txt"
    if not os.path.exists(log):
        print(f"(is step ka log nahi mila: {log} -> ye step abhi tak chala nahi)\n")
        return
    txt = open(log, errors="ignore").read()
    print(_clean(txt) if SHOW_LOG else _status(txt))
    for p in _files(TAB_DIR, step, "csv"):
        _shown.add(p)
        display(Markdown(f"**table:** `{os.path.basename(p)}`"))
        try:
            display(_read(p))
        except Exception as e:
            print("   (table parh nahi saka:", repr(e)[:80], ")")
    for p in _files(FIG_DIR, step, "png"):
        display(Markdown(f"**figure:** `{os.path.basename(p)}`"))
        display(Image(filename=p, width=width))
    print()

def show_all():
    for s in STEPS:
        show_step(s)
    rest = [p for p in sorted(glob.glob(os.path.join(TAB_DIR, "*.csv"))) if p not in _shown]     # jo kisi step se match nahi hui
    if rest:
        display(Markdown("## ▶ Baaqi tables (kisi step se match nahi hui)"))
        for p in rest:
            display(Markdown(f"**table:** `{os.path.basename(p)}`"))
            try:
                display(_read(p))
            except Exception as e:
                print("   (table parh nahi saka:", repr(e)[:80], ")")

show_all()      # sirf ek step: show_step("m2")

In [ ]:
import os, glob, shutil, subprocess, sys, time
import pandas as pd
from IPython.display import display, Image

FORCE_RERUN = True      # True: purane ablation weights/npz backup mein bhej kar sab dobara train hoga
SKIP_XAI    = True      # True: ablation ke baad wala slow XAI (K7b) skip, sirf ablation tables/runs
SEEDS       = [42, 123, 2024, 7, 99]

# (folder, tag) - CVC: 2 configs, Kvasir: 6 configs  ->  8 x 5 seeds = 40 runs
ABL_TAGS = [("results_cvc",    "cvc_frozen"),
            ("results_cvc",    "cvc_scratch"),
            ("results_kvasir", "kvasir_frozen"),
            ("results_kvasir", "kvasir_scratch"),
            ("results_kvasir", "kvasir_clahe"),
            ("results_kvasir", "kvasir_complex"),
            ("results_kvasir", "kvasir_complex_no_cbam"),
            ("results_kvasir", "kvasir_complex_no_attn")]

# 1) purane ablation results backup mein
if FORCE_RERUN:
    bk = "old_ablation_backup_" + time.strftime("%Y%m%d_%H%M%S")
    moved = 0
    for folder, tag in ABL_TAGS:
        for s in SEEDS:
            for ext in (".weights.h5", ".npz", ".json"):
                p = os.path.join(folder, f"{tag}_seed{s}{ext}")
                if os.path.exists(p):
                    os.makedirs(os.path.join(bk, folder), exist_ok=True)
                    shutil.move(p, os.path.join(bk, folder, os.path.basename(p)))
                    moved += 1
    print(f"{moved} purani files backup mein gayin -> {bk}/")

# 2) ablation-only copy of the study file (K7b XAI optional skip)
src = open("polyp_fl_complete_study.py", encoding="utf-8").read()
if SKIP_XAI:
    key = 'if have("kvasir_complex", KV_DIR) and have("kvasir_main", KV_DIR):'
    assert key in src, "K7b line nahi mili"
    src = src.replace(key, 'if False and have("kvasir_complex", KV_DIR) and have("kvasir_main", KV_DIR):')
open("polyp_fl_ablation_only.py", "w", encoding="utf-8").write(src)

# 3) run: data init + ablation step (CVC + Kvasir)
def run(steps):
    t0 = time.time()
    print("=" * 90 + f"\n>>> {steps}\n" + "=" * 90, flush=True)
    with open(f"log_{steps.replace(',', '_')}_ablation.txt", "a") as log:
        p = subprocess.Popen([sys.executable, "-u", "polyp_fl_ablation_only.py", "--steps", steps],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end="")
            log.write(line); log.flush()
        p.wait()
    print(f"\n<<< {steps}: exit {p.returncode} | {(time.time() - t0) / 60:.1f} min", flush=True)

run("ablation")

# 4) outputs: sirf ablation tables + figures
for f in sorted(glob.glob(os.path.join("outputs", "tables", "ablation_*.csv"))):
    print("\n" + "=" * 90 + f"\n{os.path.basename(f)}\n" + "=" * 90)
    display(pd.read_csv(f))
for f in sorted(glob.glob(os.path.join("outputs", "figures", "ablation_fig*.png"))):
    display(Image(f))

In [ ]:
import os, time

LOG = "log_ablation_detached.txt"
SHOW_LAST = 15          # shuru mein log ki aakhri kitni lines dikhani hain

def alive(pid):
    try:
        import psutil; return psutil.pid_exists(pid)
    except ImportError:
        try: os.kill(pid, 0); return True
        except OSError: return False

pid = int(open("ablation_run.pid").read())
print(f"PID {pid} | zinda: {alive(pid)}\n(Live view band karne ke liye Stop/Interrupt dabayein - training NAHI rukegi)\n")

with open(LOG, "r", errors="ignore") as f:
    for ln in f.readlines()[-SHOW_LAST:]:
        print(ln, end="")
    try:
        while alive(pid):
            ln = f.readline()
            if ln:
                print(ln, end="")
            else:
                time.sleep(1)
        print(f"\nProcess khatam ho gaya.")
    except KeyboardInterrupt:
        print("\nLive view band. Training background mein chal rahi hai.")

In [ ]:
import os, glob, numpy as np, pandas as pd
from IPython.display import display, Image, Markdown

pd.set_option("display.max_columns", None, "display.max_colwidth", 80, "display.width", 250)
SEEDS = [42, 123, 2024, 7, 99]

CFGS = [("CVC-ClinicDB", "results_cvc", "cvc", [("Proposed (fine-tuned)", "cvc_main"),
                                                ("Frozen encoder", "cvc_frozen"),
                                                ("No ImageNet (scratch)", "cvc_scratch")]),
        ("Kvasir-SEG", "results_kvasir", "kvasir", [("Proposed (plain, fine-tuned)", "kvasir_main"),
                                                    ("Frozen encoder", "kvasir_frozen"),
                                                    ("No ImageNet (scratch)", "kvasir_scratch"),
                                                    ("+ CLAHE", "kvasir_clahe"),
                                                    ("Complex: Full (ASPP+Attn+CBAM)", "kvasir_complex"),
                                                    ("Complex: -CBAM", "kvasir_complex_no_cbam"),
                                                    ("Complex: -Attention Gate", "kvasir_complex_no_attn")])]

# 1) quick summary: mean ± SD over seeds (per-seed mDice / mIoU) direct npz files se
for title, folder, key, rows in CFGS:
    out = []
    for name, tag in rows:
        d, i = [], []
        for s in SEEDS:
            p = os.path.join(folder, f"{tag}_seed{s}.npz")
            if os.path.exists(p):
                z = np.load(p)
                d.append(z[f"dice__{key}"].mean()); i.append(z[f"iou__{key}"].mean())
        if d:
            sd = lambda a: np.std(a, ddof=1) if len(a) > 1 else float("nan")
            out.append({"config": name, "seeds": len(d),
                        "mDice": f"{np.mean(d):.4f} ± {sd(d):.4f}",
                        "mIoU": f"{np.mean(i):.4f} ± {sd(i):.4f}"})
        else:
            out.append({"config": name, "seeds": 0, "mDice": "-", "mIoU": "-"})
    display(Markdown(f"### {title}: mean ± SD over seeds"))
    display(pd.DataFrame(out))

# 2) full tables (stat tests, HD95, collapse check) jo step ne CSV mein save ki
for f in sorted(glob.glob(os.path.join("outputs", "tables", "ablation_*.csv"))):
    display(Markdown(f"### {os.path.basename(f)}"))
    display(pd.read_csv(f))

# 3) figures
for f in sorted(glob.glob(os.path.join("outputs", "figures", "ablation_fig*.png"))):
    display(Markdown(f"**{os.path.basename(f)}**"))
    display(Image(f))

In [ ]:
# ============================================================================================
# Paste each block into its own cell at the END of your notebook and run it.
# Every figure is shown in the cell (display) AND saved (png + pdf) to outputs/figures.
# ============================================================================================

# ---------- CELL A: CPU / GPU / Python info ----------
import platform, os, subprocess, sys
try:
    import cpuinfo
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "py-cpuinfo"]); import cpuinfo
print("CPU      :", cpuinfo.get_cpu_info().get("brand_raw"))
print("Threads  :", os.cpu_count())
print("Python   :", platform.python_version())
print("OS       :", platform.platform())
try:
    print("GPU      :", subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip())
except Exception as e:
    print("GPU      : nvidia-smi not found", e)
import tensorflow as tf; print("TensorFlow:", tf.__version__)


# ---------- CELL B: fig_pr_cvc (Supp. S6) ----------
import os, re, glob, pickle, numpy as np, matplotlib.pyplot as plt
from IPython.display import Image as _Img, display as _ipy_display
_FED = globals().get("FED", "results_fed")
_FIG = globals().get("FIG_DIR", os.path.join("outputs", "figures"))
os.makedirs(_FIG, exist_ok=True)
SEEDS5 = [42, 123, 2024, 7, 99]                      # fixed: all 5 seeds
MODELS = ["CVC-only", "Kvasir-only", "Pooled (central)", "FedAvg", "FedProx", "FedBN"]
norm = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())
files = glob.glob(os.path.join(_FED, "pr_metrics", "*.pkl"))
prec, rec = {}, {}
for m in MODELS:
    P, R = [], []
    for s in SEEDS5:
        hit = [f for f in files if norm(m) in norm(os.path.basename(f)) and os.path.basename(f).endswith(f"seed{s}.pkl")]
        if not hit:
            print("MISSING:", m, "seed", s); continue
        p, r = pickle.load(open(hit[0], "rb"))["cvc"][:2]
        P.append(np.nanmean(p)); R.append(np.nanmean(r))
    prec[m], rec[m] = np.array(P), np.array(R)
    print(f"{m:18s} seeds={len(P)}  P={np.mean(P):.3f}  R={np.mean(R):.3f}")

fig, ax = plt.subplots(figsize=(7.2, 3.6))
x = np.arange(len(MODELS)); w = 0.38
for off, d, lab in ((-w/2, prec, "Precision (empty predictions excluded)"), (w/2, rec, "Recall")):
    mu = [d[m].mean() for m in MODELS]; sd = [d[m].std(ddof=1) for m in MODELS]
    ax.bar(x + off, mu, w, yerr=sd, capsize=3, label=lab, alpha=0.9)
    for xi, a, b in zip(x + off, mu, sd):
        ax.text(xi, a + b + 0.006, f"{a:.3f}", ha="center", fontsize=6)
ax.set_xticks(x); ax.set_xticklabels([m.replace(" (central)", "") for m in MODELS], fontsize=8)
ax.set_ylim(0.65, 1.0); ax.set_ylabel("per-image mean")
ax.set_title("CVC-ClinicDB: precision and recall (mean ± SD over 5 seeds)", fontsize=9)
ax.legend(fontsize=7, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(_FIG, f"fig_pr_cvc.{ext}"), dpi=300, bbox_inches="tight")
plt.close(fig)
_ipy_display(_Img(filename=os.path.join(_FIG, "fig_pr_cvc.png")))
print("saved:", os.path.join(_FIG, "fig_pr_cvc.png"))


# ---------- CELL C: fig_xai_violins (Supp. S5) ----------
import os, re, glob, pickle, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
_FED = globals().get("FED", "results_fed")
_FIG = globals().get("FIG_DIR", os.path.join("outputs", "figures"))
os.makedirs(_FIG, exist_ok=True)
MODELS = ["CVC-only", "Kvasir-only", "Pooled (central)", "FedAvg", "FedProx", "FedBN"]
XSEEDS = [42, 123, 2024]
norm = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())

cands = glob.glob(os.path.join(_FED, "xai_multiseed")) or glob.glob("**/xai_multiseed", recursive=True)
assert cands, "xai_multiseed folder not found"
XD = cands[0]; print("using:", XD)
files = glob.glob(os.path.join(XD, "*.pkl")); print(len(files), "pickle files")

def per_image(name, col):
    arrs = []
    for s in XSEEDS:
        for f in files:
            b = os.path.basename(f)
            if norm(name) in norm(b) and f"seed{s}" in b.replace("_", ""):
                arrs.append(pd.to_numeric(pickle.load(open(f, "rb"))["gradcam++"][col], errors="coerce").to_numpy(float))
                break
    if not arrs:
        print("NOT FOUND:", name); return None
    L = min(len(a) for a in arrs)
    return np.nanmean(np.stack([a[:L] for a in arrs]), 0)

fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.4))
for ax, col, title in zip(axs, ("iou_topg", "auprc"), ("Top-$g$ IoU", "AUPRC")):
    got = [(m, per_image(m, col)) for m in MODELS]
    got = [(m, d[np.isfinite(d)]) for m, d in got if d is not None]
    names = [m for m, _ in got]; data = [d for _, d in got]
    parts = ax.violinplot(data, showextrema=False, widths=0.8)
    for b in parts["bodies"]: b.set_alpha(0.55)
    ax.boxplot(data, widths=0.15, showfliers=False)
    ax.scatter(range(1, len(data) + 1), [d.mean() for d in data], color="red", zorder=5, s=14)
    for i, d in enumerate(data, 1): ax.text(i, 1.02, f"{d.mean():.3f}", ha="center", fontsize=6)
    ax.set_xticks(range(1, len(data) + 1)); ax.set_xticklabels([m.replace(" (central)", "") for m in names], rotation=30, ha="right", fontsize=7)
    ax.set_title(title, fontsize=9); ax.set_ylim(0, 1.08); ax.grid(alpha=0.3)
plt.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(_FIG, f"fig_xai_violins.{ext}"), dpi=300, bbox_inches="tight")
from IPython.display import Image as _Img, display as _ipy_display
plt.close(fig)
_ipy_display(_Img(filename=os.path.join(_FIG, "fig_xai_violins.png")))
print("saved:", os.path.join(_FIG, "fig_xai_violins.png"))

In [ ]:
import os, glob
hits = glob.glob("**/*violin*", recursive=True)
print(f"cwd: {os.getcwd()}\n")
for h in sorted(hits):
    print(os.path.abspath(h), "|", round(os.path.getsize(h) / 1024), "KB")
print("\nfig_violin_all.* mili:", any("fig_violin_all" in h for h in hits))

In [ ]:
from IPython.display import Image, display
display(Image("per_image_dice_violin_box.png"))

In [ ]:
   import shutil
   shutil.copy("per_image_dice_violin_box.png", "fig_violin_all.png")

In [ ]:
import os, shutil
from IPython.display import Image, FileLink, display

SRC = "per_image_dice_violin_box.png"
TARGETS = ["outputs/figures", "results_fed"]

assert os.path.exists(SRC), f"{SRC} nahi mili, cwd: {os.getcwd()}"

for d in TARGETS:
    os.makedirs(d, exist_ok=True)
    dst = os.path.join(d, os.path.basename(SRC))
    shutil.copy2(SRC, dst)
    print(f"{os.path.abspath(dst)} | {os.path.getsize(dst)/1024:.0f} KB | maujood: {os.path.exists(dst)}")

print("\nDownload (click karein):")
display(FileLink(SRC, result_html_prefix="Root folder wali file: "))
for d in TARGETS:
    display(FileLink(f"{d}/{SRC}", result_html_prefix=f"{d}: "))

display(Image(SRC))

In [ ]:
# ---------- CELL D: shift-tolerant check of the hash-flagged pairs (answers "SSIM penalizes shifted frames") ----------
import numpy as np, pandas as pd, cv2, os, random
from skimage.registration import phase_cross_correlation
from skimage.metrics import structural_similarity

def _prep(path, size=256):
    g = cv2.cvtColor(crop_fov(cv2.imread(path)), cv2.COLOR_BGR2GRAY)          # notebook helper: crops the black border
    return cv2.resize(g, (size, size), interpolation=cv2.INTER_AREA)

def aligned_ssim(a, b):
    """SSIM after estimating and removing a global translation (phase correlation); evaluated on the overlapping region."""
    shift, _, _ = phase_cross_correlation(a.astype(np.float32), b.astype(np.float32), upsample_factor=4)
    M = np.float32([[1, 0, shift[1]], [0, 1, shift[0]]])
    b2 = cv2.warpAffine(b, M, (b.shape[1], b.shape[0]), borderValue=0)
    mask = cv2.warpAffine(np.ones_like(b, dtype=np.uint8), M, (b.shape[1], b.shape[0]), flags=cv2.INTER_NEAREST) > 0
    ys, xs = np.where(mask)
    y0, y1, x0, x1 = ys.min() + 4, ys.max() - 4, xs.min() + 4, xs.max() - 4
    return float(structural_similarity(a[y0:y1, x0:x1], b2[y0:y1, x0:x1], data_range=255)), float(np.hypot(*shift))

def orb_inliers(a, b):
    """Number of ORB matches that survive RANSAC (similarity transform): robust to shift, rotation and small zoom."""
    orb = cv2.ORB_create(1500)
    ka, da = orb.detectAndCompute(a, None); kb, db = orb.detectAndCompute(b, None)
    if da is None or db is None or len(ka) < 8 or len(kb) < 8:
        return 0
    m = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True).match(da, db)
    if len(m) < 8:
        return 0
    pa = np.float32([ka[x.queryIdx].pt for x in m]); pb = np.float32([kb[x.trainIdx].pt for x in m])
    M, inl = cv2.estimateAffinePartial2D(pa, pb, method=cv2.RANSAC, ransacReprojThreshold=4.0)
    return int(inl.sum()) if inl is not None else 0

train_all = D["kvasir"]["paths_train"] + D["kvasir"]["paths_val"] + D["cvc"]["paths_train"] + D["cvc"]["paths_val"]
rows = []
for k in TEST:                                                                 # all five test sets
    a = audit_near_duplicates(train_all, D[k]["paths_test"], max_dist=8, min_ssim=0.9)
    for _, r in a[a["hash_flag"]].iterrows():
        x, y = _prep(r["test_path"]), _prep(r["train_path"])
        s_al, sh = aligned_ssim(x, y)
        rows.append({"test set": NAMES[k], "test_image": r["test_image"], "nearest_train": r["nearest_train"], "hash_dist": r["hash_dist"],
                     "ssim_plain": r["ssim"], "ssim_aligned": s_al, "shift_px(256)": sh, "orb_inliers": orb_inliers(x, y)})
res = pd.DataFrame(rows)

# control: unrelated pairs (Kvasir-SEG test image vs a random ClinicDB training image) -> how high do these score by chance?
random.seed(0); ctrl = []
for p in random.sample(D["kvasir"]["paths_test"], 40):
    q = random.choice(D["cvc"]["paths_train"]); x, y = _prep(p), _prep(q)
    ctrl.append({"ssim_aligned": aligned_ssim(x, y)[0], "orb_inliers": orb_inliers(x, y)})
ctrl = pd.DataFrame(ctrl)
print(f"CONTROL (unrelated pairs, n={len(ctrl)}): aligned SSIM max {ctrl.ssim_aligned.max():.2f}, ORB inliers max {ctrl.orb_inliers.max()}")
thr_s, thr_o = 0.9, max(30, int(ctrl.orb_inliers.max()) + 1)
res["confirmed_aligned_ssim"] = res["ssim_aligned"] >= thr_s
res["confirmed_orb"] = res["orb_inliers"] >= thr_o
res["confirmed_either"] = res["confirmed_aligned_ssim"] | res["confirmed_orb"]
summ = res.groupby("test set").agg(flagged=("test_image", "count"), plain_ssim_ge_0_9=("ssim_plain", lambda v: int((v >= 0.9).sum())),
                                   aligned_ssim_ge_0_9=("confirmed_aligned_ssim", "sum"), orb_confirmed=("confirmed_orb", "sum"),
                                   either=("confirmed_either", "sum"), median_aligned_ssim=("ssim_aligned", "median"), median_orb=("orb_inliers", "median"))
print(f"(ORB threshold = {thr_o} inliers, i.e. above every unrelated control pair)")
display(summ, name="S6b_shift_tolerant_check")          # uses your notebook display helper (also saves the table)
display(res, name="S6b_shift_tolerant_pairs", show=False)
res.to_csv(os.path.join(globals().get("TAB_DIR", "outputs/tables"), "S6b_shift_tolerant_pairs.csv"), index=False)